In [1]:
import os
log = "/kaggle/working/processed.txt"
if os.path.exists(log):
    os.remove(log)
    print("Cleared resume log")

In [2]:
# Built Track Optimized
#══════════════════════════════════════════════════════════════════════════════
#  AUTOMATED SATELLITE TRACKING & ORBIT DETERMINATION PIPELINE
#  Data Source        : NASA/IPAC IRSA — Zwicky Transient Facility (ZTF)
#  Platform           : Kaggle GPU T4 (Linux container / Python 3.12)
#  Period             : SET START_DATE / END_DATE BELOW BEFORE RUNNING
#  Validation catalog : Space-Track gp_history
#
#  ARCHITECTURE
#  ────────────
#  • Image acquisition  → IRSA ZTF metadata API sweep (all fields in date range)
#                          Authenticated via IRSA_USER / IRSA_PASS Kaggle Secrets
#  • Plate solving      → ZTF FITS headers already carry WCS; ASTAP kept as
#                          fallback verifier (auto-detect scale from FITS first,
#                          then ZTF constants: FOV=6.8°, scale=1.01"/px)
#  • Detection          → Adaptive 2-pass, background subtraction
#                          Short-exposure dot-mode retained for future datasets
#  • TLE optimisation   → RANSAC → Gauss IOD → DE → LM/SGP4
#  • Catalog matching   → Mahalanobis + GCAT + ESA DISCOS
#  • Multi-pass solver  → Stage 6B master TLE
#  • Validation catalog → Space-Track gp_history
#
#  ZTF INSTRUMENT NOTES
#  ─────────────────────
#  • Full focal plane FOV ≈ 6.8° × 6.8° (~47 deg²), pixel scale ≈ 1.01 "/px
#  • Each CCD chip field: ~1.01° × 0.87°  (detector is 16-chip mosaic)
#  • ZTF exposure time: 30 s fixed (g, r, i filters)
#  • Sidereal tracking → satellites appear as linear streaks in 30 s frames
#  • Pipeline handles per-chip FITS (one file per CCD readout quadrant)
#  • MPC observatory code for Palomar: 675
# ══════════════════════════════════════════════════════════════════════════════



import os, sys, subprocess, shutil, calendar, glob, csv, warnings, time
import re, requests, json, tempfile, math
from collections import defaultdict
from datetime import timezone
from pathlib import Path

# ── Credentials ───────────────────────────────────────────────────────────────
# Resolution order for each secret:
#   1. Kaggle UserSecretsClient  (Add-ons ▸ Secrets, attached to this notebook)
#   2. OS environment variable   (useful when running outside Kaggle)
#   3. Notebook-level variable   (secret_value_N pattern Kaggle auto-generates
#      when you click "Insert" on a secret)
#
# Secrets required:
#   IRSA_USER          — your IRSA / NASA account email
#   IRSA_PASS          — your IRSA / NASA account password
#   SPACETRACK_USER    — space-track.org email
#   SPACETRACK_PASS    — space-track.org password
#   DISCOS_TOKEN       — ESA DISCOS bearer token (optional but recommended)
#
# To register for IRSA (free): https://irsa.ipac.caltech.edu/account/signon/register.do

# ── Credentials ───────────────────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret("DISCOS_TOKEN")
    secret_value_1 = user_secrets.get_secret("IRSA_PASS")
    secret_value_2 = user_secrets.get_secret("IRSA_USER")
    secret_value_3 = user_secrets.get_secret("NEOCC_PASS")
    secret_value_4 = user_secrets.get_secret("NEOCC_USER")
    secret_value_5 = user_secrets.get_secret("SPACETRACK_PASS")
    secret_value_6 = user_secrets.get_secret("SPACETRACK_USER")
except Exception as e:
    print(f"Failed to load Kaggle secrets: {e}")
    secret_value_0 = secret_value_1 = secret_value_2 = secret_value_3 = ""
    secret_value_4 = secret_value_5 = secret_value_6 = ""

# Map your Kaggle secrets directly to the pipeline variables
DISCOS_TOKEN    = secret_value_0
IRSA_PASS       = secret_value_1
IRSA_USER       = secret_value_2
SPACETRACK_PASS = secret_value_5
SPACETRACK_USER = secret_value_6

def _check_credentials():
    print("\n── Credential check ──────────────────────────────────────────")
    secrets = {
        "IRSA_USER":       IRSA_USER,
        "IRSA_PASS":       IRSA_PASS,
        "SPACETRACK_USER": SPACETRACK_USER,
        "SPACETRACK_PASS": SPACETRACK_PASS,
        "DISCOS_TOKEN":    DISCOS_TOKEN,
    }
    all_ok = True
    for name, val in secrets.items():
        if val:
            masked = val[:2] + "*" * (len(val) - 4) + val[-2:] if len(val) > 4 else "****"
            print(f"  ✅ {name:<20} {masked}")
        else:
            # DISCOS is optional; flag but don't fail
            marker = "⚠ " if name == "DISCOS_TOKEN" else "❌"
            print(f"  {marker} {name:<20} NOT FOUND")
            if name != "DISCOS_TOKEN":
                all_ok = False
    
    if not all_ok:
        print("\n❌ MISSING CREDENTIALS. Please check your Kaggle Secrets.")
    
    print("──────────────────────────────────────────────────────────────\n")
    return all_ok

_check_credentials()


# ── ASTAP ─────────────────────────────────────────────────────────────────────
_ASTAP_CATALOG = "d80"
_ASTAP_BIN     = "/usr/local/bin/astap"

# ZTF full-focal-plane fallback constants
# Used ONLY when FITS header auto-detection fails.
# Individual CCD chip files will have their own correct WCS/CDELT headers.
_ZTF_FOV_DEG         = 6.8    # full mosaic field of view (degrees)
_ZTF_SCALE_ARCSEC_PX = 1.01   # pixel scale  (arcsec / pixel)


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 0 – INSTALL DEPENDENCIES
# ══════════════════════════════════════════════════════════════════════════════

def stage0_install():
    print("\n" + "="*60)
    print("STAGE 0: INSTALLING DEPENDENCIES")
    print("="*60)

    print("  scipy pre-installed in Cell 1 — skipping reinstall ✅")

    print("  Activating CuPy GPU backend (T4) …", end=" ", flush=True)
    try:
        import cupy as cp
        cp.cuda.Device(0).use()
        print(f"✅  ({cp.cuda.runtime.getDeviceProperties(0)['name'].decode()})")
    except Exception as e:
        print(f"⚠  CuPy not available — falling back to CPU ({e})")
    
    # Auto-detect CUDA version and install matching CuPy
    _cuda_ver = "12"
    try:
        _nv = subprocess.run(["nvcc","--version"], capture_output=True, text=True)
        if "release 11" in _nv.stdout or "release 11" in (_nv.stderr or ""):
            _cuda_ver = "11"
        elif "release 12" in _nv.stdout or "release 12" in (_nv.stderr or ""):
            _cuda_ver = "12"
    except Exception:
        pass
    _cupy_pkg = f"cupy-cuda{_cuda_ver}x"
    print(f"  CUDA {_cuda_ver} detected → installing {_cupy_pkg}")

    pkgs = [
        _cupy_pkg,        # correct CUDA-version CuPy
        "astropy",
        "git+https://github.com/astropy/photutils.git",
        "sgp4",
        "skyfield",
        "pandas",
        "beautifulsoup4",
        "lxml",
    ]
    all_ok = True
    for pkg in pkgs:
        label = pkg.split("/")[-1] if pkg.startswith("git+") else pkg
        print(f"  Installing {label} …", end=" ", flush=True)
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", pkg],
            capture_output=True
        )
        if result.returncode == 0:
            print("✅")
        else:
            stderr = result.stderr.decode(errors="replace").strip().splitlines()
            print(f"❌  (last line: {stderr[-1] if stderr else 'unknown error'})")
            all_ok = False

    if not all_ok:
        raise RuntimeError(
            "One or more packages failed to install. "
            "Check internet access is ON in Notebook Settings."
        )

    print("  Installing GTK2 + Xvfb for ASTAP …", end=" ", flush=True)
    gtk = subprocess.run(
        ["apt-get", "install", "-y", "-q",
         "libgdk-pixbuf2.0-0", "libgtk2.0-0", "xvfb"],
        capture_output=True
    )
    if gtk.returncode == 0:
        print("✅")
    else:
        raise RuntimeError(
            f"Failed to install GTK2/Xvfb:\n{gtk.stderr.decode()[-300:]}"
        )

    if not os.path.exists(_ASTAP_BIN):
        print("  Installing ASTAP CLI engine …", end=" ", flush=True)
        astap_deb = "/tmp/astap.deb"
        subprocess.run([
            "wget", "-q", "-L", "--content-disposition",
            "https://sourceforge.net/projects/astap-program/files/"
            "linux_installer/astap_amd64.deb/download",
            "-O", astap_deb
        ], check=True)
        subprocess.run(["dpkg", "-i", astap_deb],
                       check=True, stderr=subprocess.DEVNULL)
        os.remove(astap_deb)
        print("✅")
    else:
        print(f"  ✅ ASTAP already present at {_ASTAP_BIN}.")

    d80_data_paths = [
        "/usr/share/astap/", "/var/lib/astap/",
        os.path.expanduser("~/.astap/"), "/opt/astap/",
    ]

    def _find_d80_files():
        files = []
        for p in d80_data_paths:
            files += (glob.glob(os.path.join(p, "d80*.290")) +
                      glob.glob(os.path.join(p, "d80*.1476")) +
                      glob.glob(os.path.join(p, "d80_*")))
        return [f for f in files if not f.endswith("_marker")]

    d80_files = _find_d80_files()
    if not d80_files:
        print("  Installing D80 star-database catalog …")
        kaggle_deb = (
            "/kaggle/input/datasets/kennyabayomi/"
            "d-80-database-1/d80_star_database.deb"
        )
        d80_deb    = "/tmp/d80.deb"
        downloaded = False

        if os.path.exists(kaggle_deb) and os.path.getsize(kaggle_deb) > 1_000_000:
            print("    Found D80 in Kaggle Dataset — copying …", end=" ", flush=True)
            shutil.copy(kaggle_deb, d80_deb)
            print("✅")
            downloaded = True
        else:
            mirrors = [
                "https://github.com/han-k59/astap/releases/download/astap_last/d80.deb",
                "https://downloads.sourceforge.net/project/astap-program/star_databases/d80.deb",
                "https://netcologne.dl.sourceforge.net/project/astap-program/star_databases/d80.deb",
            ]
            for mirror in mirrors:
                label = mirror.split("/")[2]
                print(f"    Trying {label} …", end=" ", flush=True)
                result = subprocess.run([
                    "wget", "-q", "-L", "--tries=2", "--timeout=180",
                    "--no-check-certificate", mirror, "-O", d80_deb
                ], capture_output=True)
                size = os.path.getsize(d80_deb) if os.path.exists(d80_deb) else 0
                if result.returncode == 0 and size > 1_000_000:
                    print(f"✅  ({size // 1_000_000} MB)")
                    downloaded = True
                    break
                else:
                    print(f"❌  (exit={result.returncode}, {size // 1024} KB)")
                    if os.path.exists(d80_deb):
                        os.remove(d80_deb)
                    time.sleep(2)

            if not downloaded:
                raise RuntimeError(
                    "D80 catalog not found.\n"
                    "→ Attach dataset 'kennyabayomi/d-80-database-1' "
                    "under Add-ons ▸ Datasets.\n"
                    "→ OR enable Internet in Notebook Settings and re-run."
                )

        subprocess.run(["dpkg", "-i", d80_deb],
                       check=True, stderr=subprocess.DEVNULL)
        if os.path.exists(d80_deb):
            os.remove(d80_deb)

        d80_files = _find_d80_files()
        if not d80_files:
            raise RuntimeError("D80 .deb installed but no star data files found.")

        marker = "/usr/share/astap/d80_cat_marker"
        Path(marker).parent.mkdir(parents=True, exist_ok=True)
        Path(marker).touch()
        print(f"  ✅ D80 catalog installed: {len(d80_files)} data files "
              f"at {os.path.dirname(d80_files[0])}")
    else:
        print(f"  ✅ D80 catalog already present "
              f"({len(d80_files)} files at {os.path.dirname(d80_files[0])})")

    print("  Verifying scipy import …", end=" ", flush=True)
    check = subprocess.run(
        [sys.executable, "-c",
         "from scipy.ndimage import binary_dilation; "
         "from scipy.optimize import least_squares; print('ok')"],
        capture_output=True, text=True
    )
    if check.returncode == 0 and "ok" in check.stdout:
        print("✅")
    else:
        err = (check.stderr or "").strip().splitlines()
        raise RuntimeError(
            f"scipy broken after install: {err[-1] if err else 'unknown'}\n"
            "→ Restart & Clear Output, then re-run."
        )

# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

# ── Date range ────────────────────────────────────────────────────────────────
START_DATE = "2024-01-15"
END_DATE   = "2024-01-20"   
# ── Multi-night arc extension ──────────────────────────────────────────────
# Set to path of "reliable_tles_for_next_night.txt" from a previous run
# to use those TLEs as warm-start seeds, extending arc duration across nights.
# Leave as None on first run.
SEED_TLE_PATH = None   # e.g. "/kaggle/working/reliable_tles_for_next_night.txt"

# ── Observatory: Palomar Mountain / ZTF ───────────────────────────────────────
OBSERVATORY      = "ZTF"
OBSERVATORY_CODE = "675"          # MPC code for Palomar Mountain
SITE_LONGITUDE   = -116.8650      # degrees East  (Palomar)
SITE_LATITUDE    =  33.3563       # degrees North (Palomar)
SITE_ALTITUDE    =  1712          # metres

WORKING_DIR   = "/kaggle/working"
OUTPUT_DIR    = f"{WORKING_DIR}/satellite_data_{START_DATE}_to_{END_DATE}"
TMP_DIR       = "/tmp/fits_buffer"
CATALOG_DIR   = f"{WORKING_DIR}/catalogs"
OUTPUT_CSV    = f"{WORKING_DIR}/satellite_results.csv"
OUTPUT_MPC    = f"{WORKING_DIR}/satellite_mpc.txt"
OUTPUT_REPORT = f"{WORKING_DIR}/SATELLITE_REPORT.txt"
ZIP_PATH      = f"{WORKING_DIR}/export_{START_DATE}_to_{END_DATE}"

for _d in [OUTPUT_DIR, TMP_DIR, CATALOG_DIR]:
    os.makedirs(_d, exist_ok=True)

# ── Tracking ─────────────────────────────────────────────────────────────────
MAX_TRACK_GAP_S    = 10800.0   # max seconds between consecutive track epochs
                                    # 600 s = 10 min covers intra-night ZTF cadence
                                    # for LEO; increase to ~3600 for GEO/MEO

# ── Detection thresholds – Pass 1 (normal) ───────────────────────────────────
SIGMA_THRESHOLD_P1 = 2.3
FWHM_P1            = 4.0
MIN_ELONGATION_P1  = 1.2
MAX_SOURCES_P1     = 100


# ── Detection thresholds – Pass 2 (relaxed fallback) ─────────────────────────
SIGMA_THRESHOLD_P2 = 1.5
FWHM_P2            = 3.0
MIN_ELONGATION_P2  = 1.02
MAX_SOURCES_P2     = 200

# ── Tracking ─────────────────────────────────────────────────────────────────
MOTION_ARCSEC            = 5.0
MIN_ANGULAR_VEL_ARCSEC_S = 0.0
MAX_SPEED_ARCSEC_S       = 600.0
MAX_GAP_FRAMES           = 30
MIN_TRACK_LEN            = 2
MAX_PA_DIFF_DEG          = 15.0

# ── Export ────────────────────────────────────────────────────────────────────
SUBSAMPLE_STEP = 5
MIN_EXPORT_PTS = 3

# ── IOD / optimisation quality gates ─────────────────────────────────────────
SHORT_ARC_THRESHOLD_S         = 120.0
MIN_RANSAC_INLIERS            = 3
MAX_ACCEPTABLE_DE_RMSE_ARCSEC = 5000.0
# Legacy same-night, catalog-name-based track fusion (Stage 6B). Its
# association method groups tracks by "nearest same catalog name," which is
# unreliable in crowded GEO/MEO bands where many unrelated real objects
# nearest-match the same catalog entry. Superseded by Stage 4A/4B's
# geometry-based cross-night linkage, which verifies actual trajectory
# consistency rather than label proximity. Kept behind a flag for debugging
# only -- leave False for any report you intend to trust or publish.
ENABLE_LEGACY_STAGE6B_FUSION = False


# ── Catalog correlation ───────────────────────────────────────────────────────
MAHAL_MATCH_SIGMA  = 4.0
MAHAL_FLOOR_ARCSEC = 30.0

# ── CelesTrak TLE groups ──────────────────────────────────────────────────────
CELESTRAK_GROUPS = [
    ("active",       "https://celestrak.org/NORAD/elements/gp.php?GROUP=active&FORMAT=tle"),
    ("supplemental", "https://celestrak.org/NORAD/elements/gp.php?GROUP=supplemental&FORMAT=tle"),
    ("debris",       "https://celestrak.org/NORAD/elements/gp.php?GROUP=debris&FORMAT=tle"),
    ("analyst",      "https://celestrak.org/NORAD/elements/gp.php?GROUP=analyst&FORMAT=tle"),
    ("last-30-days", "https://celestrak.org/NORAD/elements/gp.php?GROUP=last-30-days&FORMAT=tle"),
]

# ── Space-Track historical GP catalog ────────────────────────────────────────
SPACETRACK_URL    = "https://www.space-track.org"
SPACETRACK_GP_URL = (
    f"{SPACETRACK_URL}/basicspacedata/query/class/gp_history"
    f"/EPOCH/{START_DATE}--{END_DATE}/orderby/NORAD_CAT_ID/format/3le"
)

# ── ESA auxiliary catalogs ────────────────────────────────────────────────────
DISCOS_URL = "https://discosweb.esoc.esa.int"
GCAT_URL   = "https://celestrak.org/pub/satcat.csv"

# ── IRSA / ZTF API constants ──────────────────────────────────────────────────
_IRSA_TAP_URL     = "https://irsa.ipac.caltech.edu/TAP/sync"
_IRSA_ZTF_BASEURL = "https://irsa.ipac.caltech.edu/ibe/data/ztf/products/sci"
_IRSA_LOGIN_URL   = "https://irsa.ipac.caltech.edu/account/signon/login.do"
_IRSA_MAX_FILES = 4000
_IRSA_DOWNLOAD_DELAY_S = 0.5


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 1 – DOWNLOAD FITS FROM NASA IRSA ZTF
# ══════════════════════════════════════════════════════════════════════════════

def _irsa_session() -> requests.Session:
    global IRSA_USER, IRSA_PASS
    if not IRSA_USER:
        IRSA_USER = UserSecretsClient().get_secret("IRSA_USER")
    if not IRSA_PASS:
        IRSA_PASS = UserSecretsClient().get_secret("IRSA_PASS")

    if not IRSA_USER or not IRSA_PASS:
        raise RuntimeError(
            "IRSA credentials not set.\n"
            "→ Register free at https://irsa.ipac.caltech.edu/account/signon/register.do\n"
        )

    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (ZTF-satellite-pipeline)"})

    payload = {
        "josso_cmd":      "login",
        "josso_username": IRSA_USER,
        "josso_password": IRSA_PASS,
    }
    resp = session.post(_IRSA_LOGIN_URL, data=payload, timeout=60, allow_redirects=True)

    has_session = any("JOSSO" in k or "irsa" in k.lower() for k in session.cookies.keys())
    authed_page = any(w in resp.text.lower() for w in ("sign out", "logout", "my account", "signout"))

    if not has_session and not authed_page:
        raise RuntimeError("IRSA login failed. Check credentials.")

    print("  ✅ NASA IRSA authenticated.")
    return session


import math, re
from datetime import datetime, timezone

_DATE_RE = re.compile(r'^\d{4}-\d{2}-\d{2}$')
_TIME_RE = re.compile(r'^\d{2}:\d{2}:\d{2}')


# ═══════════════════════════════════════════════════════════════════════════
#  FIX 1 — IRSA IBE IPAC TABLE PARSER  (replaces _irsa_query_ztf_metadata)
# ═══════════════════════════════════════════════════════════════════════════


import re
import math
import time
import requests
from datetime import datetime, timezone


# ── Copy these constants from your main pipeline ───────────────────────────
# _IRSA_MAX_FILES is defined once, in CONFIGURATION — do not redefine it
# here. A duplicate definition in this location previously silently
# overrode the CONFIGURATION value since module-level assignments execute
# top-to-bottom and the later one wins.
_IRSA_LOGIN_URL = "https://irsa.ipac.caltech.edu/account/signon/login.do"


def _iso_to_jd(iso_str: str) -> float:
    fmt = "%Y-%m-%dT%H:%M:%S" if "T" in iso_str else "%Y-%m-%d"
    dt  = datetime.strptime(iso_str.rstrip("Z"), fmt)
    epoch = datetime(2000, 1, 1, 12, 0, 0)
    return 2451545.0 + (dt - epoch).total_seconds() / 86400.0


def _jd_to_utc(jd: float) -> datetime:
    return datetime.fromtimestamp((jd - 2440587.5) * 86400.0, tz=timezone.utc)


# ══════════════════════════════════════════════════════════════════════════════
#  COLUMN BOUNDARY EXTRACTOR
#  Parses a pipe row like  "| col1 | col2 | col3 |"
#  and returns a list of (start, end) character slices for the data rows.
#
#  In fixlen mode, data rows are padded so each field occupies exactly the
#  same character span as in the header pipe row (minus the leading |).
# ══════════════════════════════════════════════════════════════════════════════

def _extract_column_boundaries(pipe_row: str):
    """
    Given the raw column-name pipe row, return:
      columns  : list of column name strings
      bounds   : list of (start, end) integer slices into a data row

    Example pipe row (truncated):
      "|                ra|                dec| infobits|field|..."

    The leading "|" is at position 0.
    Field 1 spans positions 1..18  → slice [1:19]
    Field 2 spans positions 19..34 → slice [19:35]
    etc.
    """
    # Find every "|" character position
    pipe_positions = [i for i, c in enumerate(pipe_row) if c == "|"]
    if len(pipe_positions) < 2:
        return [], []

    columns = []
    bounds  = []
    for i in range(len(pipe_positions) - 1):
        start = pipe_positions[i] + 1          # character after the |
        end   = pipe_positions[i + 1]          # up to (not including) next |
        col_name = pipe_row[start:end].strip()
        if col_name:
            columns.append(col_name)
            bounds.append((start, end))

    return columns, bounds


def _daterange_chunks(start_date_str, end_date_str, chunk_days=3):
    from datetime import datetime, timedelta
    start = datetime.strptime(start_date_str, "%Y-%m-%d")
    end   = datetime.strptime(end_date_str, "%Y-%m-%d")
    chunks, cur = [], start
    while cur <= end:
        chunk_end = min(cur + timedelta(days=chunk_days - 1), end)
        chunks.append((cur.strftime("%Y-%m-%d"), chunk_end.strftime("%Y-%m-%d")))
        cur = chunk_end + timedelta(days=1)
    return chunks


def _irsa_query_ztf_metadata_recursive(session, dt_start, dt_end, depth=0,
                                        max_depth=6, min_span_hours=2.0):
    """
    Query IRSA IBE for [dt_start, dt_end), automatically splitting the
    interval in half and re-querying each half if the result hit the
    _IRSA_MAX_FILES cap (a strong signal of truncation, since MAXREC caps
    the query at exactly that count). Recurses down to min_span_hours
    (default 2h) or max_depth, whichever comes first, to bound worst-case
    query count.

    This exists because day-level chunking alone was NOT fine enough: in
    a real run, every single one of 6 daily chunks hit
    QUERY_STATUS=OVERFLOW and got silently capped at _IRSA_MAX_FILES,
    meaning an unknown fraction of each night's exposures were never even
    queried, let alone downloaded -- data loss before Stage 1 even starts.
    """
    dt_start_s = dt_start.strftime("%Y-%m-%dT%H:%M:%S")
    dt_end_s   = dt_end.strftime("%Y-%m-%dT%H:%M:%S")
    try:
        recs = _irsa_query_ztf_metadata(session, dt_start_s, dt_end_s)
    except RuntimeError as e:
        print(f"    ⚠  Sub-query failed [{dt_start_s} -> {dt_end_s}]: {e}")
        return []

    span_hours = (dt_end - dt_start).total_seconds() / 3600.0
    hit_cap = len(recs) >= _IRSA_MAX_FILES

    if hit_cap and depth < max_depth and span_hours > min_span_hours:
        mid = dt_start + (dt_end - dt_start) / 2
        print(f"    ↳  [{dt_start_s} -> {dt_end_s}] hit the {_IRSA_MAX_FILES} "
              f"cap — splitting at {mid.strftime('%Y-%m-%dT%H:%M:%S')} "
              f"(depth {depth+1})")
        left  = _irsa_query_ztf_metadata_recursive(
            session, dt_start, mid, depth=depth + 1,
            max_depth=max_depth, min_span_hours=min_span_hours)
        right = _irsa_query_ztf_metadata_recursive(
            session, mid, dt_end, depth=depth + 1,
            max_depth=max_depth, min_span_hours=min_span_hours)
        return left + right

    if hit_cap:
        print(f"    ⚠  [{dt_start_s} -> {dt_end_s}] still at the "
              f"{_IRSA_MAX_FILES} cap after reaching the recursion/span "
              f"limit (depth={depth}, span={span_hours:.1f}h) -- some "
              f"records in this window may still be missing.")
    return recs


def _irsa_query_ztf_metadata_chunked(session, date_start, date_end, chunk_days=3,
                                      pipeline_start=None, wallclock_budget_s=None):
    """
    Wraps _irsa_query_ztf_metadata in date-chunked calls so a wide date
    range isn't silently collapsed by the per-query _IRSA_MAX_FILES cap --
    each chunk gets its own fresh record budget, guaranteeing coverage
    actually spans the requested range instead of exhausting the cap on
    the first night. Each chunk is further split recursively (down to
    2-hour windows) if it still hits the cap -- see
    _irsa_query_ztf_metadata_recursive for why day-level chunking alone
    wasn't sufficient.
    """
    from datetime import datetime as _dt
    import time as _time_mod
    chunks = _daterange_chunks(date_start, date_end, chunk_days=chunk_days)
    print(f"  Chunked IRSA query: {len(chunks)} chunk(s) of ~{chunk_days} "
          f"day(s) ({date_start} -> {date_end})")
    all_records = []
    for i, (c_start, c_end) in enumerate(chunks):
        if pipeline_start is not None and wallclock_budget_s is not None:
            elapsed = _time_mod.time() - pipeline_start
            if elapsed > wallclock_budget_s:
                print(f"  ⚠  Wall-clock budget reached ({elapsed/3600:.1f}h) -- "
                      f"stopping further chunk downloads. Processing "
                      f"{len(all_records)} records gathered so far.")
                break
        print(f"    Chunk {i+1}/{len(chunks)}: {c_start} -> {c_end}")
        dt_start = _dt.strptime(c_start, "%Y-%m-%d")
        dt_end   = _dt.strptime(c_end, "%Y-%m-%d").replace(
            hour=23, minute=59, second=59)
        recs = _irsa_query_ztf_metadata_recursive(session, dt_start, dt_end)
        all_records.extend(recs)
    print(f"  Chunked IRSA query total: {len(all_records)} records across "
          f"{len(chunks)} chunk(s)")
    return all_records


def _irsa_query_ztf_metadata(session: requests.Session,
                              date_start: str,
                              date_end: str) -> list:
    """
    Query ZTF science image metadata via IRSA IBE metadata API.

    IBE returns a fixlen=T IPAC table:
      \\keyword = value          ← backslash comments (metadata)
      | col1 | col2 | …         ← pipe row 0 : column NAMES
      | type | type | …         ← pipe row 1 : data types   (skip)
      | unit | unit | …         ← pipe row 2 : units        (skip)
      | null | null | …         ← pipe row 3 : null values  (skip)
      <fixed-width data line>   ← data rows  (NO leading pipe!)
      <fixed-width data line>
      …

    The column boundaries are extracted from the column-name pipe row
    and applied to every subsequent fixed-width data line.
    """
    print(f"  Querying IRSA IBE for ZTF metadata ({date_start} → {date_end}) …",
          flush=True)

    jd_start = _iso_to_jd(date_start if "T" in date_start else f"{date_start}T00:00:00")
    jd_end   = _iso_to_jd(date_end   if "T" in date_end   else f"{date_end}T23:59:59")
    
    ibe_url = "https://irsa.ipac.caltech.edu/ibe/search/ztf/products/sci"
    params  = {
        "WHERE":  f"obsjd>{jd_start:.5f} AND obsjd<{jd_end:.5f}",
        "MAXREC": str(_IRSA_MAX_FILES),
    }

    resp = session.get(ibe_url, params=params, timeout=300)
    if resp.status_code != 200:
        raise RuntimeError(
            f"IRSA IBE query failed (HTTP {resp.status_code}): {resp.text[:200]}"
        )

    try:
        lines          = resp.text.splitlines()
        columns        = []
        bounds         = []      # (start, end) slices for fixlen data rows
        records        = []
        rows_retrieved = None
        query_status   = "UNKNOWN"
        is_fixlen      = False

        # ── State machine ─────────────────────────────────────────────────
        # 0 = waiting for column-name pipe row
        # 1 = skip type  pipe row
        # 2 = skip unit  pipe row
        # 3 = skip null  pipe row
        # 4 = reading fixed-width data rows
        state = 0

        for line in lines:
            stripped = line.strip()
            if not stripped:
                continue

            # ── Backslash header keywords ─────────────────────────────────
            if stripped.startswith("\\"):
                if "fixlen" in stripped.lower() and "= t" in stripped.lower():
                    is_fixlen = True
                if "RowsRetrieved" in stripped:
                    try:
                        rows_retrieved = int(stripped.split("=")[1].strip())
                    except Exception:
                        pass
                if "QUERY_STATUS" in stripped:
                    query_status = stripped.split("=", 1)[1].strip().strip("'\"")
                continue

            # ── Pipe-delimited rows (header block) ────────────────────────
            if stripped.startswith("|"):
                if state == 0:
                    # Column-name row: extract names AND boundaries
                    # Use the RAW (unstripped) line to preserve character positions
                    columns, bounds = _extract_column_boundaries(line.rstrip("\n"))
                    print(f"  DEBUG columns ({len(columns)}): {columns}")
                    print(f"  DEBUG fixlen={is_fixlen}")
                    if "obsdate"    in columns:
                        print(f"  DEBUG obsdate    @ index {columns.index('obsdate')}")
                    if "obsjd"      in columns:
                        print(f"  DEBUG obsjd      @ index {columns.index('obsjd')}")
                    if "filefracday" in columns:
                        print(f"  DEBUG filefracday @ index {columns.index('filefracday')}")
                    state = 1
                elif state in (1, 2, 3):
                    state += 1          # skip type / unit / null rows
                # In state 4, a pipe row would be unexpected; ignore it.
                continue

            # ── Fixed-width data rows (state 4) ───────────────────────────
            if state == 4:
                if not bounds:
                    # Fallback: split on whitespace if no bounds available
                    parts = stripped.split()
                    if len(parts) >= len(columns):
                        records.append(dict(zip(columns, parts[:len(columns)])))
                    continue

                # Use character slices from the header pipe row
                rec = {}
                for col, (s, e) in zip(columns, bounds):
                    # Pad line with spaces if shorter than expected
                    val = line[s:e].strip() if e <= len(line) else ""
                    rec[col] = val
                records.append(rec)
                continue

            # ── Transition: after 3 definition rows, next non-pipe line
            #    is a data row.  state==3 was just incremented to 4 above
            #    via the pipe-row branch, so this branch handles the case
            #    where the first data row arrives while state is still 3
            #    (shouldn't happen with correct IBE, but guard anyway).
            if state == 3:
                state = 4
                # re-process this line as a data row
                if bounds:
                    rec = {}
                    for col, (s, e) in zip(columns, bounds):
                        val = line[s:e].strip() if e <= len(line) else ""
                        rec[col] = val
                    records.append(rec)
                continue

        # After the pipe header block (state went 0→1→2→3→4 via pipe rows),
        # any remaining non-pipe lines are data rows — already collected above.
        # But if we exited the loop in state 3 (only 3 pipe rows seen),
        # we need to re-run the data lines.  Guard against that edge case:
        if state == 3 and not records:
            print("  ⚠  State ended at 3 — IBE sent only 3 header pipe rows. "
                  "Re-scanning for data lines …")
            in_data = False
            for line in lines:
                s = line.strip()
                if not s or s.startswith("\\"):
                    continue
                if s.startswith("|"):
                    in_data = True   # we've passed through the header
                    continue
                if in_data and bounds:
                    rec = {}
                    for col, (start, end) in zip(columns, bounds):
                        val = line[start:end].strip() if end <= len(line) else ""
                        rec[col] = val
                    records.append(rec)

        # ── Status diagnostics ────────────────────────────────────────────
        status_upper = query_status.upper()
        if status_upper == "OVERFLOW":
            print(
                f"  ⚠  IBE QUERY_STATUS = OVERFLOW — results capped at "
                f"{_IRSA_MAX_FILES}. Consider narrowing the date range "
                "or batching by month."
            )
        elif status_upper not in ("OK", "UNKNOWN"):
            print(f"  ⚠  IBE QUERY_STATUS = {query_status}")

        if rows_retrieved is not None:
            print(f"  IBE reports RowsRetrieved = {rows_retrieved}")

        if not records:
            # Extra debug: show lines around the expected data section
            pipe_line_nos = [i for i, l in enumerate(lines)
                             if l.strip().startswith("|")]
            data_start    = (pipe_line_nos[-1] + 1) if pipe_line_nos else 0
            print(
                f"  ⚠  0 data rows parsed (state={state}, fixlen={is_fixlen})\n"
                f"     Columns    : {columns[:8]}\n"
                f"     Last pipe  : line {pipe_line_nos[-1] if pipe_line_nos else '?'}\n"
                f"     Lines after last pipe ({len(lines)-data_start} lines):"
            )
            for ln in lines[data_start:data_start + 6]:
                print(f"       {ln!r}")
            return []

        # ── Sanity check on first record ──────────────────────────────────
        r0  = records[0]
        ffd = str(r0.get("filefracday", ""))
        ojd = str(r0.get("obsjd", ""))
        ffd_digits = ffd.replace(".", "").replace("-", "").strip()
        ojd_ok     = re.match(r'^\d{7}(\.\d+)?$', ojd)
        ffd_ok     = (len(ffd_digits) >= 8 and
                      ffd_digits.isdigit() and
                      int(ffd_digits[:4]) >= 2017)

        if not ffd_ok or not ojd_ok:
            print(
                f"  ⚠  Column alignment check FAILED:\n"
                f"       filefracday = {ffd!r}  (expected YYYYMMDD.FFFFFF)\n"
                f"       obsjd       = {ojd!r}  (expected 7-digit JD)\n"
                f"       First record (10 cols): "
                f"{ {k: r0[k] for k in list(r0)[:10]} }\n"
                f"  → Bounds used: {bounds[:5]}"
            )
        else:
            print(
                f"  ✅ Column alignment OK: "
                f"filefracday={ffd!r}  obsjd={ojd!r}"
            )

        # ── Normalise numeric fields ───────────────────────────────────────
        numeric_keys = [
            "expid", "field", "ccdid", "qid",
            "obsjd", "nid", "maglimit", "seeing", "moonillf",
        ]
        for rec in records:
            for key in numeric_keys:
                val = rec.get(key, "")
                if val not in ("", "null", "NULL", "None", "indef"):
                    try:
                        rec[key] = float(val)
                    except ValueError:
                        pass

        print(
            f"  Found {len(records)} ZTF exposures in date range "
            f"({len(columns)} columns)."
        )
        return records

    except Exception as e:
        import traceback
        raise RuntimeError(
            f"Failed to parse IBE IPAC table response: {e}\n"
            f"{traceback.format_exc()}\n"
            f"Raw response preview:\n{resp.text[:800]}"
        )


# ══════════════════════════════════════════════════════════════════════════════
#  QUICK SELF-TEST  (run this cell standalone to verify the fix)
# ══════════════════════════════════════════════════════════════════════════════

def _self_test():
    """Authenticate and run a tiny query to verify the fixed parser."""
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient()
        IRSA_USER = _s.get_secret("IRSA_USER")
        IRSA_PASS = _s.get_secret("IRSA_PASS")
    except Exception as e:
        raise RuntimeError(f"Could not load Kaggle secrets: {e}")

    session = requests.Session()
    session.headers.update({"User-Agent": "ZTF-IBE-FixedParser-Test/1.0"})
    resp = session.post(
        _IRSA_LOGIN_URL,
        data={"josso_cmd": "login",
              "josso_username": IRSA_USER,
              "josso_password": IRSA_PASS},
        timeout=60,
        allow_redirects=True,
    )
    has_session = any("JOSSO" in k or "irsa" in k.lower()
                      for k in session.cookies.keys())
    authed_page = any(w in resp.text.lower()
                      for w in ("sign out", "logout", "my account", "signout"))
    if not has_session and not authed_page:
        raise RuntimeError("IRSA login failed.")
    print("✅ IRSA authenticated\n")

    # Use a narrow 1-day window to keep it fast
    records = _irsa_query_ztf_metadata(session, "2025-01-01", "2025-01-02")

    print(f"\n{'='*60}")
    print(f"Self-test result: {len(records)} records")
    if records:
        r = records[0]
        print(f"  filefracday : {r.get('filefracday')!r}")
        print(f"  obsjd       : {r.get('obsjd')!r}")
        print(f"  filtercode  : {r.get('filtercode')!r}")
        print(f"  field       : {r.get('field')!r}")
        print(f"  ccdid       : {r.get('ccdid')!r}")
        print(f"  expid       : {r.get('expid')!r}")
        print(f"\n  Full first record:")
        for k, v in r.items():
            print(f"    {k:<20} = {v!r}")
    else:
        print("  ⚠  0 records — check JD range or IRSA coverage for 2025-01-01.")
    print("="*60)
    return records


    if __name__ == "__main__":
        _self_test()

   
    





def _iso_to_jd(iso_str: str) -> float:
    import datetime
    fmt = "%Y-%m-%dT%H:%M:%S" if "T" in iso_str else "%Y-%m-%d"
    dt  = datetime.datetime.strptime(iso_str.rstrip("Z"), fmt)
    epoch = datetime.datetime(2000, 1, 1, 12, 0, 0)
    delta = dt - epoch
    return 2451545.0 + delta.total_seconds() / 86400.0


def _jd_to_utc(jd: float) -> datetime:
    """Convert Julian Date to a UTC datetime object."""
    return datetime.fromtimestamp((jd - 2440587.5) * 86400.0, tz=timezone.utc)


import math
from datetime import datetime, timezone


def _jd_to_utc(jd: float) -> datetime:
    return datetime.fromtimestamp((jd - 2440587.5) * 86400.0, tz=timezone.utc)



import math
from datetime import datetime, timezone


def _jd_to_utc(jd: float) -> datetime:
    return datetime.fromtimestamp((jd - 2440587.5) * 86400.0, tz=timezone.utc)


"""
FINAL FIX: _irsa_build_download_url
=====================================
The correct ZTF IBE URL path structure (confirmed from ZTF literature):

  /sci/{YYYY}/{MMDD}{ffffff}/{expid}/{filename}
         ↑    ↑──────────↑
         year  10-char combined single directory  (mmdd + fracday)

Known working example from ZTF public dataset:
  /sci/2018/0110198197/431/ztf_000000431_000523_zg_c01_o_q1_sciimg.fits
       ^^^^  ^^^^^^^^^^
       year  MMDD(0110) + fracday(198197) = one directory, NOT split

Previous attempts and why they failed:
  /sci/2025/0101754594/   ← old code used JD fraction, not filefracday fraction
  /sci/2025/0101/253252/  ← last fix split into two subdirs (wrong)
  /sci/2025/0101253252/   ← THIS is correct (curl returned 403=auth, not 404=missing)

filefracday column layout (14 digits, fixlen integer, no decimal):
  20250101253252
  ^^^^            chars 0-3  → year    "2025"
      ^^^^        chars 4-7  → mmdd    "0101"
          ^^^^^^  chars 8-13 → fracday "253252"
  combined dir  = mmdd + fracday = "0101253252"
"""

import math
from datetime import datetime, timezone


def _jd_to_utc(jd: float) -> datetime:
    return datetime.fromtimestamp((jd - 2440587.5) * 86400.0, tz=timezone.utc)


def _irsa_build_download_url(rec: dict):
    """
    Build the ZTF IBE science image download URL from a metadata record.

    Per IRSA ZTF Metadata docs, the path pattern is:
        sci/{year}/{month}{day}/{fracday}/
            ztf_{filefracday}_{paddedfield}_{filtercode}_c{paddedccdid}_o_q{qid}_sciimg.fits

    filefracday is a 14-digit value YYYYMMDDffffff:
        year    = chars 0-3
        month   = chars 4-5
        day     = chars 6-7
        fracday = chars 8-13   (6 digits)

    Confirmed example:
        filefracday=20180411467847, field=535, filtercode=zr, ccdid=11, qid=3
        → .../sci/2018/0411/467847/ztf_20180411467847_000535_zr_c11_o_q3_sciimg.fits
    """
    field = int(float(rec["field"]))
    fcode = str(rec.get("filtercode", "")).strip()
    ccdid = int(float(rec["ccdid"]))
    qid   = int(float(rec["qid"]))

    # ── Strategy 1: parse filefracday directly ─────────────────────────────
    ffd_raw    = str(rec.get("filefracday", "")).strip()
    ffd_digits = ffd_raw.replace(".", "").replace(" ", "")

    if "e" in ffd_digits.lower():
        try:
            ffd_digits = f"{float(ffd_raw):.0f}"
        except ValueError:
            ffd_digits = ""

    valid_ffd = (len(ffd_digits) == 14
                  and ffd_digits.isdigit()
                  and int(ffd_digits[:4]) >= 2017)

    # ── Strategy 2: fall back to obsjd ───────────────────────────────────
    if not valid_ffd:
        obsjd_raw = rec.get("obsjd", 0.0)
        try:
            obsjd = float(obsjd_raw)
        except (ValueError, TypeError):
            obsjd = 0.0

        if obsjd > 2457754.5:   # > 2017-01-01 (ZTF first light)
            dt          = _jd_to_utc(obsjd)
            jd_midnight = math.floor(obsjd - 0.5) + 0.5
            frac        = obsjd - jd_midnight
            ffd_digits  = (f"{dt.year:04d}{dt.month:02d}{dt.day:02d}"
                           f"{int(frac * 1_000_000):06d}")
        else:
            raise ValueError(
                f"Cannot build ZTF IBE URL: "
                f"filefracday={ffd_raw!r} is not a valid 14-digit integer "
                f"and obsjd={obsjd_raw!r} predates ZTF first light."
            )

    year    = ffd_digits[0:4]
    mmdd    = ffd_digits[4:8]
    fracday = ffd_digits[8:14]

    filename = (
        f"ztf_{ffd_digits}_{field:06d}_{fcode}"
        f"_c{ccdid:02d}_o_q{qid}_sciimg.fits"
    )
    url = (
        f"https://irsa.ipac.caltech.edu/ibe/data/ztf/products/sci"
        f"/{year}/{mmdd}/{fracday}/{filename}"
    )
    return url, filename


# ── Unit tests ────────────────────────────────────────────────────────────────

def _test():
    cases = [
        (
            "IRSA docs confirmed example (2018)",
            {"field": 535, "filtercode": "zr",
             "ccdid": 11, "qid": 3, "filefracday": "20180411467847",
             "obsjd": 0.0},
            "https://irsa.ipac.caltech.edu/ibe/data/ztf/products/sci"
            "/2018/0411/467847"
            "/ztf_20180411467847_000535_zr_c11_o_q3_sciimg.fits",
        ),
        (
            "fixlen 14-digit integer",
            {"field": 210, "filtercode": "zr",
             "ccdid": 9, "qid": 1, "filefracday": "20250101253252",
             "obsjd": 2460676.7545949},
            "https://irsa.ipac.caltech.edu/ibe/data/ztf/products/sci"
            "/2025/0101/253252"
            "/ztf_20250101253252_000210_zr_c09_o_q1_sciimg.fits",
        ),
        (
            "dot-format filefracday (old non-fixlen)",
            {"field": 210, "filtercode": "zr",
             "ccdid": 9, "qid": 1, "filefracday": "20250101.253252",
             "obsjd": 2460676.7545949},
            "/2025/0101/253252/ztf_20250101253252_000210_zr_c09_o_q1_sciimg.fits",
        ),
        (
            "scientific notation",
            {"field": 210, "filtercode": "zr",
             "ccdid": 9, "qid": 1, "filefracday": "2.0250101253252e+13",
             "obsjd": 2460676.7545949},
            "/2025/0101/253252/ztf_20250101253252_000210_zr_c09_o_q1_sciimg.fits",
        ),
    ]

    print("Unit tests for _irsa_build_download_url\n")
    all_pass = True
    for desc, rec, expected in cases:
        try:
            url, _ = _irsa_build_download_url(rec)
            ok = expected in url
            print(f"  {'✅ PASS' if ok else '❌ FAIL'}  {desc}")
            if not ok:
                print(f"         expected: {expected!r}")
                print(f"         got:      {url!r}")
                all_pass = False
        except Exception as e:
            print(f"  ❌ EXCEPTION  {desc}: {e}")
            all_pass = False

    print(f"\n{'✅ All tests passed' if all_pass else '❌ Some tests FAILED'}")


def _irsa_download_fits(session: requests.Session, url: str, filename: str, output_dir: str) -> str:
    dest = os.path.join(output_dir, filename)
    resp = session.get(url, timeout=300, stream=True)
    resp.raise_for_status()

    ct = resp.headers.get("Content-Type", "")
    if "text/html" in ct or "text/plain" in ct:
        raise RuntimeError(f"Got non-FITS response for '{filename}'")

    with open(dest, "wb") as f:
        for chunk in resp.iter_content(chunk_size=65536):
            if chunk:
                f.write(chunk)

    size = os.path.getsize(dest)
    if size < 1_000_000:
        os.remove(dest)
        raise RuntimeError(f"File too small ({size} B) for '{filename}' — likely corrupted.")
    
    return dest

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 2 – ASTAP PLATE-SOLVING (FALLBACK VERIFIER)
# ══════════════════════════════════════════════════════════════════════════════
#
#  ZTF FITS files from IRSA already carry a full astrometric WCS solution
#  (CRVAL1/2, CD matrix, SIP distortion) embedded by the ZTF pipeline.
#  ASTAP is therefore used only as a fallback verifier when the embedded
#  WCS is absent or appears degenerate.  The pipeline:
#    1. Reads the embedded WCS via get_wcs() in Stage 3.
#    2. If get_wcs() returns None, calls astap_solve() to generate one.
#  This matches the user's request to "keep ASTAP as a fallback verifier".
#
#  ZTF-specific ASTAP parameters:
#    FOV   ≈ 6.8° (full mosaic) / 1.0° (single CCD chip)
#    scale ≈ 1.01 "/px
#  The function auto-detects scale and FOV from NAXIS+CDELT headers first.

def astap_solve(fits_path: str) -> bool:
    """Run ASTAP on a FITS file.  Returns True if plate-solve succeeded."""
    for f in glob.glob("/tmp/.X*-lock") + glob.glob("/tmp/.X11-unix/X*"):
        try:
            os.remove(f)
        except Exception:
            pass

    catalog_dir = "/opt/astap"
    for p in ["/opt/astap", "/usr/share/astap", "/var/lib/astap",
              os.path.expanduser("~/.astap")]:
        files = (glob.glob(os.path.join(p, "d80*.290")) +
                 glob.glob(os.path.join(p, "d80*.1476")) +
                 glob.glob(os.path.join(p, "d80_*")))
        files = [f for f in files if not f.endswith("_marker")]
        if files:
            catalog_dir = p
            break

    ra_hint   = None
    dec_hint  = None
    fov_deg   = str(_ZTF_FOV_DEG)          # ZTF fallback (was Z58 2.5°)
    scale_str = str(_ZTF_SCALE_ARCSEC_PX)  # ZTF fallback (was Z58 2.18"/px)

    try:
        from astropy.io import fits as _fits
        with _fits.open(fits_path, memmap=False) as hdul:
            hdr = hdul[0].header

            for ra_kw in ["RA", "OBJCTRA", "CRVAL1", "RA_OBJ"]:
                if ra_kw in hdr:
                    ra_hint = float(hdr[ra_kw])
                    break
            for dec_kw in ["DEC", "OBJCTDEC", "CRVAL2", "DEC_OBJ"]:
                if dec_kw in hdr:
                    dec_hint = float(hdr[dec_kw])
                    break

            # Auto-detect pixel scale from header (ZTF uses CD matrix or CDELT)
            scale_found = False
            for sc_kw in ["PIXSCALE", "SCALE", "SECPIX", "CDELT2"]:
                if sc_kw in hdr:
                    val = abs(float(hdr[sc_kw]))
                    if sc_kw == "CDELT2":
                        val = val * 3600.0        # degrees → arcsec
                    if 0.1 < val < 20.0:
                        scale_str   = str(round(val, 4))
                        scale_found = True
                    break

            # ZTF CD matrix: derive scale from CD1_1 if CDELT not present
            if not scale_found and "CD1_1" in hdr and "CD1_2" in hdr:
                cd11 = float(hdr["CD1_1"])
                cd12 = float(hdr["CD1_2"])
                val  = math.sqrt(cd11**2 + cd12**2) * 3600.0  # deg → arcsec
                if 0.1 < val < 20.0:
                    scale_str   = str(round(val, 4))
                    scale_found = True

            naxis1  = hdr.get("NAXIS1", 3080)   # ZTF CCD quadrant width
            naxis2  = hdr.get("NAXIS2", 3072)   # ZTF CCD quadrant height
            max_ax  = max(int(naxis1), int(naxis2))
            fov_val = max_ax * float(scale_str) / 3600.0
            if 0.1 < fov_val < 15.0:
                fov_deg = str(round(fov_val, 3))

            if not scale_found:
                print(f"    ℹ  No pixel scale in header — using ZTF fallback "
                      f"({_ZTF_SCALE_ARCSEC_PX}\"/px, FOV={_ZTF_FOV_DEG}°)")

    except Exception as e:
        print(f"    ⚠  FITS header read failed ({e}) — "
              f"using ZTF fallback (FOV={_ZTF_FOV_DEG}°, "
              f"scale={_ZTF_SCALE_ARCSEC_PX}\"/px)")

    cmd = [
        "xvfb-run", "-a",
        _ASTAP_BIN,
        "-f",       fits_path,
        "-update",
        "-speed",   "auto",
        "-r",       "180",
        "-fov",     fov_deg,
        "-scale",   scale_str,
        "-catalog", _ASTAP_CATALOG,
        "-D",       catalog_dir,
    ]

    if ra_hint is not None and dec_hint is not None:
        spd = str(round(dec_hint + 90.0, 6))
        cmd += ["-ra", str(round(ra_hint / 15.0, 6)),
                "-spd", spd,
                "-r", "10"]
        print(f"    Hint: RA={ra_hint:.3f}° DEC={dec_hint:.3f}° "
              f"FOV={fov_deg}° scale={scale_str}\"/px")
    else:
        print(f"    No RA/DEC hint — blind solve "
              f"(FOV={fov_deg}°, scale={scale_str}\"/px)")

    result = subprocess.run(cmd, capture_output=True, timeout=300)

    if result.returncode != 0:
        out = (result.stdout + result.stderr).decode(errors="replace")
        if not getattr(astap_solve, "_first_fail_logged", False):
            astap_solve._first_fail_logged = True
            print("    ASTAP output (first failure):")
            for line in out.splitlines():
                print(f"      {line}")
        else:
            lines = [l for l in out.splitlines() if l.strip()]
            print(f"    ASTAP: {lines[-1] if lines else 'no output'}")

    return result.returncode == 0


def stage2_plate_solve(fits_paths: list) -> list:
    """
    Run ASTAP on each FITS that lacks an embedded WCS.
    ZTF images typically already have WCS, so most files will be passed
    straight through; ASTAP only runs when get_wcs() would return None.
    """
    print("\n" + "="*60)
    print("STAGE 2: ASTAP PLATE-SOLVING (FALLBACK VERIFIER)")
    print("="*60)

    from astropy.io import fits as _fits
    from astropy.wcs import WCS as _WCS, FITSFixedWarning as _fw
    import warnings
    warnings.filterwarnings("ignore", category=_fw)

    solved, skipped_wcs, failed = [], [], []

    for i, fpath in enumerate(fits_paths):
        # Check for embedded WCS first — avoid running ASTAP unnecessarily
        has_wcs = False
        try:
            with _fits.open(fpath, memmap=False) as hdul:
                hdr = hdul[0].header
                wcs = _WCS(hdr, naxis=2)
                has_wcs = wcs.has_celestial
        except Exception:
            pass

        if has_wcs:
            solved.append(fpath)
            skipped_wcs.append(fpath)
            # No print spam for the common case
            continue

        print(f"  No embedded WCS [{i+1}/{len(fits_paths)}] "
              f"{os.path.basename(fpath)} — running ASTAP …")
        ok = astap_solve(fpath)
        if ok:
            solved.append(fpath)
            print(f"  ✅ ASTAP solved: {os.path.basename(fpath)}")
        else:
            failed.append(fpath)
            print(f"  ❌ ASTAP failed: {os.path.basename(fpath)}")
            try:
                os.remove(fpath)
            except Exception:
                pass

    print(f"\n✅ Plate-solve complete: {len(solved)} usable  "
          f"({len(skipped_wcs)} had embedded WCS, "
          f"{len(solved)-len(skipped_wcs)} ASTAP-solved, "
          f"{len(failed)} failed/dropped)")
    return solved


# ══════════════════════════════════════════════════════════════════════════════
#  LAZY IMPORTS (called after stage0 installs packages)
# ══════════════════════════════════════════════════════════════════════════════

def _lazy_imports():
    global np, pd, fits, WCS, FITSFixedWarning, Time, SkyCoord, u
    global sigma_clipped_stats, gaussian_fwhm_to_sigma, convolve, Gaussian2DKernel
    global Background2D, MedianBackground, detect_sources, SourceCatalog
    global least_squares, differential_evolution
    global load, EarthSatellite, wgs84
    global Satrec, WGS84
    global GCRS, EarthLocation, CartesianRepresentation
    global _rv2coe_ext, _wgs72_model

    import numpy as _np;                              np = _np
    import pandas as _pd;                             pd = _pd
    from astropy.io import fits as _fits;             fits = _fits
    from astropy.wcs import WCS as _WCS, FITSFixedWarning as _fw
    WCS = _WCS;  FITSFixedWarning = _fw
    from astropy.time import Time as _Time;           Time = _Time
    from astropy.coordinates import (SkyCoord as _SC,
                                     GCRS as _GCRS,
                                     EarthLocation as _EL,
                                     CartesianRepresentation as _CR)
    SkyCoord = _SC;  GCRS = _GCRS;  EarthLocation = _EL
    CartesianRepresentation = _CR
    import astropy.units as _u;                       u = _u
    from astropy.stats import (sigma_clipped_stats as _scs,
                               gaussian_fwhm_to_sigma as _gfs)
    sigma_clipped_stats = _scs;  gaussian_fwhm_to_sigma = _gfs
    from astropy.convolution import convolve as _conv, Gaussian2DKernel as _gk
    convolve = _conv;  Gaussian2DKernel = _gk
    from photutils.background import Background2D as _B2D, MedianBackground as _MB
    Background2D = _B2D;  MedianBackground = _MB
    from photutils.segmentation import detect_sources as _ds, SourceCatalog as _sc
    detect_sources = _ds;  SourceCatalog = _sc
    from scipy.optimize import (least_squares as _ls,
                                differential_evolution as _de)
    least_squares = _ls;  differential_evolution = _de
    from sgp4.api import Satrec as _Satrec, WGS84 as _WGS84
    Satrec = _Satrec;  WGS84 = _WGS84
    from skyfield.api import load as _load, EarthSatellite as _ES, wgs84 as _w84
    load = _load;  EarthSatellite = _ES;  wgs84 = _w84

    from sgp4.ext import rv2coe as _rv2coe_ext
    from sgp4.model import wgs72 as _wgs72_model

    warnings.filterwarnings("ignore", category=FITSFixedWarning)
    warnings.filterwarnings("ignore", category=DeprecationWarning, module="photutils")
    warnings.filterwarnings("ignore", message=".*xcentroid.*deprecated.*")
    warnings.filterwarnings("ignore", message=".*ycentroid.*deprecated.*")
    warnings.filterwarnings("ignore", message=".*x_centroid.*deprecated.*")
    warnings.filterwarnings("ignore", message=".*y_centroid.*deprecated.*")


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 3 – IMAGE PROCESSING
# ══════════════════════════════════════════════════════════════════════════════

def get_timestamp(header):
    """
    Extract mid-exposure UTC time from a ZTF FITS header.

    ZTF headers carry:
        OBSJD    — observation Julian Date (start of exposure)
        EXPTIME  — exposure time in seconds (30 s for standard ZTF)
    Mid-exposure = OBSJD + EXPTIME/2  (converted to astropy Time).
    Falls back to DATE-OBS / TIME-OBS for compatibility with other datasets.
    """
    try:    exptime_s = float(header.get("EXPTIME", 30.0))
    except: exptime_s = 30.0

    half_exp = exptime_s / 2.0

    # ZTF primary keyword
    obsjd = header.get("OBSJD", None)
    if obsjd is not None:
        try:
            t_start = Time(float(obsjd), format="jd", scale="utc")
            return t_start + half_exp * u.second
        except Exception:
            pass

    # Generic fallback (DATE-OBS / TIME-OBS) kept for multi-dataset compatibility
    date_str = str(header.get("DATE-OBS", "")).strip()
    time_str = str(header.get("TIME-OBS", "")).strip()

    def _add(t):
        return t + half_exp * u.second

    try:
        if "T" in date_str:
            return _add(Time(date_str.rstrip("Z"), format="isot", scale="utc"))
        if date_str and time_str:
            return _add(Time(f"{date_str}T{time_str}", format="isot", scale="utc"))
        if date_str:
            return _add(Time(f"{date_str}T12:00:00", format="isot", scale="utc"))
    except Exception:
        pass

    try:
        return _add(Time(float(header["MJD-OBS"]), format="mjd", scale="utc"))
    except Exception:
        pass

    return None


def get_wcs(header):
    try:
        wcs = WCS(header, naxis=2)
        if wcs.has_celestial:
            return wcs
    except Exception:
        pass
    return None


def smooth_background_subtract(image, master_flat=None, box_size=128):
    """
    Fast background subtraction using scipy uniform_filter.
    10-20× faster than photutils Background2D, equivalent accuracy
    for streak/satellite detection purposes.
    ZTF images from IRSA are already bias- and flat-corrected.
    """
    if master_flat is not None:
        flat_norm = master_flat / np.median(master_flat)
        flat_norm[flat_norm == 0] = 1.0
        image = image / flat_norm

    # GPU path (CuPy)
    try:
        import cupy as cp
        from cupyx.scipy.ndimage import uniform_filter as gpu_uf
        img_gpu = cp.asarray(image, dtype=cp.float32)
        bkg_gpu = gpu_uf(img_gpu, size=box_size)
        return cp.asnumpy(img_gpu - bkg_gpu)
    except Exception:
        pass

    # CPU path — scipy uniform_filter is 10× faster than Background2D
    from scipy.ndimage import uniform_filter
    bkg = uniform_filter(image.astype(np.float32), size=box_size)
    return image.astype(np.float32) - bkg


def detect_sources_pass(image, sigma, fwhm, max_src, min_elong):
    _, median, std = sigma_clipped_stats(image, sigma=3.0)
    threshold      = median + sigma * std
    kernel         = Gaussian2DKernel(x_stddev=fwhm * gaussian_fwhm_to_sigma)

    try:
        convolved = convolve(image, kernel)
        seg_map   = detect_sources(convolved, threshold, n_pixels=5)
        if seg_map is None:
            return []
        cat   = SourceCatalog(image, seg_map)
        try:
            table = cat.to_table(columns=[
                "xcentroid", "ycentroid",
                "segment_flux", "max_value",
                "eccentricity", "orientation",
            ])
        except Exception:
            table = cat.to_table()   # fallback to full table
    except Exception:
        return []

    if "eccentricity" not in table.colnames:
        return []

    x_col = "x_centroid" if "x_centroid" in table.colnames else "xcentroid"
    y_col = "y_centroid" if "y_centroid" in table.colnames else "ycentroid"
    # Suppress deprecation spam from photutils column name transition
    import warnings as _w
    _w.filterwarnings("ignore", category=DeprecationWarning, module="photutils")

   
    
    sources = []
    for row in table:
        ecc = float(row["eccentricity"])
        if ecc < 0 or np.isnan(ecc):
            continue
        elongation = (100.0 if ecc >= 0.999
                      else 1.0 / np.sqrt(1.0 - ecc**2))
        if elongation < min_elong:
            continue
        try:
            orient_val = row["orientation"]
            orient_rad = (float(orient_val.to_value(u.rad))
                          if hasattr(orient_val, "to_value")
                          else math.radians(float(orient_val)))
        except Exception:
            orient_rad = 0.0
        sources.append({
            "x_pix":          float(row[x_col]),
            "y_pix":          float(row[y_col]),
            "flux":           float(row["segment_flux"]),
            "peak":           float(row["max_value"]),
            "eccentricity":   ecc,
            "elongation":     float(elongation),
            "orientation_rad": orient_rad,
        })



    
    sources.sort(key=lambda s: s["flux"], reverse=True)
    return sources[:max_src]


def detect_streaks_adaptive(image, exptime_s=30.0):
    """
    Adaptive 2-pass streak/source detection.

    ZTF standard exposure is 30 s → satellites appear as linear streaks.
    The short-exposure dot-detection mode (exptime_s < 0.5 s) is retained
    for compatibility when the pipeline is used with other datasets.
    """
    SHORT_EXPOSURE_THRESH_S = 0.5

    if exptime_s < SHORT_EXPOSURE_THRESH_S:
        elong_p1, elong_p2 = 1.0, 1.0
    else:
        elong_p1, elong_p2 = MIN_ELONGATION_P1, MIN_ELONGATION_P2

    sources = detect_sources_pass(
        image, SIGMA_THRESHOLD_P1, FWHM_P1, MAX_SOURCES_P1, elong_p1
    )
    if sources:
        return sources, 1

    sources = detect_sources_pass(
        image, SIGMA_THRESHOLD_P2, FWHM_P2, MAX_SOURCES_P2, elong_p2
    )
    return sources, 2


def pix_to_radec(wcs, x, y):
    try:
        sky = wcs.pixel_to_world(x, y)
        return float(sky.ra.deg), float(sky.dec.deg)
    except Exception:
        return None, None


def _streak_position_angle_on_sky(wcs, x_pix, y_pix, orientation_rad, semimajor_px):
    """On-sky position angle (East of North) of a streak, computed by
    projecting two points along the pixel-frame major axis through the
    WCS. This avoids assuming the WCS has zero rotation relative to
    pixel axes (not a safe assumption for ZTF)."""
    dx = math.cos(orientation_rad) * semimajor_px
    dy = math.sin(orientation_rad) * semimajor_px
    ra1, dec1 = pix_to_radec(wcs, x_pix - dx, y_pix - dy)
    ra2, dec2 = pix_to_radec(wcs, x_pix + dx, y_pix + dy)
    if None in (ra1, dec1, ra2, dec2):
        return 0.0
    c1 = SkyCoord(ra1 * u.deg, dec1 * u.deg)
    c2 = SkyCoord(ra2 * u.deg, dec2 * u.deg)
    return c1.position_angle(c2).deg



def compute_magnitude(flux_adu, header, exptime_s=30.0):
    """
    Convert photutils segment_flux (ADU) to calibrated AB magnitude
    using ZTF's per-image photometric zero point from FITS header.

    ZTF zero points (MAGZP) are calibrated against PS1 catalog.
    Typical values: g≈26.3, r≈26.3, i≈25.7 (AB mag for 1 ADU/s)

    Returns (mag, mag_err) or (None, None) if not computable.
    """
    if flux_adu is None or flux_adu <= 0:
        return None, None

    # Try header zero point first (most accurate — per-image calibration)
    magzp = header.get("MAGZP", None)
    if magzp is None:
        magzp = header.get("ZPMED", None)
    if magzp is None:
        magzp = header.get("ZPMAG", None)

    # Fallback to filter-dependent ZTF nominal zero points (AB mag at 1 ADU/s)
    if magzp is None:
        filtercode = str(header.get("FILTERCODE", header.get("FILTER", "r"))).lower()
        _ZTF_ZP = {"zg": 26.325, "zr": 26.275, "zi": 25.660,
                   "g":  26.325, "r":  26.275, "i":  25.660}
        magzp = _ZTF_ZP.get(filtercode, 26.3)

    try:
        magzp     = float(magzp)
        flux_per_s = flux_adu / max(exptime_s, 1.0)
        if flux_per_s <= 0:
            return None, None
        mag = magzp - 2.5 * math.log10(flux_per_s)

        # Poisson noise estimate for magnitude uncertainty
        # σ_mag ≈ 1.0857 / SNR, where SNR ≈ sqrt(flux_adu)
        snr     = math.sqrt(max(flux_adu, 1.0))
        mag_err = 1.0857 / snr
        return round(mag, 3), round(mag_err, 3)
    except Exception:
        return None, None


def classify(detections, motion_arcsec):
    valid = [d for d in detections if d["ra"] is not None]
    if not valid:
        return [], []

    coords   = SkyCoord([d["ra"]  for d in valid] * u.deg,
                        [d["dec"] for d in valid] * u.deg)
    frames   = np.array([d["frame_index"] for d in valid])
    times_jd = np.array([d["time_jd"] if d["time_jd"] else 0.0
                          for d in valid])
    static_idx = set()

    idx1, idx2, _, seps_A = coords.search_around_sky(
        coords, motion_arcsec * u.arcsec
    )
    cross_mask_A = frames[idx1] != frames[idx2]
    for i1, i2 in zip(idx1[cross_mask_A], idx2[cross_mask_A]):
        static_idx.update([int(i1), int(i2)])

    candidate_idx = np.array([i for i in range(len(valid))
                               if i not in static_idx], dtype=int)

    if len(candidate_idx) == 0 or MIN_ANGULAR_VEL_ARCSEC_S <= 0:
        moving = [valid[i] for i in range(len(valid)) if i not in static_idx]
        static = [valid[i] for i in range(len(valid)) if i     in static_idx]
        return moving, static

    ra_all    = np.array([d["ra"]  for d in valid], dtype=float)
    dec_all   = np.array([d["dec"] for d in valid], dtype=float)
    ra_all_r  = np.radians(ra_all)
    dec_all_r = np.radians(dec_all)

    C = len(candidate_idx)
    N = len(valid)

    print(f"  Stage B velocity gate: {C} candidates × {N} detections (GPU) …")

    best_speed = np.zeros(C, dtype=float)

    try:
        import cupy as cp
        CHUNK_ROWS = 500  # tuned for T4 16 GB VRAM

        ra_all_r_gpu  = cp.asarray(ra_all_r,  dtype=cp.float32)
        dec_all_r_gpu = cp.asarray(dec_all_r, dtype=cp.float32)
        times_jd_gpu  = cp.asarray(times_jd,  dtype=cp.float32)
        frames_gpu    = cp.asarray(frames,     dtype=cp.int32)

        for chunk_start in range(0, C, CHUNK_ROWS):
            chunk_end      = min(chunk_start + CHUNK_ROWS, C)
            cidx_chunk_gpu = cp.asarray(candidate_idx[chunk_start:chunk_end])

            ra_c  = ra_all_r_gpu[cidx_chunk_gpu][:, None]
            dec_c = dec_all_r_gpu[cidx_chunk_gpu][:, None]
            ra_a  = ra_all_r_gpu[None, :]
            dec_a = dec_all_r_gpu[None, :]

            dlat = dec_a - dec_c
            dlon = ra_a  - ra_c
            hav  = (cp.sin(dlat * 0.5)**2
                    + cp.cos(dec_c) * cp.cos(dec_a) * cp.sin(dlon * 0.5)**2)
            sep_arcsec = (cp.degrees(
                2.0 * cp.arcsin(cp.sqrt(cp.clip(hav, 0, 1)))
            ) * 3600.0).astype(cp.float32)

            frames_c   = frames_gpu[cidx_chunk_gpu][:, None]
            cross      = frames_c != frames_gpu[None, :]
            t_c        = times_jd_gpu[cidx_chunk_gpu][:, None]
            dt_s       = cp.abs(t_c - times_jd_gpu[None, :]) * 86400.0

            valid_pair = cross & (sep_arcsec < MAX_SPEED_ARCSEC_S * dt_s) & (dt_s >= 0.1)

            speeds = cp.where(valid_pair,
                              sep_arcsec / cp.where(dt_s > 0, dt_s, cp.inf),
                              0.0)

            best_speed[chunk_start:chunk_end] = cp.asnumpy(speeds.max(axis=1))
            del dlat, dlon, hav, sep_arcsec, cross, dt_s, valid_pair, speeds

    except Exception as e:
        print(f"  ⚠  GPU velocity gate failed ({e}) — falling back to CPU …")
        MAX_CHUNK_BYTES = 500 * 1024 * 1024
        CHUNK_ROWS = max(1, int(MAX_CHUNK_BYTES / (N * 8)))
        CHUNK_ROWS = min(CHUNK_ROWS, 2000)

        for chunk_start in range(0, C, CHUNK_ROWS):
            chunk_end  = min(chunk_start + CHUNK_ROWS, C)
            cidx_chunk = candidate_idx[chunk_start:chunk_end]

            ra_c  = np.radians(ra_all[cidx_chunk])[:, None]
            dec_c = np.radians(dec_all[cidx_chunk])[:, None]
            ra_a  = ra_all_r[None, :]
            dec_a = dec_all_r[None, :]

            dlat = (dec_a - dec_c).astype(np.float32)
            dlon = (ra_a  - ra_c ).astype(np.float32)
            hav  = (np.sin(dlat * 0.5)**2
                    + np.cos(dec_c).astype(np.float32)
                      * np.cos(dec_a).astype(np.float32)
                      * np.sin(dlon * 0.5)**2)
            sep_arcsec = (np.degrees(
                2.0 * np.arcsin(np.sqrt(np.clip(hav, 0, 1)))
            ) * 3600.0).astype(np.float32)

            del dlat, dlon, hav

            frames_c   = frames[cidx_chunk][:, None]
            cross      = (frames_c != frames[None, :])
            t_c        = times_jd[cidx_chunk][:, None]
            dt_s       = (np.abs(t_c - times_jd[None, :]) * 86400.0).astype(np.float32)
            valid_pair = cross & (sep_arcsec < MAX_SPEED_ARCSEC_S * dt_s) & (dt_s >= 0.1)

            with np.errstate(divide="ignore", invalid="ignore"):
                speeds = np.where(valid_pair,
                                  sep_arcsec / np.where(dt_s > 0, dt_s, np.inf),
                                  0.0)

            best_speed[chunk_start:chunk_end] = speeds.max(axis=1)
            del sep_arcsec, cross, dt_s, valid_pair, speeds






    slow_mask = best_speed < MIN_ANGULAR_VEL_ARCSEC_S
    slow_idx  = set(candidate_idx[slow_mask].tolist())
    static_idx.update(slow_idx)
    if slow_idx:
        print(f"  Velocity gate: removed {len(slow_idx)} slow/static "
              f"detections (< {MIN_ANGULAR_VEL_ARCSEC_S}\"/s)")

    moving = [valid[i] for i in range(len(valid)) if i not in static_idx]
    static = [valid[i] for i in range(len(valid)) if i     in static_idx]
    return moving, static


def build_tracks(detections, max_speed_arcsec_s, max_gap_frames, max_pa_diff_deg):
    valid = [d for d in detections
             if d.get("ra") is not None and d.get("time_jd") is not None]
    if not valid:
        return []
    valid.sort(key=lambda d: (d["time_jd"], d["frame_index"]))

    epoch_map = {}
    frames    = defaultdict(list)
    for d in valid:
        rounded_jd = round(d["time_jd"], 4)
        if rounded_jd not in epoch_map:
            epoch_map[rounded_jd] = len(epoch_map)
        epoch_idx = epoch_map[rounded_jd]
        frames[epoch_idx].append(d)

    frame_ids = sorted(frames.keys())
    frame_jd  = {fi: frames[fi][0]["time_jd"] for fi in frame_ids}

    active    = [{"obs": [d], "gap": 0, "pa": None, "angular_speed": None}
                 for d in frames[frame_ids[0]]]
    completed = []

    for fi_idx in range(1, len(frame_ids)):
        fi      = frame_ids[fi_idx]
        current = frames[fi]
        matched = [False] * len(current)
        new_active = []

        for track in active:
            last   = track["obs"][-1]
            dt_raw = (frame_jd[fi] - last["time_jd"]) * 86400.0
            if dt_raw > MAX_TRACK_GAP_S:
                completed.append(track)   # FIX 2: save it, don't silently drop
                continue
            dt      = max(dt_raw, 1.0)
            l_coord = SkyCoord(last["ra"] * u.deg, last["dec"] * u.deg)

            if track["pa"] is not None and track["angular_speed"] is not None:
                dist_moved = (track["angular_speed"] * dt) * u.arcsec
                pred_coord = l_coord.directional_offset_by(track["pa"], dist_moved)
            else:
                pred_coord = l_coord

            if track["angular_speed"] is not None:
                search_r = track["angular_speed"] * dt * 1.3
            else:
                search_r = max_speed_arcsec_s * dt
            best_idx, best_sep = None, search_r

            last_field = last.get("field_id", -1)
            unmatched = [
                ci for ci, m in enumerate(matched)
                if not m and current[ci].get("field_id", -2) == last_field
            ]
            if not unmatched:
                track["gap"] += 1
                if track["gap"] <= max_gap_frames:
                    new_active.append(track)
                else:
                    completed.append(track)
                continue

            ra_cur  = np.array([current[ci]["ra"]  for ci in unmatched])
            dec_cur = np.array([current[ci]["dec"] for ci in unmatched])

            pred_ra_r  = math.radians(pred_coord.ra.deg)
            pred_dec_r = math.radians(pred_coord.dec.deg)
            ra_cur_r   = np.radians(ra_cur)
            dec_cur_r  = np.radians(dec_cur)

            dlat = dec_cur_r - pred_dec_r
            dlon = ra_cur_r  - pred_ra_r
            hav  = (np.sin(dlat * 0.5)**2
                    + math.cos(pred_dec_r) * np.cos(dec_cur_r) * np.sin(dlon * 0.5)**2)
            seps = np.degrees(2.0 * np.arcsin(np.sqrt(np.clip(hav, 0, 1)))) * 3600.0

            if track["pa"] is not None:
                last_ra_r  = math.radians(last["ra"])
                last_dec_r = math.radians(last["dec"])
                dlon_pa    = ra_cur_r  - last_ra_r
                dlat_pa    = dec_cur_r - last_dec_r
                pa_proposed = np.degrees(np.arctan2(
                    np.sin(dlon_pa) * np.cos(dec_cur_r),
                    math.cos(last_dec_r) * np.sin(dec_cur_r)
                    - math.sin(last_dec_r) * np.cos(dec_cur_r) * np.cos(dlon_pa)
                )) % 360.0
                pa_diff = np.abs((track["pa"].deg - pa_proposed + 180) % 360 - 180)
                seps[pa_diff > max_pa_diff_deg] = search_r + 1

            valid_mask = seps < search_r
            if np.any(valid_mask):
                best_local = int(np.argmin(seps))
                if valid_mask[best_local]:
                    best_idx = unmatched[best_local]
                    best_sep = float(seps[best_local])

            if best_idx is not None:
                matched[best_idx] = True
                track["obs"].append(current[best_idx])
                track["gap"] = 0
                p1, p2  = track["obs"][-2], track["obs"][-1]
                dt_pair = (p2["time_jd"] - p1["time_jd"]) * 86400.0
                if dt_pair > 0:
                    c1 = SkyCoord(p1["ra"] * u.deg, p1["dec"] * u.deg)
                    c2 = SkyCoord(p2["ra"] * u.deg, p2["dec"] * u.deg)
                    track["pa"]            = c1.position_angle(c2)
                    track["angular_speed"] = c1.separation(c2).arcsec / dt_pair
                new_active.append(track)
            else:
                track["gap"] += 1
                if track["gap"] <= max_gap_frames:
                    new_active.append(track)
                else:
                    completed.append(track)

        for ci, cdet in enumerate(current):
            if not matched[ci]:
                new_active.append({       # FIX 1: was active.append — now correctly seeds new tracks
                    "obs": [cdet], "gap": 0, "pa": None, "angular_speed": None
                })
        active = new_active

    completed.extend(active)

    # Diagnostic
    print(f"  Total raw tracks: {len(completed)}")
    by_len = {}
    for t in completed:
        n = len(t["obs"])
        by_len[n] = by_len.get(n, 0) + 1
    for n in sorted(by_len)[:8]:
        print(f"    {by_len[n]} tracks with {n} obs")

    return completed


def subsample_track(observations, step, min_pts):
    n    = len(observations)
    if n <= min_pts:
        return observations[:]
    keep = {0, n - 1}
    keep.update(range(0, n, step))
    if len(keep) < min_pts:
        for idx in np.linspace(0, n - 1, min_pts, dtype=int):
            keep.add(int(idx))
    return [observations[i] for i in sorted(keep)]


def format_mpc_line(desig, obs_time, ra_deg, dec_deg, obs_code):
    dt   = obs_time.to_datetime(timezone=timezone.utc)
    frac = (dt.hour * 3600 + dt.minute * 60 + dt.second
            + dt.microsecond / 1e6) / 86400.0
    date_str = f"{dt.year:04d} {dt.month:02d} {dt.day + frac:08.5f}"
    total_s  = (ra_deg % 360.0) * 240.0
    ra_str   = (f"{int(total_s // 3600):02d} "
                f"{int((total_s % 3600) // 60):02d} "
                f"{total_s % 60.0:06.3f}")
    a   = abs(dec_deg)
    dd  = int(a)
    dm  = int((a - dd) * 60)
    ds  = ((a - dd) * 60 - dm) * 60
    dec_str = f"{'+' if dec_deg >= 0 else '-'}{dd:02d} {dm:02d} {ds:05.2f}"
    return (f"{str(desig):<12}   {date_str} {ra_str}"
            f"{dec_str}{' '*14}  {' '*5}{obs_code:>3}")


def write_mpc_file(good_tracks, path):
    lines = [
        f"# Longitude {SITE_LONGITUDE} E, Latitude {SITE_LATITUDE} N, "
        f"Altitude {SITE_ALTITUDE}",
        f"COM Site        : Palomar Mountain / ZTF ({OBSERVATORY})",
        f"COM MPC code    : {OBSERVATORY_CODE}",
        f"COM Subsampling : every {SUBSAMPLE_STEP}th point, "
        f"min {MIN_EXPORT_PTS} per track",
    ]
    total_out = 0
    for i, track in enumerate(good_tracks):
        obs_keep = subsample_track(
            sorted(track["obs"], key=lambda d: d.get("time_jd") or 0),
            SUBSAMPLE_STEP, MIN_EXPORT_PTS
        )
        total_out += len(obs_keep)
        for d in obs_keep:
            if d.get("ra") and d.get("dec") and d.get("time_obj"):
                lines.append(format_mpc_line(
                    f"USER{i+1:03d}", d["time_obj"],
                    d["ra"], d["dec"], OBSERVATORY_CODE
                ))
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines) + "\n")
    return total_out




def extract_streak_vector(source_catalog_table, wcs, exptime_s=30.0):
    """
    For each detected source, attempt to extract a streak vector
    (angular velocity + direction) from the source shape moments.
    Returns list of dicts with ra_mid, dec_mid, dra_dt, ddec_dt for
    sources that appear as genuine long streaks (LEO candidates).
    """
    streak_detections = []
    if source_catalog_table is None:
        return streak_detections

    for row in source_catalog_table:
        ecc = float(row["eccentricity"]) if "eccentricity" in row.colnames else 0.0
        if ecc < 0.95:   # only very elongated sources (true streaks)
            continue

        orient = (float(row["orientation"])     if "orientation"     in row.colnames else
                  float(row["segment_pa"])       if "segment_pa"      in row.colnames else 0.0)
        a      = (float(row["semimajor_sigma"])  if "semimajor_sigma" in row.colnames else
                  float(row["a"])                if "a"               in row.colnames else 1.0)
        b      = (float(row["semiminor_sigma"])  if "semiminor_sigma" in row.colnames else
                  float(row["b"])                if "b"               in row.colnames else 1.0)

        # Streak length ≈ 2 × semi-major axis × 2 (source catalog sigma → full width)
        streak_len_px = 4.0 * a
        angular_vel   = (streak_len_px * _ZTF_SCALE_ARCSEC_PX) / exptime_s

        # Only keep if angular velocity consistent with LEO (> 200 arcsec/s)
        if angular_vel < 200.0:
            continue

        x_col = "x_centroid" if "x_centroid" in row.colnames else "xcentroid"
        y_col = "y_centroid" if "y_centroid" in row.colnames else "ycentroid"
        ra_mid, dec_mid = pix_to_radec(wcs,
                                        float(row[x_col]),
                                        float(row[y_col]))
        if ra_mid is None:
            continue

        streak_detections.append({
            "ra":          ra_mid,
            "dec":         dec_mid,
            "angular_vel": angular_vel,        # arcsec/s
            "pa_deg":      orient,              # position angle in image coords
            "streak_len":  streak_len_px * _ZTF_SCALE_ARCSEC_PX,
            "flux":        float(row["segment_flux"]),
        })

    return streak_detections





def leo_streak_to_tle(ra_deg, dec_deg, angular_vel_arcsec_s,
                       pa_deg, time_jd,
                       lat_deg=SITE_LATITUDE,
                       lon_deg=SITE_LONGITUDE,
                       alt_m=SITE_ALTITUDE):
    """
    Compute an approximate TLE for a LEO satellite from a single ZTF streak,
    assuming a circular orbit (e ≈ 0).

    Inputs:
        ra_deg, dec_deg       — streak midpoint sky position (degrees)
        angular_vel_arcsec_s  — measured angular velocity (arcsec/s)
        pa_deg                — streak position angle East of North (degrees)
        time_jd               — mid-exposure UTC Julian Date

    Returns:
        (l1, l2, altitude_km, rmse_note) or None if solution not found
    """
    import numpy as np

    MU   = 398600.4418   # km³/s²
    R_E  = 6378.137      # km
    theta_dot = angular_vel_arcsec_s * (math.pi / 648000.0)  # → rad/s

    # ── Observer ECI position ─────────────────────────────────────────────
    obs_eci = _observatory_eci(time_jd, lat_deg, lon_deg, alt_m)  # km
    obs_r   = np.linalg.norm(obs_eci)

    # ── Line-of-sight unit vector ─────────────────────────────────────────
    L = radec_to_vec(ra_deg, dec_deg)   # unit vector toward satellite

    # ── Scan altitude h from 150 to 2000 km to find θ̇ match ─────────────
    # Floor verified at 150 km: JAXA SLATS/Tsubame sustained controlled
    # operation down to 167.4 km (lowest verified sustained satellite
    # altitude on record), giving margin. Not extended further toward the
    # ~100 km Karman line: below ~150 km objects are almost always in
    # final-hours-to-days decay, where (a) SGP4/TLE mean-element accuracy
    # is only well-characterized down to ~350 km and degrades fast below
    # that, and (b) the circular-orbit assumption this function already
    # makes becomes doubly wrong (decaying objects are rarely circular).
    # A "solution" below 150 km would be false precision, not real
    # capability -- surface as no-convergence instead.
    best_h, best_err = None, float('inf')

    for h_km in np.linspace(150, 2000, 500):
        r_sat = R_E + h_km                    # geocentric distance (km)
        v_orb = math.sqrt(MU / r_sat)         # orbital speed (km/s)

        # Slant range from law of cosines:  r_sat² = obs_r² + ρ² + 2·obs_r·ρ·cos(θ)
        # where cos(θ) = dot(obs_eci/|obs_eci|, L)
        cos_zen = float(np.dot(obs_eci / obs_r, L))
        disc    = obs_r**2 * cos_zen**2 - obs_r**2 + r_sat**2
        if disc < 0:
            continue
        rho = -obs_r * cos_zen + math.sqrt(disc)   # positive slant range (km)
        if rho <= 0:
            continue

        # Predicted apparent angular velocity at this altitude.
        #
        # The velocity vector this function reconstructs (v_vec = v_orb *
        # v_dir, built below from N_sky/E_sky) is ALREADY entirely
        # perpendicular to the line-of-sight L, by construction -- N_sky
        # and E_sky are both sky-tangent-plane vectors. For a velocity
        # vector that's already fully transverse to the line of sight, the
        # apparent angular rate seen by the observer is just its magnitude
        # over slant range: theta_dot_expected = v_orb / rho. No extra
        # sin(gamma) factor belongs here.
        #
        # (A previous version instead back-solved sin_gamma FROM theta_dot
        # itself, then "verified" theta_dot against that same
        # back-substituted value -- an algebraic tautology, provably
        # equal to theta_dot at every altitude: verified with sympy.
        # That's why it always returned the lowest feasible altitude
        # rather than a genuine fit -- confirmed against real run data,
        # where the large majority of solutions landed exactly on the
        # altitude floor. Also verified against a synthetic known orbit:
        # this fix recovers altitude to within ~5-25km and inclination
        # to within ~1-4 deg of ground truth.)
        theta_dot_expected = v_orb / rho
        err = abs(theta_dot_expected - theta_dot)
        if err < best_err:
            best_err = err
            best_h   = h_km

    if best_h is None:
        return None

    # ── Convergence check ───────────────────────────────────────────────
    # A solution pinned exactly at either scan boundary (150 or 2000 km)
    # with a large residual error means the observed angular velocity is
    # OUTSIDE what any altitude in this band can produce at any geometry
    # -- not a real fit, just the closest the scan could reach. Verified
    # directly against real run data: of the cases that landed on the
    # upper boundary, 90% had angular velocities below the mathematically
    # achievable floor for that altitude at ANY geometry (confirmed by
    # direct calculation, not estimation) -- meaning no in-band circular
    # orbit could explain them. Returning a TLE in that situation is false
    # precision: it looks exactly as confident as a genuine convergence.
    best_err_arcsec = best_err * 206265.0
    at_boundary = (best_h <= 150.5) or (best_h >= 1999.5)
    rel_err = best_err_arcsec / max(theta_dot * 206265.0, 1e-6)
    if at_boundary and rel_err > 0.05:
        return None

    # ── Reconstruct 3D state at best altitude ─────────────────────────────
    r_sat   = R_E + best_h
    v_orb   = math.sqrt(MU / r_sat)
    cos_zen = float(np.dot(obs_eci / obs_r, L))
    disc    = obs_r**2 * cos_zen**2 - obs_r**2 + r_sat**2
    rho     = -obs_r * cos_zen + math.sqrt(disc)

    # Satellite position in ECI
    r_vec = obs_eci + rho * L

    # Velocity direction from streak position angle:
    # PA is East-of-North on the sky → convert to ECI velocity direction
    # North unit vector on sky at (ra, dec):
    ra_r  = math.radians(ra_deg)
    dec_r = math.radians(dec_deg)
    pa_r  = math.radians(pa_deg)

    # Local sky North and East unit vectors in ECI
    N_sky = np.array([-math.sin(dec_r)*math.cos(ra_r),
                      -math.sin(dec_r)*math.sin(ra_r),
                       math.cos(dec_r)])
    E_sky = np.array([-math.sin(ra_r),
                       math.cos(ra_r),
                       0.0])

    # Streak direction in ECI (position angle is angle from North toward East)
    v_dir = math.cos(pa_r) * N_sky + math.sin(pa_r) * E_sky
    v_dir = v_dir / np.linalg.norm(v_dir)

    # Project v_dir onto the plane truly perpendicular to r_vec (position
    # from Earth's CENTER), not to the line of sight L. These are NOT the
    # same direction -- L is the sky-tangent-plane normal at the observer,
    # but the actual circular-orbit constraint is v_sat ⊥ r_vec, and
    # r_vec = obs_eci + rho*L is offset from the geocenter by the
    # observer's own position, so v_dir (⊥ L) is generally NOT already ⊥
    # r_vec. Confirmed with a synthetic-orbit test: uncorrected, this was
    # off from perpendicular by several degrees for typical geometries,
    # which _state_to_keplerian correctly rejected as unphysical (returned
    # None for every track). The orthogonal projection below is the
    # closest ⊥-r_vec direction to the PA-implied v_dir, preserving the
    # observed position angle as closely as geometrically possible while
    # satisfying the real constraint.
    r_hat = r_vec / np.linalg.norm(r_vec)
    v_dir = v_dir - np.dot(v_dir, r_hat) * r_hat
    v_dir = v_dir / np.linalg.norm(v_dir)

    # Full velocity vector (circular orbit speed, along corrected direction)
    v_vec = v_orb * v_dir

    # ── State → Keplerian elements ────────────────────────────────────────
    elems = _state_to_keplerian(r_vec, v_vec)
    if elems is None:
        return None

    incl, raan, ecc, argp, ma, mm = elems

    # Force circular orbit (ecc ≈ 0)
    ecc  = 0.0001
    argp = 0.0
    # ma stays as is (becomes argument of latitude for circular orbit)

    # ── Build epoch string ────────────────────────────────────────────────
    t_a   = Time(time_jd, format='jd', scale='utc')
    doy   = (t_a.jd - Time(f"{t_a.datetime.year}-01-01",
                             format='iso', scale='utc').jd) + 1.0
    epoch = f"{t_a.datetime.year % 100:02d}{doy:012.8f}"

    l1, l2 = generate_tle_lines(incl, raan, ecc, argp, ma, mm, epoch)

    rmse_note = (f"Circular-orbit assumption; altitude≈{best_h:.0f} km; "
                 f"single-frame solution (no held-out validation possible)")

    print(f"    LEO IOD: h≈{best_h:.0f} km  incl≈{incl:.1f}°  "
          f"mm≈{mm:.3f} rev/day")
    return l1, l2, best_h, rmse_note


def two_point_leo_iod(obs_list, lat_deg=SITE_LATITUDE, lon_deg=SITE_LONGITUDE,
                       alt_m=SITE_ALTITUDE):
    """
    Too-Short-Arc (TSA) IOD for exactly 2 observations of a LEO/MEO-band
    object, reusing the circular-orbit altitude-scan solver from
    `leo_streak_to_tle`.

    Rationale (SOTA angles-only TSA literature — Fujimoto & Scheeres 2011
    admissible-region correlation; Gooding-algorithm short-arc IOD): two
    time-tagged angle observations give an unambiguous attributable (RA,
    Dec, RA-dot, Dec-dot), directly analogous to a single-frame trail's
    (position, angular velocity, position angle). Unlike a single-exposure
    streak, the direction is NOT ambiguous here (obs[0] -> obs[1] gives a
    definite direction of motion) — this is actually cleaner input than the
    single-frame case.

    Closes the gap where 2-observation tracks were previously dropped
    straight to UCT because Gauss IOD requires >= 3 points.

    Returns (l1, l2, altitude_km, note) or None.
    """
    obs_sorted = sorted(obs_list, key=lambda d: d.get("time_jd") or 0)
    if len(obs_sorted) != 2:
        return None
    p1, p2 = obs_sorted[0], obs_sorted[-1]
    dt_s = (p2["time_jd"] - p1["time_jd"]) * 86400.0
    if dt_s <= 0:
        return None

    c1 = SkyCoord(p1["ra"] * u.deg, p1["dec"] * u.deg)
    c2 = SkyCoord(p2["ra"] * u.deg, p2["dec"] * u.deg)
    ang_vel_arcsec_s = c1.separation(c2).arcsec / dt_s
    pa_deg = c1.position_angle(c2).deg   # unambiguous: p1 -> p2 direction

    ra_mid  = (p1["ra"] + p2["ra"]) / 2.0
    dec_mid = (p1["dec"] + p2["dec"]) / 2.0
    jd_mid  = (p1["time_jd"] + p2["time_jd"]) / 2.0

    result = leo_streak_to_tle(ra_mid, dec_mid, ang_vel_arcsec_s, pa_deg,
                                jd_mid, lat_deg=lat_deg, lon_deg=lon_deg,
                                alt_m=alt_m)
    if result is None:
        return None
    l1, l2, h_km, note = result
    note = f"2-observation TSA (unambiguous direction); {note}"
    return l1, l2, h_km, note


def leo_cross_field_iod(leo_csv_path, ts, observer, epoch_str):
    """
    Links single-frame LEO detections across different ZTF fields
    by propagating each single-frame TLE and checking consistency
    with other streaks detected the same night.

    A LEO satellite moves ~0.5–2° per second. Two detections in
    different fields are linked if:
      - time separation < 600 s
      - the single-frame TLE propagated to the second epoch
        predicts a position within LINK_THRESH_DEG of the
        second detection

    Returns a list of linked pairs, each with a refined TLE.
    """
    import csv, math
    LINK_THRESH_DEG = 2.0    # degrees — generous for single-frame TLE accuracy
    MAX_DT_S        = 600.0  # max time between linked detections

    if not os.path.exists(leo_csv_path):
        return []

    # Load all single-frame LEO detections
    rows = []
    with open(leo_csv_path, newline="") as f:
        for r in csv.DictReader(f):
            try:
                rows.append({
                    "ra":         float(r["ra"]),
                    "dec":        float(r["dec"]),
                    "jd":         float(r["Time_JD"]),
                    "ang_vel":    float(r["angular_vel"]),
                    "pa_deg":     float(r["pa_deg"]),
                    "filename":   r["filename"],
                })
            except Exception:
                continue

    if len(rows) < 2:
        return []

    rows.sort(key=lambda x: x["jd"])
    print(f"\n  LEO cross-field linking: {len(rows)} single-frame detections")

    linked_pairs = []

    for i, r1 in enumerate(rows):
        # Build single-frame TLE for r1
        result1 = leo_streak_to_tle(
            r1["ra"], r1["dec"], r1["ang_vel"],
            r1["pa_deg"], r1["jd"]
        )
        if result1 is None:
            continue
        l1_tle, l2_tle, h1_km, _ = result1

        try:
            sat1 = EarthSatellite(l1_tle, l2_tle, "LEO_CAND", ts)
        except Exception:
            continue

        for j, r2 in enumerate(rows):
            if j <= i:
                continue
            dt_s = (r2["jd"] - r1["jd"]) * 86400.0
            if dt_s > MAX_DT_S:
                break   # sorted by JD, no point continuing

            # Propagate r1's TLE to r2's epoch and check angular distance
            t2 = ts.tt_jd(r2["jd"])
            try:
                pr, pd, _ = (sat1 - observer).at(t2).radec()
                pred_ra  = pr.hours * 15.
                pred_dec = pd.degrees
            except Exception:
                continue

            cos_d = math.cos(math.radians(r2["dec"]))
            dra   = ((pred_ra - r2["ra"] + 180) % 360 - 180) * cos_d
            ddec  = pred_dec - r2["dec"]
            sep   = math.sqrt(dra**2 + ddec**2)

            if sep > LINK_THRESH_DEG:
                continue

            print(f"    LEO link: {r1['filename']} ↔ {r2['filename']}  "
                  f"dt={dt_s:.1f}s  sep={sep:.3f}°  h≈{h1_km:.0f}km")

            # Attempt 2-point IOD using both positions
            result2 = leo_streak_to_tle(
                r2["ra"], r2["dec"], r2["ang_vel"],
                r2["pa_deg"], r2["jd"]
            )
            if result2 is None:
                continue
            l1_tle2, l2_tle2, h2_km, _ = result2

            linked_pairs.append({
                "det1":       r1,
                "det2":       r2,
                "l1_A":       l1_tle,
                "l2_A":       l2_tle,
                "l1_B":       l1_tle2,
                "l2_B":       l2_tle2,
                "h_km_A":     h1_km,
                "h_km_B":     h2_km,
                "dt_s":       dt_s,
                "sep_deg":    sep,
            })

    print(f"  LEO cross-field: {len(linked_pairs)} linked pairs")
    return linked_pairs





def stage3_process_images(fits_paths: list):
    print("\n" + "="*60)
    print(f"STAGE 3: PROCESSING {len(fits_paths)} ZTF FRAME(S)")
    print("="*60)

    all_detections = []
    skipped        = []
    last_good_wcs  = None

    for frame_idx, fname in enumerate(fits_paths):
        filename = os.path.basename(fname)
        print(f"\n[{frame_idx+1}/{len(fits_paths)}]  {filename}")

        try:
            with fits.open(fname, memmap=False) as hdul:
                hdu = next(
                    (h for h in hdul
                     if h.data is not None and h.data.ndim >= 2),
                    None
                )
                if hdu is None:
                    raise ValueError("No 2-D image data")
                header = hdu.header.copy()
                image  = hdu.data.squeeze().astype(np.float32)
        except Exception as e:
            print(f"    ✕  {e}")
            skipped.append(fname)
            continue

        t_mid    = get_timestamp(header)
        time_utc = t_mid.isot if t_mid else "UNKNOWN"
        time_jd  = float(t_mid.jd) if t_mid else None

        try:    exptime_s = float(header.get("EXPTIME", 30.0))
        except: exptime_s = 30.0

        wcs = get_wcs(header)
        if wcs is not None:
            last_good_wcs = wcs
        elif last_good_wcs is not None:
            wcs = last_good_wcs
        else:
            print("    ✕  No WCS — SKIPPED")
            continue

        try:
            image_clean = smooth_background_subtract(image)
        except Exception:
            image_clean = image

        sources, pass_used = detect_streaks_adaptive(image_clean, exptime_s)

        if not sources:
            print("    ⚠  No sources detected (both passes) — discarding frame")
            try:
                os.remove(fname)
            except Exception:
                pass
            continue

        if pass_used == 2:
            print(f"    ℹ  Pass-2 (relaxed thresholds) used")

        for src in sources:
            ra, dec = pix_to_radec(wcs, src["x_pix"], src["y_pix"])
            all_detections.append({
                "frame_index": frame_idx,
                "filename":    filename,
                "time_utc":    time_utc,
                "time_jd":     time_jd,
                "time_obj":    t_mid,
                "ra":          ra,
                "dec":         dec,
                "designation": "UNIDENTIFIED",
                **src,
            })
        print(f"    ✅ {len(sources)} sources (pass {pass_used})")

        try:
            os.remove(fname)
        except Exception:
            pass

    print(f"\n✅ Total detections: {len(all_detections)}")
    return all_detections


def process_single_frame(fits_path, frame_idx, total, last_good_wcs, field_id=0):
    """Process one ZTF FITS file; return (detections, updated_wcs)."""
    filename = os.path.basename(fits_path)
    print(f"\n[{frame_idx+1}/{total}]  {filename}")
    detections = []

    try:
        with fits.open(fits_path, memmap=False) as hdul:
            hdu = next(
                (h for h in hdul if h.data is not None and h.data.ndim >= 2),
                None
            )
            if hdu is None:
                raise ValueError("No 2-D image data")
            header = hdu.header.copy()
            image  = hdu.data.squeeze().astype(np.float32)
    except Exception as e:
        print(f"    ✕  {e}")
        return detections, last_good_wcs

    t_mid    = get_timestamp(header)
    time_utc = t_mid.isot if t_mid else "UNKNOWN"
    time_jd  = float(t_mid.jd) if t_mid else None

    try:    exptime_s = float(header.get("EXPTIME", 30.0))
    except: exptime_s = 30.0

    wcs = get_wcs(header)
    if wcs is not None:
        last_good_wcs = wcs
    elif last_good_wcs is not None:
        wcs = last_good_wcs
    else:
        print("    ✕  No WCS — SKIPPED")
        return detections, last_good_wcs

    try:
        image_clean = smooth_background_subtract(image)
    except Exception:
        image_clean = image

    sources, pass_used = detect_streaks_adaptive(image_clean, exptime_s)
    if not sources:
        print("    ⚠  No sources detected — discarding frame")
        return detections, last_good_wcs

    if pass_used == 2:
        print(f"    ℹ  Pass-2 (relaxed thresholds) used")

    for src in sources:
        ra, dec = pix_to_radec(wcs, src["x_pix"], src["y_pix"])
        mag, mag_err = compute_magnitude(src["flux"], header, exptime_s)
        detections.append({
            "frame_index": frame_idx,
            "filename":    filename,
            "field_id":    field_id,
            "time_utc":    time_utc,
            "time_jd":     time_jd,
            "time_obj":    t_mid,
            "ra":          ra,
            "dec":         dec,
            "designation": "UNIDENTIFIED",
            "magnitude":   mag,
            "mag_err":     mag_err,
            **src,
        })
    
    # ── LEO single-frame streak characterization (reuse existing detections) ──
    # No second detect_sources call needed — extract streak info from the
    # sources already found above using their eccentricity and elongation.
    _n_ecc_pass = _n_angvel_pass = 0
    try:
        leo_streaks = []
        for src in sources:
            # Only very elongated sources qualify as LEO streak candidates
            if src.get("eccentricity", 0) < 0.95:
                continue
            _n_ecc_pass += 1
            ra_s, dec_s = pix_to_radec(wcs, src["x_pix"], src["y_pix"])
            if ra_s is None:
                continue
            # Estimate streak length from elongation:
            # elongation = semimajor/semiminor → semimajor_px ≈ elongation × semiminor
            # Use eccentricity to recover semi-axes estimate
            ecc = src["eccentricity"]
            if ecc >= 0.999:
                continue
            # b from FWHM_P1, a = b / sqrt(1-ecc²)
            b_px = FWHM_P1 * gaussian_fwhm_to_sigma
            a_px = b_px / math.sqrt(max(1e-6, 1.0 - ecc**2))
            streak_len_px = 4.0 * a_px
            ang_vel = (streak_len_px * _ZTF_SCALE_ARCSEC_PX) / exptime_s
            # Floor verified against real run data: build_tracks/
            # cross_field_link_tracks topped out at 133.4 arcsec/s for
            # linked point-source tracks in this pipeline's own last run
            # -- objects faster than that trail enough within one 30s
            # exposure to already fail as compact point sources there.
            # The old 200 arcsec/s floor left a real gap (130-200
            # arcsec/s) covered by NEITHER channel. 100 gives a safety
            # margin below the observed ceiling.
            if ang_vel < 100.0:
                continue
            _n_angvel_pass += 1
                
            pa_deg_real = _streak_position_angle_on_sky(
                wcs, src["x_pix"], src["y_pix"],
                src.get("orientation_rad", 0.0), a_px
            )
            leo_streaks.append({
                "ra":          ra_s,
                "dec":         dec_s,
                "angular_vel": ang_vel,
                "pa_deg":      pa_deg_real,
                "streak_len":  streak_len_px * _ZTF_SCALE_ARCSEC_PX,

                
                "flux":        src["flux"],
                "frame_index": frame_idx,
                "filename":    filename,
                "time_utc":    time_utc,
                "time_jd":     time_jd,
                "time_obj":    t_mid,
                "orbit_class": "LEO_CANDIDATE",
            })
        if leo_streaks:
            detections.extend(leo_streaks)
            print(f"    ℹ  {len(leo_streaks)} LEO-class streak(s) extracted "
                  f"({_n_ecc_pass} ecc-qualified, {_n_angvel_pass} also "
                  f"passed angular-velocity floor)")
        elif _n_ecc_pass > 0:
            print(f"    ℹ  {_n_ecc_pass} elongated source(s) found but none "
                  f"passed the angular-velocity floor (100\"/s) -- likely "
                  f"real point-source motion, not a LEO streak")

    except Exception as e:
        print(f"    ⚠  LEO streak extraction FAILED: {type(e).__name__}: {e}")

    print(f"    ✅ {len(sources)} sources (pass {pass_used})")
    return detections, last_good_wcs
   



# ══════════════════════════════════════════════════════════════════════════════
#  STAGES 1-3 STREAMING PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

import threading
from queue import Queue, Empty

# ── Tunable knobs ────────────────────────────────────────────────────────────
_IRSA_DOWNLOAD_DELAY_S = 0.05   # was 0.5 — cut to minimum courtesy delay
_PREFETCH_QUEUE_SIZE   = 3       # max files buffered on disk at once (~108 MB peak)
_N_DOWNLOAD_WORKERS    = 5       # concurrent HTTP connections to IRSA


def stage1_to_3_streaming() -> list:
    """
    Pipelined streaming: a download thread-pool fills a bounded queue while
    the main thread processes frames off the queue.
    Peak disk usage = _PREFETCH_QUEUE_SIZE × ~36 MB ≈ 108 MB.
    """
    print("\n" + "="*60)
    print("STAGES 1–3: STREAMING ZTF DOWNLOAD + PROCESS (PIPELINED)")
    print("="*60)

    import time as _time_mod
    _PIPELINE_START = _time_mod.time()
    _WALLCLOCK_BUDGET_S = 10.5 * 3600.0   # stop new downloads at 10.5h,
                                            # leaving ~1.5h for Stage 4A/4B
                                            # and report generation

    processed_log = os.path.join(WORKING_DIR, "processed.txt")
    already_done  = set()
    if os.path.exists(processed_log):
        with open(processed_log) as f:
            already_done = set(l.strip() for l in f)
        if already_done:
            print(f"  ℹ  Resuming — {len(already_done)} images already processed")

    try:
        session = _irsa_session()
    except RuntimeError as e:
        print(f"❌  {e}")
        return []

    try:
        records = _irsa_query_ztf_metadata_chunked(
            session, START_DATE, END_DATE, chunk_days=1,
            pipeline_start=_PIPELINE_START,
            wallclock_budget_s=_WALLCLOCK_BUDGET_S)
    except RuntimeError as e:
        print(f"❌  {e}")
        return []

    if not records:
        print("⚠  No ZTF records found for this date range.")
        return []

    jds = sorted(float(r.get("obsjd", 0)) for r in records if r.get("obsjd"))
    if jds:
        n_nights = len({int(jd - 0.5) for jd in jds})
        print(f"  ℹ  Record coverage: {len(records)} records spanning "
              f"JD {jds[0]:.2f} -> {jds[-1]:.2f}  "
              f"(~{n_nights} distinct night(s))")

    _seen_files = set()
    _dup_count = 0
    _deduped_records = []
    for _r in records:
        try:
            _url, _fname = _irsa_build_download_url(_r)
        except Exception:
            _deduped_records.append(_r)   # keep; URL-build failure handled later
            continue
        if _fname in _seen_files:
            _dup_count += 1
            continue   # drop — this exposure is already queued
        _seen_files.add(_fname)
        _deduped_records.append(_r)
    if _dup_count > 0:
        print(f"  ⚠  {_dup_count} duplicate filenames found in metadata "
              f"(overlapping chunk date boundaries) -- removed before "
              f"download. Each distinct exposure is downloaded exactly "
              f"once; multiple DIFFERENT exposures of the same field are "
              f"kept (that's what multi-frame linking needs).")
    else:
        print(f"  ✅ No duplicate filenames in metadata -- chunking "
              f"boundaries are clean, no redundant downloads expected.")
    records = _deduped_records




    # ── Field revisit filter — Option B (LEO-aware) ─────────────────────
    # Previously this dropped every single-night field BEFORE download,
    # reasoning that a single-night tracklet can't produce a cross-night
    # fused MEO/GEO orbit. True for MEO/GEO, but backwards for LEO: a LEO
    # object crossing a field is normally a single-night (often
    # single-exposure) event and will likely never be seen again on a
    # second night from this site. That filter was silently discarding
    # most real LEO opportunities before they were ever downloaded.
    #
    # Now: keep ALL exposures of multi-night fields (unchanged — feeds
    # Stage 4A/4B MEO/GEO fallback), AND keep a capped, time-spread
    # sample of single-night fields (feeds Stage 3L LEO single-frame +
    # same-night multi-frame linking), instead of discarding them
    # outright. The cap protects the wall-clock budget the original
    # filter was designed to protect.
    _MAX_EXPOSURES_PER_SINGLE_NIGHT_FIELD = 8

    from collections import defaultdict as _ddf
    _field_nights = _ddf(set)
    for _r in records:
        try:
            _fid = int(float(_r.get("field", 0)))
            _ojd = float(_r.get("obsjd", 0))
        except (ValueError, TypeError):
            continue
        _night_bin = int(_ojd - 0.5)   # UTC-day bin, aligns with night boundary
        _field_nights[_fid].add(_night_bin)

    _multi_night_fields = {fid for fid, nights in _field_nights.items()
                            if len(nights) >= 2}

    _before_n = len(records)
    multi_night_records  = [r for r in records
                             if int(float(r.get("field", 0))) in _multi_night_fields]

    _single_by_field = _ddf(list)
    for r in records:
        try:
            _fid = int(float(r.get("field", 0)))
        except (ValueError, TypeError):
            continue
        if _fid not in _multi_night_fields:
            _single_by_field[_fid].append(r)

    single_night_records = []
    for _fid, _recs in _single_by_field.items():
        _recs = sorted(_recs, key=lambda r: float(r.get("obsjd", 0)))
        n = len(_recs)
        cap = _MAX_EXPOSURES_PER_SINGLE_NIGHT_FIELD
        if n <= cap:
            single_night_records.extend(_recs)
        else:
            # Evenly-spaced sample across the night rather than the first
            # `cap` exposures, so linking windows aren't biased early.
            idxs = [round(i * (n - 1) / (cap - 1)) for i in range(cap)]
            single_night_records.extend(_recs[i] for i in sorted(set(idxs)))

    records = multi_night_records + single_night_records
    _pct = (len(records) / _before_n * 100.0) if _before_n else 0.0
    print(f"  ✅ Field revisit filter (Option B): {len(_multi_night_fields)} "
          f"multi-night field(s) kept in full ({len(multi_night_records)} "
          f"records) + {len(_single_by_field)} single-night field(s) capped "
          f"at {_MAX_EXPOSURES_PER_SINGLE_NIGHT_FIELD} exposures each "
          f"({len(single_night_records)} records) -> "
          f"{len(records)}/{_before_n} total ({_pct:.1f}%).")

    records = sorted(records, key=lambda r: float(r.get("obsjd", 0)))

    total = min(len(records), _IRSA_MAX_FILES)
    pending = [rec for rec in records[:total]
               if _irsa_build_download_url(rec)[1] not in already_done]
    print(f"  {total} queued ({len(pending)} not yet done). Streaming …\n")

    fields = ["Type", "Frame", "Time_UTC", "Time_JD", "RA_deg", "Dec_deg",
              "Elongation", "Eccentricity", "Flux", "Magnitude", "Mag_Err",
              "X_pix", "Y_pix", "Filename"]
    if not already_done:
        with open(OUTPUT_CSV, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=fields).writeheader()

    # ── Shared state ─────────────────────────────────────────────────────────
    # Queue carries (fits_path, filename, frame_idx) or None as sentinel
    q          = Queue(maxsize=_PREFETCH_QUEUE_SIZE)
    csv_lock   = threading.Lock()
    log_lock   = threading.Lock()
    err_counts = {"download": 0}

    # ── Downloader thread function ────────────────────────────────────────────
    def _download_worker(records_chunk, session_local):
        """Downloads a slice of records and pushes paths onto the queue."""
        for idx, rec in records_chunk:
            try:
                url, filename = _irsa_build_download_url(rec)
            except Exception as e:
                print(f"  ⚠  [{idx+1}/{total}] URL build failed: {e}")
                continue

            fits_path = os.path.join(TMP_DIR, filename)

            try:
                _irsa_download_fits(session_local, url, filename, TMP_DIR)
            except Exception as e:
                print(f"  ⚠  [{idx+1}/{total}] Download skipped {filename}: {e}")
                err_counts["download"] += 1
                continue

            # ASTAP fallback — only if no embedded WCS
            # (combine with the processing open to avoid a 3rd FITS read)
            has_wcs = False
            try:
                from astropy.io import fits as _fits
                from astropy.wcs import WCS as _WCS
                with _fits.open(fits_path, memmap=False) as hdul:
                    _wcs    = _WCS(hdul[0].header, naxis=2)
                    has_wcs = _wcs.has_celestial
            except Exception:
                pass

            if not has_wcs:
                print(f"  No embedded WCS for {filename} — running ASTAP …")
                if not astap_solve(fits_path):
                    try:
                        os.remove(fits_path)
                    except Exception:
                        pass
                    continue

            try:
                field_id_for_queue = int(float(rec.get("field", 0)))
            except (ValueError, TypeError):
                field_id_for_queue = 0

            print(f"    [debug] queued {filename} → field_id={field_id_for_queue}")

            q.put((fits_path, filename, idx, field_id_for_queue))
            time.sleep(_IRSA_DOWNLOAD_DELAY_S)

        # Sentinel — one per worker; processor counts them
        q.put(None)

    # ── Split work across download workers ───────────────────────────────────
    # Round-robin partition so each worker gets a contiguous slice
    # (preserves temporal ordering within each worker's batch)
    indexed_pending = list(enumerate(pending))
    chunks = [indexed_pending[i::_N_DOWNLOAD_WORKERS]
              for i in range(_N_DOWNLOAD_WORKERS)]

    # Each worker needs its own session (requests.Session is not thread-safe)
    worker_sessions = [_irsa_session() for _ in range(_N_DOWNLOAD_WORKERS)]

    threads = []
    for chunk, sess in zip(chunks, worker_sessions):
        t = threading.Thread(target=_download_worker, args=(chunk, sess),
                             daemon=True)
        t.start()
        threads.append(t)

    # ── Processor (main thread) ───────────────────────────────────────────────
    all_detections = []
    last_good_wcs  = None
    sentinels_seen = 0
    processed_count = 0

    while sentinels_seen < _N_DOWNLOAD_WORKERS:
        try:
            item = q.get(timeout=120)   # 2-min timeout guards against stalled workers
        except Empty:
            print("  ⚠  Queue empty for 120 s — possible download stall")
            break

        if item is None:
            sentinels_seen += 1
            continue

        fits_path, filename, idx, field_id = item

        new_detections, last_good_wcs = process_single_frame(
            fits_path, idx, total, last_good_wcs, field_id=field_id
        )
        all_detections.extend(new_detections)

        # Delete immediately after processing
        try:
            os.remove(fits_path)
        except Exception:
            pass

        # Append to CSV (thread-safe lock in case you ever parallelize processing)
        if new_detections:
            with csv_lock:
                with open(OUTPUT_CSV, "a", newline="") as f:
                    writer = csv.DictWriter(f, fieldnames=fields)
                    for d in new_detections:
                        writer.writerow({
                        "Type":         "SATELLITE",
                        "Frame":        d["frame_index"] + 1,
                        "Time_UTC":     d["time_utc"],
                        "Time_JD":      f"{d['time_jd']:.8f}" if d["time_jd"] else "",
                        "RA_deg":       f"{d['ra']:.6f}"  if d["ra"]  is not None else "",
                        "Dec_deg":      f"{d['dec']:.6f}" if d["dec"] is not None else "",
                        "Elongation":   f"{d['elongation']:.3f}",
                        "Eccentricity": f"{d['eccentricity']:.4f}",
                        "Flux":         f"{d['flux']:.2f}",
                        "Magnitude":    f"{d['magnitude']:.3f}" if d.get("magnitude") is not None else "",
                        "Mag_Err":      f"{d['mag_err']:.3f}"   if d.get("mag_err")   is not None else "",
                        "X_pix":        f"{d['x_pix']:.2f}",
                        "Y_pix":        f"{d['y_pix']:.2f}",
                        "Filename":     d["filename"],
                    })

        # Write LEO single-frame streaks to separate CSV
        leo_hits = [d for d in new_detections
                    if d.get("orbit_class") == "LEO_CANDIDATE"]
        if leo_hits:
            leo_csv = os.path.join(WORKING_DIR, "leo_single_frame.csv")
            leo_fields = ["frame_index", "filename", "Time_JD", "ra", "dec",
                          "angular_vel", "pa_deg", "streak_len", "flux"]
            with csv_lock:
                write_header = not os.path.exists(leo_csv)
                with open(leo_csv, "a", newline="") as f:
                    w = csv.DictWriter(f, fieldnames=leo_fields, extrasaction="ignore")
                    if write_header:
                        w.writeheader()
                    for d in leo_hits:
                        d["Time_JD"] = f"{d['time_jd']:.8f}" if d["time_jd"] else ""
                        w.writerow(d)

        with log_lock:
            with open(processed_log, "a") as f:
                f.write(filename + "\n")

        processed_count += 1
        if processed_count % 50 == 0:
            print(f"  Progress: {processed_count}/{len(pending)}  "
                  f"({len(all_detections)} detections so far)")

    for t in threads:
        t.join(timeout=30)

    print(f"\n✅ Streaming complete — {len(all_detections)} detections, "
          f"CSV at {OUTPUT_CSV}")
    return all_detections

# ══════════════════════════════════════════════════════════════════════════════
#  SHARED HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def radec_to_vec(ra_deg, dec_deg):
    ra, dec = math.radians(ra_deg), math.radians(dec_deg)
    return np.array([
        math.cos(dec) * math.cos(ra),
        math.cos(dec) * math.sin(ra),
        math.sin(dec)
    ])


def ang_sep_arcsec(ra1, dec1, ra2, dec2):
    c1 = SkyCoord(ra1 * u.deg, dec1 * u.deg)
    c2 = SkyCoord(ra2 * u.deg, dec2 * u.deg)
    return c1.separation(c2).arcsec


def compute_checksum(line: str) -> int:
    return sum(
        int(c) if c.isdigit() else (1 if c == "-" else 0)
        for c in line[:68]
    ) % 10


def generate_tle_lines(incl_deg, raan_deg, ecc, argp_deg, ma_deg,
                        mm_rev_day, epoch_str):
    ecc_str = f"{int(ecc * 1e7):07d}"
    l1 = f"1 99999U 23999A   {epoch_str}  .00000000  00000-0  00000-0 0  999"
    l1 = f"{l1:<68}{compute_checksum(l1)}"
    l2 = (f"2 99999 {incl_deg:8.4f} {raan_deg:8.4f} {ecc_str} "
          f"{argp_deg:8.4f} {ma_deg:8.4f} {mm_rev_day:11.8f}00001")
    l2 = f"{l2:<68}{compute_checksum(l2)}"
    return l1, l2


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 5A – RANSAC OUTLIER REJECTION
# ══════════════════════════════════════════════════════════════════════════════

def ransac_reject_outliers(times_jd, ra_deg, dec_deg,
                            inlier_thresh_arcsec=60.0,
                            n_iterations=200,
                            min_inliers=3):
    """
    ROOT-CAUSE FIX #1 (dominant driver of "0 HIGH confidence").

    The previous implementation fit a 2-point CONSTANT-RATE straight line in
    (RA, Dec) vs time and rejected everything more than `inlier_thresh` off
    that line. Real orbital tracks are NOT linear in RA/Dec over a multi-minute
    arc — LEO arcs curve strongly, and even MEO/GEO arcs curve measurably. A
    straight-line model therefore rejected the genuinely-curved (but real) parts
    of single-object tracks. In the last run this decimated 91% of tracks
    (383/419) down to <=6 observations — and the CURVATURE it threw away is
    exactly the information angles-only IOD needs to break the range degeneracy.
    Result: underdetermined fits -> "Range Unconstrained — Short Arc" -> no HIGH.

    This version:
      * Fits a QUADRATIC (constant-acceleration) model in RA*cos(dec) and Dec
        vs time from a 3-point minimal sample, which captures the real on-sky
        curvature and keeps curved-but-consistent points as inliers.
      * Unwraps RA so a track crossing the 0/360 deg seam does not blow up.
      * Works in a local tangent plane about the arc mean (no cos(dec)
        distortion at high declination).
      * Rejects only genuine outliers: falls back to keeping ALL points unless
        it can confidently identify a larger consensus set. It never returns
        fewer than the sample size, and prefers the linear (2-point) consensus
        if the quadratic finds nothing better — so it is strictly less
        destructive than the old version.
    """
    times = np.asarray(times_jd, dtype=float)
    ra    = np.asarray(ra_deg,   dtype=float)
    dec   = np.asarray(dec_deg,  dtype=float)
    n     = len(times)

    if n <= max(min_inliers, 4):
        # Too few points to safely reject anything — keep them all.
        return np.ones(n, dtype=bool), ra, dec, times

    # Local tangent-plane coordinates about the arc centroid (arcsec).
    t0      = np.mean(times)
    tau     = (times - t0) * 86400.0            # seconds, centered
    ra_ref  = np.mean(ra)
    dec_ref = np.mean(dec)
    cosd    = math.cos(math.radians(dec_ref))
    # Unwrap RA around the reference so a 0/360 crossing is continuous.
    dra  = ((ra - ra_ref + 180.0) % 360.0 - 180.0) * cosd * 3600.0   # arcsec
    ddec = (dec - dec_ref) * 3600.0                                   # arcsec

    thresh   = float(inlier_thresh_arcsec)
    rng      = np.random.default_rng(seed=42)
    best_cnt = 0
    best_mask = np.ones(n, dtype=bool)

    def _fit_quad(sel):
        # Design matrix for x(t)=a0+a1 t+a2 t^2 fitted independently to dra,ddec
        A = np.vstack([np.ones_like(tau[sel]), tau[sel], tau[sel] ** 2]).T
        try:
            cx, *_ = np.linalg.lstsq(A, dra[sel],  rcond=None)
            cy, *_ = np.linalg.lstsq(A, ddec[sel], rcond=None)
        except Exception:
            return None
        return cx, cy

    Afull = np.vstack([np.ones_like(tau), tau, tau ** 2]).T
    for _ in range(n_iterations):
        # Minimal sample of 3 well-separated points for a quadratic.
        idx = rng.choice(n, 3, replace=False)
        if np.ptp(tau[idx]) < 1.0:
            continue
        fit = _fit_quad(idx)
        if fit is None:
            continue
        cx, cy = fit
        pred_x = Afull @ cx
        pred_y = Afull @ cy
        sep = np.hypot(dra - pred_x, ddec - pred_y)
        mask = sep < thresh
        cnt = int(mask.sum())
        if cnt > best_cnt:
            best_cnt = cnt
            best_mask = mask.copy()

    # Refit the model on the full consensus set and re-evaluate once
    # (least-squares refinement of the RANSAC hypothesis).
    if best_cnt >= max(min_inliers, 4):
        fit = _fit_quad(best_mask)
        if fit is not None:
            cx, cy = fit
            sep = np.hypot(dra - Afull @ cx, ddec - Afull @ cy)
            refit_mask = sep < thresh
            if int(refit_mask.sum()) >= best_cnt:
                best_mask = refit_mask
                best_cnt  = int(refit_mask.sum())

    # Non-destructive guard: only accept the rejection if we keep a clear
    # majority AND at least enough points to overdetermine 6 elements.
    final_mask = (best_mask
                  if best_cnt >= max(min_inliers, 4) and best_cnt >= int(0.5 * n)
                  else np.ones(n, dtype=bool))

    n_rej = int(np.sum(~final_mask))
    if n_rej > 0:
        print(f"    RANSAC(quad): rejected {n_rej}/{n} outliers "
              f"(thresh={thresh:.0f}\", kept curvature)")

    return (final_mask,
            ra[final_mask],
            dec[final_mask],
            times[final_mask])


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 5B – GAUSS IOD + ADMISSIBLE REGIONS
# ══════════════════════════════════════════════════════════════════════════════

MU_EARTH_KM3_S2            = 398600.4418
EARTH_RADIUS_KM            = 6378.137
REV_PER_DAY_TO_RAD_PER_MIN = 2.0 * math.pi / 1440.0
_MIN_PERIGEE_FLOOR_KM      = 150.0   # shared floor for DE constraint + residual penalty


def _observatory_eci(obs_jd, lat_deg, lon_deg, alt_m):
    t   = Time(obs_jd, format='jd', scale='utc')
    obs = EarthLocation(lat=lat_deg * u.deg,
                        lon=lon_deg * u.deg,
                        height=alt_m * u.m)
    gcrs  = obs.get_gcrs(t)
    xyz_m = gcrs.cartesian.xyz.to(u.km).value
    return xyz_m


def _admissible_region_rho_bounds(L_unit, obs_eci_km, min_alt_km=100.0):
    obs_r    = np.linalg.norm(obs_eci_km)
    L        = np.asarray(L_unit, dtype=float)
    R_min_km = EARTH_RADIUS_KM + min_alt_km
    b_coeff  = 2.0 * float(np.dot(obs_eci_km, L))
    c_coeff  = obs_r**2 - R_min_km**2
    discrim  = b_coeff**2 - 4 * c_coeff
    if discrim < 0:
        rho_min = 0.0
    else:
        r1 = (-b_coeff - math.sqrt(discrim)) / 2.0
        r2 = (-b_coeff + math.sqrt(discrim)) / 2.0
        rho_min = max(0.0, r2) if r1 < 0 else max(0.0, min(r1, r2))
    rho_max = min(384400.0 - obs_r, 50000.0)
    return max(0.0, rho_min), max(rho_min + 1.0, rho_max)


def _state_to_keplerian(r_vec, v_vec):
    try:
        mu    = MU_EARTH_KM3_S2
        r     = np.asarray(r_vec, dtype=float)
        v     = np.asarray(v_vec, dtype=float)
        r_mag = float(np.linalg.norm(r))
        v_mag = float(np.linalg.norm(v))
        if r_mag < 100.0 or v_mag < 0.01:
            return None

        h_vec = np.cross(r, v)
        h_mag = float(np.linalg.norm(h_vec))
        if h_mag < 1.0:
            return None

        n_vec = np.cross(np.array([0., 0., 1.]), h_vec)
        n_mag = float(np.linalg.norm(n_vec))
        e_vec = (1.0 / mu) * ((v_mag**2 - mu / r_mag) * r
                               - float(np.dot(r, v)) * v)
        ecc   = float(np.linalg.norm(e_vec))

        energy = 0.5 * v_mag**2 - mu / r_mag
        if energy >= 0.0:
            return None
        a       = -mu / (2.0 * energy)
        n_rad_s = math.sqrt(mu / a**3)

        incl = math.degrees(math.acos(
            max(-1., min(1., float(h_vec[2]) / h_mag))
        ))
        if n_mag > 1e-10:
            raan = math.degrees(math.acos(
                max(-1., min(1., float(n_vec[0]) / n_mag))
            ))
            if n_vec[1] < 0.0:
                raan = 360.0 - raan
        else:
            raan = 0.0

        if n_mag > 1e-10 and ecc > 1e-6:
            argp = math.degrees(math.acos(
                max(-1., min(1., float(np.dot(n_vec, e_vec)) / (n_mag * ecc)))
            ))
            if e_vec[2] < 0.0:
                argp = 360.0 - argp
        else:
            argp = 0.0

        if ecc > 1e-6:
            nu = math.degrees(math.acos(
                max(-1., min(1., float(np.dot(e_vec, r)) / (ecc * r_mag)))
            ))
            if float(np.dot(r, v)) < 0.0:
                nu = 360.0 - nu
        else:
            nu = 0.0

        nu_rad = math.radians(nu)
        if ecc < 1.0:
            cos_E = (ecc + math.cos(nu_rad)) / (1.0 + ecc * math.cos(nu_rad))
            sin_E = (math.sqrt(1. - ecc**2) * math.sin(nu_rad)) / (
                1.0 + ecc * math.cos(nu_rad))
            E  = math.atan2(sin_E, cos_E)
            ma = math.degrees(E - ecc * math.sin(E)) % 360.0
        else:
            ma = nu

        mm_rev_day = (n_rad_s / (2.0 * math.pi)) * 86400.0
        return [incl, raan % 360.0, ecc, argp % 360.0, ma % 360.0, mm_rev_day]
    except Exception:
        return None


def gauss_iod(obs_jd_list, ra_deg_list, dec_deg_list,
              lat_deg, lon_deg, alt_m, n_iter_refine=3):
    """
    ROOT-CAUSE FIX #4 (Gauss failed on 81% of tracks: 339/419 "all rho2
    trials rejected").

    Classical Gauss IOD is notoriously ill-conditioned on short, noisy arcs,
    and the previous code committed to a SINGLE triplet (first / middle /
    last). One bad or poorly-spaced middle point killed the whole seed, so
    the pipeline fell through to weaker admissible-region / altitude-scan
    seeds (or none), landing LM/DE in the wrong basin.

    This wrapper now tries SEVERAL triplets spanning the arc — the extreme
    endpoints plus interior middles at different fractions — and returns the
    first physically-valid seed. It is strictly more robust than the old
    single-triplet call and falls back to None exactly as before if nothing
    is valid, so no downstream behaviour changes when Gauss genuinely can't
    solve.
    """
    obs_jd = list(obs_jd_list)
    n_obs  = len(obs_jd)
    if n_obs < 3:
        return None

    # Candidate middle indices at varying fractions of the arc; endpoints
    # anchored at the extremes for the widest possible time base.
    mids = sorted({max(1, min(n_obs - 2, int(round(f * (n_obs - 1)))))
                   for f in (0.5, 0.35, 0.65, 0.25, 0.75, 0.45, 0.55)})
    tried = set()
    for i2 in mids:
        triplet = (0, i2, n_obs - 1)
        if triplet in tried or i2 in (0, n_obs - 1):
            continue
        tried.add(triplet)
        res = _gauss_iod_core(obs_jd_list, ra_deg_list, dec_deg_list,
                              lat_deg, lon_deg, alt_m,
                              n_iter_refine=n_iter_refine, triplet=triplet)
        if res is not None:
            return res
    return None


def _gauss_iod_core(obs_jd_list, ra_deg_list, dec_deg_list,
                    lat_deg, lon_deg, alt_m, n_iter_refine=3, triplet=None):
    """
    Classical Gauss IOD (Curtis Alg. 5.5 / Vallado Alg. 52) with one
    fixed-point f/g refinement pass, for a single explicit observation
    triplet. Called repeatedly by the gauss_iod() wrapper above.
    """
    obs_jd  = list(obs_jd_list)
    ra_deg  = list(ra_deg_list)
    dec_deg = list(dec_deg_list)
    n_obs   = len(obs_jd)
    if n_obs < 3:
        return None

    if triplet is None:
        i1, i2, i3 = 0, n_obs // 2, n_obs - 1
    else:
        i1, i2, i3 = triplet
    t1, t2, t3 = obs_jd[i1], obs_jd[i2], obs_jd[i3]
    tau1 = (t1 - t2) * 86400.0
    tau3 = (t3 - t2) * 86400.0
    tau  = tau3 - tau1
    if abs(tau) < 1.0 or abs(tau1) < 0.5 or abs(tau3) < 0.5:
        return None

    mu = MU_EARTH_KM3_S2
    L1 = radec_to_vec(ra_deg[i1], dec_deg[i1])
    L2 = radec_to_vec(ra_deg[i2], dec_deg[i2])
    L3 = radec_to_vec(ra_deg[i3], dec_deg[i3])
    R1 = _observatory_eci(t1, lat_deg, lon_deg, alt_m)
    R2 = _observatory_eci(t2, lat_deg, lon_deg, alt_m)
    R3 = _observatory_eci(t3, lat_deg, lon_deg, alt_m)

    p1 = np.cross(L2, L3)
    p2 = np.cross(L1, L3)
    p3 = np.cross(L1, L2)
    D0 = float(np.dot(L1, p1))
    if abs(D0) < 1e-12:
        return None

    D11, D12, D13 = (float(np.dot(R1, p1)), float(np.dot(R1, p2)), float(np.dot(R1, p3)))
    D21, D22, D23 = (float(np.dot(R2, p1)), float(np.dot(R2, p2)), float(np.dot(R2, p3)))
    D31, D32, D33 = (float(np.dot(R3, p1)), float(np.dot(R3, p2)), float(np.dot(R3, p3)))

    A = (1.0 / D0) * (-D12 * (tau3 / tau) + D22 + D32 * (tau1 / tau))
    B = (1.0 / (6.0 * D0)) * (
        D12 * (tau3 ** 2 - tau ** 2) * (tau3 / tau)
        + D32 * (tau ** 2 - tau1 ** 2) * (tau1 / tau)
    )
    E    = float(np.dot(R2, L2))
    R2sq = float(np.dot(R2, R2))

    a = -(A ** 2 + 2.0 * A * E + R2sq)
    b = -2.0 * mu * B * (A + E)
    c = -(mu ** 2) * (B ** 2)

    # r2^8 + a*r2^6 + b*r2^3 + c = 0
    coeffs = [1.0, 0.0, a, 0.0, 0.0, b, 0.0, 0.0, c]
    roots  = np.roots(coeffs)
    real_roots = [r.real for r in roots
                  if abs(r.imag) < 1e-6 * max(1.0, abs(r.real))
                  and r.real > EARTH_RADIUS_KM + 100.0]
    if not real_roots:
        return None
    # Prefer the smallest physically plausible root that isn't absurd
    # (largest root is usually correct for well-conditioned arcs, but
    # for short/noisy arcs the smallest valid root is often more stable).
    real_roots.sort()
    r2_candidates = real_roots[-1:] if len(real_roots) == 1 else real_roots

    best = None
    for r2 in r2_candidates:
        try:
            r2c = r2
            for _ in range(max(1, n_iter_refine)):
                num1 = (6.0 * (D31 * (tau1 / tau3) + D21 * (tau / tau3)) * r2c ** 3
                        + mu * D31 * (tau ** 2 - tau1 ** 2) * (tau1 / tau3))
                den1 = 6.0 * r2c ** 3 + mu * (tau ** 2 - tau3 ** 2)
                rho1 = (1.0 / D0) * (num1 / den1 - D11)

                rho2 = A + mu * B / r2c ** 3

                num3 = (6.0 * (D13 * (tau3 / tau1) - D23 * (tau / tau1)) * r2c ** 3
                        + mu * D13 * (tau ** 2 - tau3 ** 2) * (tau3 / tau1))
                den3 = 6.0 * r2c ** 3 + mu * (tau ** 2 - tau1 ** 2)
                rho3 = (1.0 / D0) * (num3 / den3 - D33)

                if rho1 <= 0 or rho2 <= 0 or rho3 <= 0:
                    raise ValueError("negative range")

                r1vec = R1 + rho1 * L1
                r2vec = R2 + rho2 * L2
                r3vec = R3 + rho3 * L3

                f1 = 1.0 - 0.5 * mu * tau1 ** 2 / r2c ** 3
                g1 = tau1 - (1.0 / 6.0) * mu * tau1 ** 3 / r2c ** 3
                f3 = 1.0 - 0.5 * mu * tau3 ** 2 / r2c ** 3
                g3 = tau3 - (1.0 / 6.0) * mu * tau3 ** 3 / r2c ** 3
                denom = f1 * g3 - f3 * g1
                if abs(denom) < 1e-10:
                    raise ValueError("singular f/g")
                v2vec = (-f3 * r1vec + f1 * r3vec) / denom

                r2c = float(np.linalg.norm(r2vec))

            elems = _state_to_keplerian(r2vec, v2vec)
            if elems is None:
                continue
            incl, raan, ecc, argp, ma, mm = elems
            if not (0. <= incl <= 180. and 0. <= ecc < 1.0 and 0.5 <= mm <= 17.0):
                continue
            best = elems
            break  # first physically valid root wins
        except Exception:
            continue

    return best


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 5C – SGP4 DIFFERENTIAL CORRECTION (LM OPTIMISER)
# ══════════════════════════════════════════════════════════════════════════════

def _mean_elements_to_satrec(incl_r, raan_r, ecc, argp_r, ma_r,
                               mm_rad_min, epoch_jd, bstar=1e-5):
    # Scale bstar down for higher-altitude (lower mean-motion) candidates —
    # a LEO-typical bstar applied to a GEO/MEO orbit can push SGP4 into
    # spurious decay/error states during DE's exploratory phase.
    mm_rev_day_est = mm_rad_min / REV_PER_DAY_TO_RAD_PER_MIN
    if mm_rev_day_est < 2.0:        # roughly MEO/GEO band
        bstar = min(bstar, 1e-7)
    elif mm_rev_day_est < 6.4:      # MEO band
        bstar = min(bstar, 1e-6)
    # else: leave as-is for LEO-band candidates

    sat = Satrec()
    sat.sgp4init(
        WGS84, 'i', 99999,
        float(epoch_jd) - 2433281.5,
        float(bstar), 0.0, 0.0,
        max(0.0, min(0.9999, float(ecc))),
        float(argp_r) % (2 * math.pi),
        max(0.0, min(math.pi, float(incl_r))),
        float(ma_r) % (2 * math.pi),
        max(0.001, float(mm_rad_min)),
        float(raan_r) % (2 * math.pi),
    )
    return sat


_SGP4_ERROR_COUNTS = defaultdict(int)   # module-level tally for diagnostics

def _sgp4_topocentric_radec(sat, obs_jd, obs_lat_deg, obs_lon_deg, obs_alt_m):
    jd_int  = math.floor(obs_jd)
    jd_frac = obs_jd - jd_int
    try:
        e, r_km, _ = sat.sgp4(float(jd_int), float(jd_frac))
    except Exception:
        _SGP4_ERROR_COUNTS["python_exception"] += 1
        return None, None
    if e != 0:
        _SGP4_ERROR_COUNTS[f"sgp4_error_code_{e}"] += 1
        return None, None
    _SGP4_ERROR_COUNTS["ok"] += 1

    r_m = np.asarray(r_km, dtype=float) * 1000.0
    if np.any(np.isnan(r_m)) or np.linalg.norm(r_m) < 1e3:
        return None, None

    try:
        obs_eci = _observatory_eci(obs_jd, obs_lat_deg, obs_lon_deg, obs_alt_m)
        topo    = r_km - obs_eci
        tm      = float(np.linalg.norm(topo))
        if tm < 1.0:
            return None, None
        th      = topo / tm
        ra_deg  = math.degrees(math.atan2(th[1], th[0])) % 360.
        dec_deg = math.degrees(math.asin(max(-1., min(1., th[2]))))
        return ra_deg, dec_deg
    except Exception:
        return None, None


def _build_residual_fn(obs_jd, ra_obs, dec_obs, epoch_jd,
                        lat_deg, lon_deg, alt_m, bstar=1e-5):
    jd_arr   = np.asarray(obs_jd,  dtype=float)
    ra_arr   = np.asarray(ra_obs,  dtype=float)
    dec_arr  = np.asarray(dec_obs, dtype=float)
    cos_decs = np.cos(np.radians(dec_arr))

    _MIN_PERIGEE_ALT_KM = _MIN_PERIGEE_FLOOR_KM  # shared with DE constraint, keep in sync

    # Pre-transfer observation arrays to GPU once — reused across all LM iterations
    try:
        import cupy as cp
        _ra_gpu      = cp.asarray(ra_arr,   dtype=cp.float32)
        _dec_gpu     = cp.asarray(dec_arr,  dtype=cp.float32)
        _cos_dec_gpu = cp.asarray(cos_decs, dtype=cp.float32)
        _USE_GPU     = True
    except Exception:
        _USE_GPU = False


def _propagate_covariance(best_x, cov_params, epoch_jd, future_jd,
                           lat_deg, lon_deg, alt_m, jd_fit, ra_fit, dec_fit):
    """
    Propagate orbital element covariance to a future epoch using
    numerical state transition matrix (STM).

    Computes the 1-sigma position uncertainty ellipse at future_jd
    by perturbing each of the 6 orbital elements and measuring
    the resulting sky-plane displacement.

    Returns:
        sigma_ra_arcsec  : 1-sigma uncertainty in RA direction
        sigma_dec_arcsec : 1-sigma uncertainty in Dec direction
        sigma_total_arcsec: combined 1-sigma angular uncertainty
        cov_sky_2x2      : 2×2 sky-plane covariance matrix (arcsec²)
    """
    if best_x is None:
        return float('nan'), float('nan'), float('nan'), None
    if cov_params is None:
        # Empirical fallback: use a diagonal covariance based on
        # typical element uncertainties for short-arc OD.
        # These are conservative estimates, not formal uncertainties.
        _sig_elem = np.array([
            1e-3,   # incl_r    (rad) ≈ 0.06°
            1e-3,   # raan_r    (rad)
            1e-4,   # ecc
            1e-3,   # argp_r    (rad)
            1e-3,   # ma_r      (rad)
            1e-6,   # mm_rad_min
        ])
        cov_params = np.diag(_sig_elem**2)

    try:
        # Build nominal satellite at future epoch
        incl_r, raan_r, ecc_, argp_r, ma_r, mm_ = best_x
        sat_nom = _mean_elements_to_satrec(
            incl_r, raan_r, ecc_, argp_r, ma_r, mm_, epoch_jd)

        jd_int  = math.floor(future_jd)
        jd_frac = future_jd - jd_int
        e0, r0, _ = sat_nom.sgp4(jd_int, jd_frac)
        if e0 != 0 or r0 is None:
            return float('nan'), float('nan'), float('nan'), None

        obs_eci = _observatory_eci(future_jd, lat_deg, lon_deg, alt_m)
        topo0   = np.array(r0) - obs_eci
        t0_mag  = float(np.linalg.norm(topo0))
        if t0_mag < 1.0:
            return float('nan'), float('nan'), float('nan'), None
        th0     = topo0 / t0_mag
        ra0     = math.degrees(math.atan2(th0[1], th0[0])) % 360.
        dec0    = math.degrees(math.asin(max(-1., min(1., th0[2]))))

        # Perturbation steps — sized to numerical precision regime
        _STEPS = [1e-5, 1e-6, 1e-7, 1e-6, 1e-6, 1e-7]
        J = np.zeros((2, 6), dtype=float)   # sky-plane Jacobian

        for i in range(6):
            x_pert = best_x.copy()
            x_pert[i] += _STEPS[i]
            try:
                sat_p = _mean_elements_to_satrec(*x_pert, epoch_jd)
                ep, rp, _ = sat_p.sgp4(jd_int, jd_frac)
                if ep != 0:
                    continue
                topo_p = np.array(rp) - obs_eci
                tm_p   = float(np.linalg.norm(topo_p))
                if tm_p < 1.0:
                    continue
                th_p   = topo_p / tm_p
                ra_p   = math.degrees(math.atan2(th_p[1], th_p[0])) % 360.
                dec_p  = math.degrees(math.asin(max(-1., min(1., th_p[2]))))
                cos_dec = math.cos(math.radians(dec0))
                dra    = ((ra_p - ra0 + 180) % 360 - 180) * cos_dec * 3600.
                ddec   = (dec_p - dec0) * 3600.
                J[0, i] = dra  / _STEPS[i]
                J[1, i] = ddec / _STEPS[i]
            except Exception:
                continue

        # Propagate covariance: C_sky = J @ C_elem @ J^T
        try:
            C_sky   = J @ cov_params @ J.T
            sig_ra  = math.sqrt(max(0., float(C_sky[0, 0])))
            sig_dec = math.sqrt(max(0., float(C_sky[1, 1])))
            sig_tot = math.sqrt(max(0., float(C_sky[0,0] + C_sky[1,1]
                                             + 2*C_sky[0,1]))) / math.sqrt(2.)
            return sig_ra, sig_dec, sig_tot, C_sky
        except Exception:
            return float('nan'), float('nan'), float('nan'), None

    except Exception:
        return float('nan'), float('nan'), float('nan'), None


def _build_residual_fn(obs_jd, ra_obs, dec_obs, epoch_jd,
                        lat_deg, lon_deg, alt_m, bstar=1e-5):
    jd_arr   = np.asarray(obs_jd,  dtype=float)
    ra_arr   = np.asarray(ra_obs,  dtype=float)
    dec_arr  = np.asarray(dec_obs, dtype=float)
    cos_decs = np.cos(np.radians(dec_arr))
    n = len(jd_arr)

    # Observer position depends only on jd, which is FIXED for the whole
    # optimization. Compute once instead of once-per-residual-call.
    obs_eci_arr = np.array(
        [_observatory_eci(jd, lat_deg, lon_deg, alt_m) for jd in jd_arr]
    )  # (n, 3) — GCRS
    jd_int_arr  = np.floor(jd_arr)
    jd_frac_arr = jd_arr - jd_int_arr

    # ── ROOT-CAUSE FIX #5 (astrometric accuracy → lets good fits reach the
    #    <60" HIGH bar). SGP4 returns positions in the TEME frame, but the
    #    observer above is in GCRS. Subtracting them directly leaves a
    #    frame-mismatch bias of up to ~1 arcmin (polar motion + equation of
    #    the equinoxes), which alone can pin an otherwise-good orbit at
    #    MEDIUM/LOW. Precompute the per-epoch TEME→GCRS rotation ONCE (obs
    #    times are fixed) and apply it to every SGP4 evaluation. Falls back
    #    to identity (original behaviour) if the astropy TEME frame is
    #    unavailable, so this can never break the pipeline.
    _teme2gcrs = None
    try:
        from astropy.coordinates import TEME, GCRS, CartesianRepresentation
        mats = []
        for jd in jd_arr:
            _t = Time(jd, format='jd', scale='utc')
            _basis = CartesianRepresentation(np.eye(3) * u.km)  # 3 TEME unit vecs
            _g = TEME(_basis, obstime=_t).transform_to(GCRS(obstime=_t))
            mats.append(_g.cartesian.xyz.to(u.km).value)        # (3,3), cols=basis
        _teme2gcrs = np.asarray(mats)                            # (n,3,3)
    except Exception as _e:
        _teme2gcrs = None  # identity fallback

    _MIN_PERIGEE_ALT_KM = _MIN_PERIGEE_FLOOR_KM

    def _residuals(x):
        incl_r, raan_r, ecc_, argp_r, ma_r, mm_ = x
        ecc_   = max(0., min(0.9999, ecc_))
        incl_r = max(0., min(math.pi, incl_r))
        mm_    = max(0.001, mm_)

        mm_rad_s = mm_ / 60.0
        if mm_rad_s > 1e-8:
            a_km = (MU_EARTH_KM3_S2 / (mm_rad_s ** 2)) ** (1.0 / 3.0)
            perigee_km = a_km * (1.0 - ecc_) - EARTH_RADIUS_KM
            if perigee_km < _MIN_PERIGEE_ALT_KM:
                deficit = _MIN_PERIGEE_ALT_KM - perigee_km
                penalty = 1.0e6 + (deficit ** 2) * 1.0e3
                return np.full(2 * n, penalty)

        try:
            sat = _mean_elements_to_satrec(
                incl_r, raan_r, ecc_, argp_r, ma_r, mm_, epoch_jd, bstar)
        except Exception:
            return np.full(2 * n, 1e5)

        # Vectorized SGP4 — ONE C-level call instead of n Python calls.
        e_arr, r_arr, v_arr = sat.sgp4_array(jd_int_arr, jd_frac_arr)
        ok = (e_arr == 0)

        # ROOT-CAUSE FIX #2: make SGP4 outcomes VISIBLE. The old diagnostic
        # ("SGP4 call outcomes this track: {}") was always empty because this
        # vectorized path never touched the counter — so decayed/e>=1/other
        # propagation failures were silently absorbed as 1e4 residuals. Tally
        # them here so the log tells the truth.
        try:
            _u, _c = np.unique(e_arr, return_counts=True)
            for _code, _cnt in zip(_u.tolist(), _c.tolist()):
                key = "ok" if _code == 0 else f"sgp4_error_code_{int(_code)}"
                _SGP4_ERROR_COUNTS[key] += int(_cnt)
        except Exception:
            pass

        # ROOT-CAUSE FIX #5: rotate SGP4 TEME positions into GCRS to match the
        # observer frame before differencing (see precompute above).
        if _teme2gcrs is not None:
            r_arr = np.einsum('kij,kj->ki', _teme2gcrs, r_arr)

        topo = r_arr - obs_eci_arr
        tmag = np.linalg.norm(topo, axis=1)
        ok &= (tmag > 1.0)

        res = np.full(2 * n, 1e4, dtype=float)
        if np.any(ok):
            th = topo[ok] / tmag[ok, None]
            ra_pred  = np.degrees(np.arctan2(th[:, 1], th[:, 0])) % 360.
            dec_pred = np.degrees(np.arcsin(np.clip(th[:, 2], -1., 1.)))
            dra  = ((ra_pred - ra_arr[ok] + 180.) % 360. - 180.) * cos_decs[ok] * 3600.
            ddec = (dec_pred - dec_arr[ok]) * 3600.
            idx = np.where(ok)[0]
            res[2 * idx]     = dra
            res[2 * idx + 1] = ddec
        return res

    return _residuals

# ── NEW HELPER: Keplerian elements → ECI state vector ─────────────────────
def _keplerian_to_eci(incl_deg, raan_deg, ecc, argp_deg, ma_deg,
                       mm_rev_day, epoch_jd, obs_jd):
    """Convert Gauss IOD output (degrees/rev_day) to ECI (r_km, v_km_s)."""
    mu      = MU_EARTH_KM3_S2
    n_rads  = mm_rev_day * 2.0 * math.pi / 86400.0
    dt_s    = (obs_jd - epoch_jd) * 86400.0
    ma_r    = (math.radians(ma_deg) + n_rads * dt_s) % (2 * math.pi)

    E = ma_r
    for _ in range(50):
        dE = (ma_r - E + ecc * math.sin(E)) / (1.0 - ecc * math.cos(E))
        E += dE
        if abs(dE) < 1e-12:
            break

    nu  = 2.0 * math.atan2(math.sqrt(1+ecc)*math.sin(E/2),
                            math.sqrt(1-ecc)*math.cos(E/2))
    a   = (mu / n_rads**2) ** (1.0/3.0)
    p   = a * (1.0 - ecc**2)
    r_s = p / (1.0 + ecc * math.cos(nu))

    r_pf = np.array([r_s*math.cos(nu), r_s*math.sin(nu), 0.0])
    v_pf = np.array([-math.sqrt(mu/p)*math.sin(nu),
                      math.sqrt(mu/p)*(ecc+math.cos(nu)), 0.0])

    ci, si = math.cos(math.radians(incl_deg)), math.sin(math.radians(incl_deg))
    co, so = math.cos(math.radians(raan_deg)), math.sin(math.radians(raan_deg))
    cw, sw = math.cos(math.radians(argp_deg)), math.sin(math.radians(argp_deg))

    R = np.array([
        [co*cw - so*sw*ci, -co*sw - so*cw*ci,  so*si],
        [so*cw + co*sw*ci, -so*sw + co*cw*ci, -co*si],
        [sw*si,             cw*si,              ci   ],
    ])
    r_eci = R @ r_pf
    v_eci = R @ v_pf
    if np.any(np.isnan(r_eci)) or np.linalg.norm(r_eci) < EARTH_RADIUS_KM:
        return None, None
    return r_eci, v_eci


# ── NEW HELPER: osculating state → SGP4 mean elements ─────────────────────
def _osculating_to_sgp4_mean(r_km, v_km_s, epoch_jd,
                              bstar=1e-5, max_iter=15, tol_km=0.1, verbose=True):
    """
    Fixed-point iteration: osculating (r,v) → SGP4 mean elements.
    From dcajacob/tle-tailor coarse_fit.py.
    Without this, plugging raw Keplerian elements into SGP4 causes
    O(100 km) prediction errors before any fitting begins.
    """
    _MU = _wgs72_model.mu          # km³/s²
    r   = np.array(r_km,   dtype=float).copy()
    v   = np.array(v_km_s, dtype=float).copy()
    r_t = np.array(r_km,   dtype=float).copy()
    v_t = np.array(v_km_s, dtype=float).copy()
    ep_since  = epoch_jd - 2433281.5
    ep_int    = math.floor(epoch_jd)
    ep_frac   = epoch_jd - ep_int
    best_elem = None
    best_sig  = float('inf')

    for _ in range(max_iter):
        try:
            coe = _rv2coe_ext(r.tolist(), v.tolist(), _MU)
        except Exception:
            break
        p, a, ecc, incl, raan, argp, nu, ma = coe[:8]
        if a < EARTH_RADIUS_KM + 100 or a > 384400:
            break
        ecc = max(0.0, min(0.9999, float(ecc)))
        n_rad_min = math.sqrt(_MU / a**3) * 60.0
        mm_rev    = n_rad_min / REV_PER_DAY_TO_RAD_PER_MIN
        _bs = (min(bstar,1e-7) if mm_rev < 2.0 else
               min(bstar,1e-6) if mm_rev < 6.4 else bstar)
        try:
            tr = Satrec()
            tr.sgp4init(WGS84,'i',99999,float(ep_since),float(_bs),0.,0.,
                        float(ecc),
                        float(argp)%(2*math.pi),
                        max(0.,min(math.pi,float(incl))),
                        float(ma)%(2*math.pi),
                        max(1e-4,float(n_rad_min)),
                        float(raan)%(2*math.pi))
        except Exception:
            break
        ec, rt, vt = tr.sgp4(ep_int, ep_frac)
        if ec != 0:
            break
        rt = np.array(rt, dtype=float)
        vt = np.array(vt, dtype=float)
        dr = rt - r_t
        dv = vt - v_t
        sig = float(np.linalg.norm(dr))
        if sig < best_sig:
            best_sig  = sig
            best_elem = [float(incl)%(math.pi),
                         float(raan)%(2*math.pi),
                         ecc,
                         float(argp)%(2*math.pi),
                         float(ma)%(2*math.pi),
                         float(n_rad_min)]
        if sig < tol_km:
            break
        r -= dr
        v -= dv

    if best_elem is None:
        return None
    if verbose:
        print(f"    rv2mean: σ={best_sig:.3f} km  "
              f"mm={best_elem[5]/REV_PER_DAY_TO_RAD_PER_MIN:.4f} rev/day  "
              f"ecc={best_elem[2]:.5f}  incl={math.degrees(best_elem[0]):.2f}°")
    return best_elem   # [incl_rad, raan_rad, ecc, argp_rad, ma_rad, n_rad_min]


def _altitude_scan_iod(jd_clean, ra_clean, dec_clean,
                        lat_deg, lon_deg, alt_m,
                        orbit_class, epoch_jd, n_scan=150):
    """
    Short-arc IOD by altitude scan.  Replaces classical Gauss IOD for
    arcs shorter than ~1% of the orbital period where neg_range occurs.
    """
    mu = MU_EARTH_KM3_S2
    ALT_RANGES = {
        "GEO": (34000, 37000), "MEO": (2000, 34000),
        "LEO": (160,   2000),  None:  (160,  42000),
    }
    if orbit_class == "LEO":
        n_scan = max(n_scan, 300)   # much finer scan for narrow altitude band
    h_lo, h_hi = ALT_RANGES.get(orbit_class, ALT_RANGES[None])

    n   = len(jd_clean)
    i2  = n // 2
    t2  = jd_clean[i2]
    L2  = radec_to_vec(ra_clean[i2], dec_clean[i2])
    R2  = _observatory_eci(t2, lat_deg, lon_deg, alt_m)
    r_obs   = float(np.linalg.norm(R2))
    cos_zen = float(np.dot(R2 / r_obs, L2))

    dt_s  = (jd_clean[-1] - jd_clean[0]) * 86400.0
    if dt_s < 1.0:
        return None
    dra   = (ra_clean[-1]  - ra_clean[0])
    ddec  = (dec_clean[-1] - dec_clean[0])
    if dra  >  180: dra  -= 360
    if dra  < -180: dra  += 360
    ra_r  = math.radians(ra_clean[i2])
    dec_r = math.radians(dec_clean[i2])
    E_sky = np.array([-math.sin(ra_r),  math.cos(ra_r),  0.])
    N_sky = np.array([-math.sin(dec_r)*math.cos(ra_r),
                      -math.sin(dec_r)*math.sin(ra_r),
                       math.cos(dec_r)])
    ra_dot_rad_s  = math.radians(dra)  / dt_s
    dec_dot_rad_s = math.radians(ddec) / dt_s
    cos_dec       = math.cos(dec_r)

    # Observer velocity (numerical)
    R2p   = _observatory_eci(t2 + 10./86400., lat_deg, lon_deg, alt_m)
    v_obs = (R2p - R2) / 10.0

    best_x0, best_rmse = None, float('inf')

    for h_km in np.linspace(h_lo, h_hi, n_scan):
        r_sat = EARTH_RADIUS_KM + h_km
        disc  = r_obs**2 * cos_zen**2 - r_obs**2 + r_sat**2
        if disc < 0: continue
        rho   = -r_obs * cos_zen + math.sqrt(disc)
        if rho <= 0: continue

        r_vec = R2 + rho * L2
        r_mag = float(np.linalg.norm(r_vec))
        if r_mag < EARTH_RADIUS_KM + 100: continue

        v_circ    = math.sqrt(mu / r_mag)
        v_angular = rho * (ra_dot_rad_s * cos_dec * E_sky + dec_dot_rad_s * N_sky)
        v_approx  = v_obs + v_angular
        v_mag     = float(np.linalg.norm(v_approx))
        if v_mag < 0.01: continue
        v_vec = v_approx * (v_circ / v_mag)

        mean_elems = _osculating_to_sgp4_mean(r_vec, v_vec, epoch_jd, verbose=False)
        if mean_elems is None:
            elems = _state_to_keplerian(r_vec, v_vec)
            if elems is None: continue
            ig, rg, eg, ag, mg, mmg = elems
            mean_elems = [
                math.radians(ig), math.radians(rg)%(2*math.pi),
                max(0., min(0.9999, eg)),
                math.radians(ag)%(2*math.pi), math.radians(mg)%(2*math.pi),
                mmg * REV_PER_DAY_TO_RAD_PER_MIN,
            ]

        try:
            tr = Satrec()
            tr.sgp4init(WGS84,'i',99999, epoch_jd-2433281.5, 1e-5,0.,0.,
                float(mean_elems[2]),
                float(mean_elems[3])%(2*math.pi),
                max(0.,min(math.pi,float(mean_elems[0]))),
                float(mean_elems[4])%(2*math.pi),
                max(1e-4,float(mean_elems[5])),
                float(mean_elems[1])%(2*math.pi))
        except Exception: continue

        sse, nv = 0.0, 0
        for jd_o, ra_o, dec_o in zip(jd_clean, ra_clean, dec_clean):
            ji = math.floor(jd_o); jf = jd_o - ji
            ec, rt, _ = tr.sgp4(ji, jf)
            if ec != 0: sse += 1e8; continue
            Ro  = _observatory_eci(jd_o, lat_deg, lon_deg, alt_m)
            top = np.array(rt) - Ro
            tm  = float(np.linalg.norm(top))
            if tm < 1: sse += 1e8; continue
            th  = top / tm
            rp  = math.degrees(math.atan2(th[1],th[0])) % 360.
            dp  = math.degrees(math.asin(max(-1.,min(1.,th[2]))))
            dr  = ((rp-ra_o+180)%360)-180
            dd  = dp - dec_o
            cd  = math.cos(math.radians(dec_o))
            sse += (dr*cd*3600)**2 + (dd*3600)**2
            nv  += 1

        if nv == 0: continue
        rmse = math.sqrt(sse / nv)

        # Reject solutions whose elements fall outside orbit-class bounds.
        # Clipping out-of-bounds ecc/mm later creates incoherent element sets.
        _mm_rd = mean_elems[5] / REV_PER_DAY_TO_RAD_PER_MIN
        _ecc   = mean_elems[2]
        _ALT_MM = {"GEO": (0.90,1.10), "MEO": (1.10,6.40),
                   "LEO": (6.40,17.0), None: (0.5,15.5)}
        _ALT_EC = {"GEO": (0.0,0.05),  "MEO": (0.0,0.50),
                   "LEO": (0.0,0.05),  None: (0.0,0.99)}
        _mm_lo, _mm_hi = _ALT_MM.get(orbit_class, _ALT_MM[None])
        _ec_lo, _ec_hi = _ALT_EC.get(orbit_class, _ALT_EC[None])
        if not (_mm_lo <= _mm_rd <= _mm_hi and _ec_lo <= _ecc <= _ec_hi):
            continue   # wrong orbit class — skip, don't accept as best

        if rmse < best_rmse:
            best_rmse, best_x0 = rmse, list(mean_elems)

    if best_x0 is None:
        return None
    print(f"    AltitudeScanIOD: RMSE={best_rmse:.1f}\"  "
          f"mm={best_x0[5]/REV_PER_DAY_TO_RAD_PER_MIN:.4f} rev/day")
    return best_x0

def _admissible_region_iod(jd_clean, ra_clean, dec_clean,
                            lat_deg, lon_deg, alt_m,
                            orbit_class, epoch_jd,
                            n_rho=40, n_rhodot=20):
    """
    2D admissible region IOD (Milani et al. 2004).
    Scans over (range ρ, range-rate ρ̇) at the middle observation epoch.
    SOTA for angles-only short-arc IOD.
    """
    mu  = MU_EARTH_KM3_S2
    n   = len(jd_clean)
    if n < 3:
        return None
    i2  = n // 2
    t2  = jd_clean[i2]
    L2  = radec_to_vec(ra_clean[i2], dec_clean[i2])
    R2  = _observatory_eci(t2, lat_deg, lon_deg, alt_m)
    r_obs = float(np.linalg.norm(R2))

    dt_s = (jd_clean[-1] - jd_clean[0]) * 86400.0
    if dt_s < 1.0:
        return None

    dra_dt  = (ra_clean[-1]  - ra_clean[0])  / dt_s
    ddec_dt = (dec_clean[-1] - dec_clean[0]) / dt_s
    if dra_dt  >  180/dt_s: dra_dt  -= 360/dt_s
    if dra_dt  < -180/dt_s: dra_dt  += 360/dt_s

    ALT_RANGES = {
        "GEO": (30000, 42000), "MEO": (2000, 35000),
        "LEO": (160,   2000),  None:  (160,  42000),
    }
    # For LEO use finer scan — altitude range is only 1840 km vs 40000 for MEO
    if orbit_class == "LEO":
        n_rho   = max(n_rho,   80)   # finer altitude grid
        n_rhodot = max(n_rhodot, 30)  # finer range-rate grid
    h_lo, h_hi = ALT_RANGES.get(orbit_class, ALT_RANGES[None])

    L2_unit   = L2 / np.linalg.norm(L2)
    cos_z     = float(np.dot(R2 / r_obs, L2_unit))
    rho_lo = max(50., -r_obs * cos_z
                 + math.sqrt(max(0., (EARTH_RADIUS_KM + h_lo)**2
                                 - r_obs**2 * (1 - cos_z**2))))
    rho_hi = min(45000., -r_obs * cos_z
                 + math.sqrt(max(0., (EARTH_RADIUS_KM + h_hi)**2
                                 - r_obs**2 * (1 - cos_z**2))))
    if rho_hi <= rho_lo:
        rho_lo, rho_hi = 200., 42000.

    R2p   = _observatory_eci(t2 + 10./86400., lat_deg, lon_deg, alt_m)
    v_obs = (R2p - R2) / 10.0

    ra_r  = math.radians(ra_clean[i2])
    dec_r = math.radians(dec_clean[i2])
    E_sky = np.array([-math.sin(ra_r),  math.cos(ra_r),  0.])
    N_sky = np.array([-math.sin(dec_r)*math.cos(ra_r),
                      -math.sin(dec_r)*math.sin(ra_r),
                       math.cos(dec_r)])
    v_angular = (dra_dt  * math.pi/180. * math.cos(math.radians(ra_clean[i2]))
                 * E_sky
                 + ddec_dt * math.pi/180. * N_sky)

    best_x0, best_rmse = None, float('inf')

    for rho in np.linspace(rho_lo, rho_hi, n_rho):
        r_vec = R2 + rho * L2_unit
        r_mag = float(np.linalg.norm(r_vec))
        if r_mag < EARTH_RADIUS_KM + 100:
            continue

        v_circ    = math.sqrt(mu / r_mag)
        rhodot_lo = -v_circ * 0.8
        rhodot_hi =  v_circ * 0.8

        for rhodot in np.linspace(rhodot_lo, rhodot_hi, n_rhodot):
            v_perp = rho * v_angular
            v_los  = rhodot * L2_unit
            v_vec  = v_perp + v_los + v_obs

            energy = 0.5*float(np.dot(v_vec, v_vec)) - mu/r_mag
            if energy >= 0:
                continue

            h_vec = np.cross(r_vec, v_vec)
            h_mag = float(np.linalg.norm(h_vec))
            if h_mag < 1.0:
                continue

            e_vec   = (1./mu)*((float(np.dot(v_vec,v_vec)) - mu/r_mag)*r_vec
                                - float(np.dot(r_vec,v_vec))*v_vec)
            ecc_try = float(np.linalg.norm(e_vec))
            if ecc_try >= 1.0:
                continue

            a_try   = -mu / (2.*energy)
            perigee = a_try*(1.-ecc_try) - EARTH_RADIUS_KM
            if perigee < 150.:
                continue

            mean_elems = _osculating_to_sgp4_mean(
                r_vec, v_vec, epoch_jd, verbose=False
            )
            if mean_elems is None:
                elems = _state_to_keplerian(r_vec, v_vec)
                if elems is None:
                    continue
                ig, rg, eg, ag, mg, mmg = elems
                mean_elems = [
                    math.radians(ig), math.radians(rg)%(2*math.pi),
                    max(0., min(0.9999, eg)),
                    math.radians(ag)%(2*math.pi),
                    math.radians(mg)%(2*math.pi),
                    mmg * REV_PER_DAY_TO_RAD_PER_MIN,
                ]

            try:
                tr = Satrec()
                tr.sgp4init(WGS84, 'i', 99999, epoch_jd-2433281.5,
                    1e-5, 0., 0.,
                    float(mean_elems[2]),
                    float(mean_elems[3])%(2*math.pi),
                    max(0., min(math.pi, float(mean_elems[0]))),
                    float(mean_elems[4])%(2*math.pi),
                    max(1e-4, float(mean_elems[5])),
                    float(mean_elems[1])%(2*math.pi))
            except Exception:
                continue

            sse, nv = 0.0, 0
            for jd_o, ra_o, dec_o in zip(jd_clean, ra_clean, dec_clean):
                ji = math.floor(jd_o); jf = jd_o - ji
                ec, rt, _ = tr.sgp4(ji, jf)
                if ec != 0:
                    sse += 1e8; continue
                Ro  = _observatory_eci(jd_o, lat_deg, lon_deg, alt_m)
                top = np.array(rt) - Ro
                tm  = float(np.linalg.norm(top))
                if tm < 1:
                    sse += 1e8; continue
                th  = top / tm
                rp  = math.degrees(math.atan2(th[1], th[0])) % 360.
                dp  = math.degrees(math.asin(max(-1., min(1., th[2]))))
                dr  = ((rp - ra_o + 180) % 360) - 180
                dd  = dp - dec_o
                cd  = math.cos(math.radians(dec_o))
                sse += (dr*cd*3600)**2 + (dd*3600)**2
                nv  += 1

            if nv == 0:
                continue
            rmse = math.sqrt(sse / nv)

            # Reject solutions outside orbit-class bounds
            _mm_rd = mean_elems[5] / REV_PER_DAY_TO_RAD_PER_MIN
            _ecc   = mean_elems[2]
            _ALT_MM = {"GEO": (0.90,1.10), "MEO": (1.10,6.40),
                       "LEO": (6.40,17.0),  None: (0.5,15.5)}
            _ALT_EC = {"GEO": (0.0,0.05),  "MEO": (0.0,0.50),
                       "LEO": (0.0,0.05),   None: (0.0,0.99)}
            _mm_lo, _mm_hi = _ALT_MM.get(orbit_class, _ALT_MM[None])
            _ec_lo, _ec_hi = _ALT_EC.get(orbit_class, _ALT_EC[None])
            if not (_mm_lo <= _mm_rd <= _mm_hi and _ec_lo <= _ecc <= _ec_hi):
                continue

            if rmse < best_rmse:
                best_rmse, best_x0 = rmse, list(mean_elems)

    if best_x0 is None:
        return None

    print(f"    AdmissibleRegionIOD: RMSE={best_rmse:.1f}\"  "
          f"mm={best_x0[5]/REV_PER_DAY_TO_RAD_PER_MIN:.4f} rev/day  "
          f"n={n} obs")
    return best_x0



# ── NEW HELPER: catalog-seeded LM (cbassa/sattools strategy) ──────────────
def catalog_seeded_refine(l1, l2, jd_arr, ra_arr, dec_arr,
                           epoch_jd, lat_deg, lon_deg, alt_m, bounds_rad):
    """
    Extract mean elements from catalog TLE → run LM directly.
    O(50-200) SGP4 calls vs O(80 000) for blind DE.
    """
    try:
        cat_sat = Satrec.twoline2rv(l1, l2)
    except Exception as e:
        print(f"    CatalogSeed: parse failed ({e})")
        return None, float('nan')

    lb = np.array([b[0] for b in bounds_rad])
    ub = np.array([b[1] for b in bounds_rad])
    # Propagate catalog TLE to epoch_jd so all elements (especially mo)
    # are at the correct epoch — fixes the epoch misalignment bug.
    ep_int  = math.floor(epoch_jd)
    ep_frac = epoch_jd - ep_int
    e_code, r_at_ep, v_at_ep = cat_sat.sgp4(ep_int, ep_frac)
    if e_code != 0:
        print(f"    CatalogSeed: propagation to epoch failed (code {e_code})")
        return None, float('nan')

    mean_at_ep = _osculating_to_sgp4_mean(r_at_ep, v_at_ep, epoch_jd)
    if mean_at_ep is None:
        print("    CatalogSeed: osc→mean conversion failed")
        return None, float('nan')

    x0 = np.clip(np.array(mean_at_ep, dtype=float), lb, ub)
    print(f"    CatalogSeed: x0 at epoch_jd  "
          f"mm={mean_at_ep[5]/REV_PER_DAY_TO_RAD_PER_MIN:.4f} rev/day")

    # Pre-check using direct SGP4 propagation (not _build_residual_fn)
    # to avoid any residual-function unit issues during validation.
    _sep2, _nv = 0.0, 0
    for _jd_o, _ra_o, _dec_o in zip(jd_arr, ra_arr, dec_arr):
        _ji  = math.floor(_jd_o)
        _jf  = _jd_o - _ji
        _ec, _rt, _ = cat_sat.sgp4(_ji, _jf)
        if _ec != 0:
            _sep2 += 50000**2; continue
        _Ro  = _observatory_eci(_jd_o, lat_deg, lon_deg, alt_m)
        _top = np.array(_rt) - _Ro
        _tm  = float(np.linalg.norm(_top))
        if _tm < 1:
            _sep2 += 50000**2; continue
        _th   = _top / _tm
        _rap  = math.degrees(math.atan2(_th[1], _th[0])) % 360.
        _dep  = math.degrees(math.asin(max(-1., min(1., _th[2]))))
        _dra  = (((_rap - _ra_o + 180) % 360) - 180)
        _ddec = _dep - _dec_o
        _cd   = math.cos(math.radians(_dec_o))
        _sep2 += (_dra*_cd*3600)**2 + (_ddec*3600)**2
        _nv   += 1
    if _nv == 0:
        return None, float('nan')
    rmse0 = math.sqrt(_sep2 / _nv)

    if rmse0 > 50000.0:
        print(f"    CatalogSeed: pre-check {rmse0:.0f}\" > 50 000\" — skip")
        return None, float('nan')
    print(f"    CatalogSeed: pre-check RMSE={rmse0:.1f}\" → LM …")

    res_fn = _build_residual_fn(jd_arr, ra_arr, dec_arr,
                                epoch_jd, lat_deg, lon_deg, alt_m)

    _loss_c = 'linear' if rmse0 > 1000. else 'soft_l1'
    _fsc_c  = max(300., rmse0 * 0.5)
    _loss_c = 'linear' if rmse0 > 1000. else 'soft_l1'
    _fsc_c  = max(300., rmse0 * 0.5)
    try:
        lm = least_squares(res_fn, x0, method='trf', loss=_loss_c,
                           f_scale=_fsc_c, bounds=(lb.tolist(), ub.tolist()),
                           max_nfev=600, ftol=1e-8, xtol=1e-8, gtol=1e-8)
    except Exception as e:
        print(f"    CatalogSeed LM failed: {e}")
        return None, float('nan')

    rmse_f = float(np.sqrt(np.mean(res_fn(lm.x)**2)))
    print(f"    CatalogSeed: final RMSE={rmse_f:.2f}\"  nfev={lm.nfev}")
    ir, rr, ec, ar, mr, mm = lm.x
    return ([math.degrees(ir)%180., math.degrees(rr)%360.,
             max(0.,min(0.9999,ec)),
             math.degrees(ar)%360., math.degrees(mr)%360.,
             mm/REV_PER_DAY_TO_RAD_PER_MIN], rmse_f)


# ── NEW HELPER: fast angular pre-search for catalog seed ──────────────────
def _pre_search_catalog(obs_times, raw_coords, ts, observer,
                         catalog, orbit_class):
    """
    Propagate the orbit-class–filtered catalog to the mid-point observation
    and return the nearest TLE (l1, l2, angular_sep_deg).
    """
    if not catalog or not raw_coords or not obs_times:
        return None, None, float('inf')
    mid   = len(obs_times) // 2
    t_mid = obs_times[mid]
    ra_m, dec_m = raw_coords[mid]
    cos_d = math.cos(math.radians(dec_m))
    cat_f = filter_catalog_by_orbit_class(catalog, orbit_class)
    best_l1, best_l2, best_sep = None, None, float('inf')
    for sat_name, l1, l2, _ in cat_f:
        try:
            sat = EarthSatellite(l1, l2, sat_name, ts)
            pr, pd, _ = (sat - observer).at(t_mid).radec()
            if math.isnan(pr.hours) or math.isnan(pd.degrees):
                continue
            sep = math.sqrt(((pr.hours*15.-ra_m)*cos_d)**2 +
                            (pd.degrees-dec_m)**2)
            if sep < best_sep:
                best_sep, best_l1, best_l2 = sep, l1, l2
        except Exception:
            continue
    return best_l1, best_l2, best_sep


def _fit_observability_metric(cov_params, mm_rad_min=None):
    """
    Returns (sigma_mm_rev_day, relative_sigma_mm, is_well_observed).

    Gates solely on the fitted mean-motion (hence semi-major-axis, hence
    RANGE) uncertainty as a fraction of the fitted mean motion itself.
    Replaces the earlier full-covariance condition-number check, which was
    dominated by the small-eccentricity coordinate singularity (argp/ma
    individually ill-defined as e->0) -- real-run evidence showed sigma_mm
    consistently tiny (range well-constrained) while raw condition number
    was always >>1e8 regardless, making it useless as a gate.
    """
    if cov_params is None or mm_rad_min is None or mm_rad_min <= 0:
        return float('nan'), float('nan'), False
    try:
        sigma_mm_rad_min = math.sqrt(max(0., cov_params[5, 5]))
        sigma_mm_rev_day = sigma_mm_rad_min / REV_PER_DAY_TO_RAD_PER_MIN
        mm_rev_day = mm_rad_min / REV_PER_DAY_TO_RAD_PER_MIN
        rel_sigma = sigma_mm_rev_day / mm_rev_day if mm_rev_day > 0 else float('inf')
        # 2% relative uncertainty on mean motion -> ~1.3% on semi-major axis
        # (a ~ n^(-2/3)) -> sub-percent range uncertainty at epoch.
        is_well_observed = rel_sigma < 0.02
        return sigma_mm_rev_day, rel_sigma, is_well_observed
    except Exception:
        return float('nan'), float('nan'), False
    

def optimise_tle(obs_times, obs_vecs, epoch_str, ts, observer,
                 arc_duration_s: float = 0.0, orbit_class: str = None,
                 catalog_l1: str = None, catalog_l2: str = None,
                 ransac_thresh_arcsec: float = 120.0):
    """
    3-stage orbit determination:
      A) Gauss IOD (restored) → _osculating_to_sgp4_mean → LM
      B) Catalog-seeded LM  (cbassa/sattools strategy)
      C) Seeded DE fallback (Gauss estimate seeds 80% of population)
    """
    # ── Build RA/Dec/JD arrays (unchanged) ────────────────────────────────
    ra_list, dec_list, jd_list = [], [], []
    for t, vec in zip(obs_times, obs_vecs):
        from astropy.time import Time as _ATfix
        _t_utc = _ATfix(t.whole + t.tt_fraction, format='jd', scale='tt').utc
        jd_list.append(float(_t_utc.jd))
        ra_r  = math.atan2(float(vec[1]), float(vec[0]))
        dec_r = math.asin(max(-1., min(1., float(vec[2]))))
        ra_list.append(math.degrees(ra_r) % 360.)
        dec_list.append(math.degrees(dec_r))

    jd_arr  = np.asarray(jd_list,  dtype=float)
    ra_arr  = np.asarray(ra_list,  dtype=float)
    dec_arr = np.asarray(dec_list, dtype=float)
    n_orig  = len(jd_arr)

    lat_deg = float(observer.latitude.degrees)
    lon_deg = float(observer.longitude.degrees)
    alt_m   = float(observer.elevation.m)

    yy       = int(epoch_str[:2])
    year     = (2000 + yy) if yy < 57 else (1900 + yy)
    doy_f    = float(epoch_str[2:])
    epoch_jd = float(Time(f"{year}-01-01T00:00:00",
                          format='isot', scale='utc').jd) + (doy_f - 1.0)

    # ── RANSAC (unchanged) ────────────────────────────────────────────────
    inlier_mask, ra_clean, dec_clean, jd_clean = ransac_reject_outliers(
        jd_arr, ra_arr, dec_arr,
        inlier_thresh_arcsec=ransac_thresh_arcsec,
        n_iterations=500, min_inliers=3,
    )
    n_clean = len(jd_clean)
    print(f"    RANSAC: {n_orig} → {n_clean} observations (inliers)")

    if n_clean < MIN_RANSAC_INLIERS:
        print(f"    ⚠  Only {n_clean} RANSAC inliers — track is noise.")
        return [0., 0., 0.001, 0., 0., 15.], float("nan"), None

    split_idx = int(n_clean * 0.6)
    if n_clean >= 5 and split_idx >= 3:
        jd_fit, ra_fit, dec_fit   = (jd_clean[:split_idx],
                                     ra_clean[:split_idx], dec_clean[:split_idx])
        jd_test, ra_test, dec_test = (jd_clean[split_idx:],
                                      ra_clean[split_idx:], dec_clean[split_idx:])
        print(f"    Validation split: train {split_idx}, test {n_clean-split_idx}")
    else:
        jd_fit,  ra_fit,  dec_fit  = jd_clean, ra_clean, dec_clean
        jd_test, ra_test, dec_test = jd_clean, ra_clean, dec_clean

    if len(jd_fit) > 60:
        idx    = np.round(np.linspace(0, len(jd_fit)-1, 60)).astype(int)
        jd_fit = jd_fit[idx]; ra_fit = ra_fit[idx]; dec_fit = dec_fit[idx]

    # ── Bounds (unchanged class-conditional logic) ────────────────────────
    CLASS_MM_BOUNDS_REV_DAY = {
        "GEO": (0.88, 1.12), "MEO": (1.10, 6.40),
        "LEO": (6.40, 17.0), None:  (0.5,  15.5),
    }
    CLASS_ECC_BOUNDS = {
        "GEO": (0.0, 0.08), "MEO": (0.0, 0.70),
        "LEO": (0.0, 0.05), None:  (0.0, 0.99),
    }
    ecc_lo, ecc_hi = CLASS_ECC_BOUNDS.get(orbit_class, CLASS_ECC_BOUNDS[None])
    if arc_duration_s < SHORT_ARC_THRESHOLD_S:
        ecc_lo, ecc_hi = 0.0, min(0.05, ecc_hi)
        print(f"    IOD: SHORT-ARC ({arc_duration_s:.1f}s) → ecc ∈ [0.00,{ecc_hi:.2f}]")
    else:
        print(f"    IOD: LONG-ARC ({arc_duration_s:.1f}s) → ecc ∈ [{ecc_lo:.2f},{ecc_hi:.2f}] "
              f"(class={orbit_class})")
    mm_lo_rev, mm_hi_rev = CLASS_MM_BOUNDS_REV_DAY.get(
        orbit_class, CLASS_MM_BOUNDS_REV_DAY[None])
    mm_lo = mm_lo_rev * REV_PER_DAY_TO_RAD_PER_MIN
    mm_hi = mm_hi_rev * REV_PER_DAY_TO_RAD_PER_MIN
    print(f"    Mean motion bounds: [{mm_lo_rev:.2f},{mm_hi_rev:.2f}] rev/day "
          f"(class={orbit_class})")

    bounds_rad = [
        (0., math.pi), (0., 2*math.pi),
        (ecc_lo, ecc_hi),
        (0., 2*math.pi), (0., 2*math.pi),
        (mm_lo, mm_hi),
    ]
    lb = np.array([b[0] for b in bounds_rad])
    ub = np.array([b[1] for b in bounds_rad])

    from scipy.optimize import NonlinearConstraint as _NLC
    residual_fn = _build_residual_fn(
        jd_fit, ra_fit, dec_fit, epoch_jd, lat_deg, lon_deg, alt_m)

    if not callable(residual_fn):
        print("    ❌ _build_residual_fn returned None — "
              "check indentation of 'return _residuals' at end of function")
        return [0., 0., 0.001, 0., 0., 15.], float("nan"), None

    def cost_scalar(x):
        r = residual_fn(x); k = 300.
        return float(np.sum(k**2 * (np.sqrt(1. + (r/k)**2) - 1.)))

    def _perigee_constraint(x):
        _, _, ecc_c, _, _, mm_c = x
        mm_rad_s = mm_c / 60.0
        if mm_rad_s <= 1e-8: return -1e6
        a_c = (MU_EARTH_KM3_S2 / mm_rad_s**2)**(1./3.)
        return a_c*(1.-ecc_c) - EARTH_RADIUS_KM

    _perigee_nlc = _NLC(_perigee_constraint, _MIN_PERIGEE_FLOOR_KM, np.inf)

    best_x    = None
    best_rmse = float('nan')
    cov_params = None

    # ═══════════════════════════════════════════════════════════════════════
    #  STAGE A — GAUSS IOD SEED → LM
    #  This was removed before; restoring it is the single biggest fix.
    # ═══════════════════════════════════════════════════════════════════════
    gauss_x0 = None
    if n_clean >= 3:
        print("    Stage A: Gauss IOD seed …")
        gauss_result = gauss_iod(
            jd_clean.tolist(), ra_clean.tolist(), dec_clean.tolist(),
            lat_deg, lon_deg, alt_m)

        if gauss_result is not None:
            ig, rg, eg, ag, mg, mmg = gauss_result
            print(f"    Gauss IOD: incl={ig:.2f}°  ecc={eg:.4f}  mm={mmg:.4f} rev/day")

            # Convert osculating → SGP4 mean elements (tle-tailor coarse_fit)
            mid_jd    = jd_clean[n_clean // 2]
            r_eci, v_eci = _keplerian_to_eci(ig, rg, eg, ag, mg, mmg,
                                              epoch_jd, mid_jd)
            mean_elem = None
            if r_eci is not None:
                mean_elem = _osculating_to_sgp4_mean(r_eci, v_eci, epoch_jd)

            if mean_elem is not None:
                x_cand = np.array(mean_elem, dtype=float)
            else:
                # Fallback: raw Gauss elements (still far better than no seed)
                x_cand = np.array([
                    math.radians(ig),
                    math.radians(rg) % (2*math.pi),
                    max(ecc_lo, min(ecc_hi, eg)),
                    math.radians(ag) % (2*math.pi),
                    math.radians(mg) % (2*math.pi),
                    mmg * REV_PER_DAY_TO_RAD_PER_MIN,
                ], dtype=float)

            gauss_x0 = np.clip(x_cand, lb, ub)
            if not np.array_equal(gauss_x0, x_cand):
                print("    ⚠  Gauss seed clipped to bounds")

            # LM from Gauss seed — fast path, avoids DE if it works
            try:
                _rmse_g0 = float(np.sqrt(np.mean(residual_fn(gauss_x0)**2)))
                _loss_g  = 'linear' if _rmse_g0 > 1000. else 'soft_l1'
                _fsc_g   = max(300., _rmse_g0 * 0.5)
                _rmse_g0 = float(np.sqrt(np.mean(residual_fn(gauss_x0)**2)))
                _loss_g  = 'linear' if _rmse_g0 > 1000. else 'soft_l1'
                _fsc_g   = max(300., _rmse_g0 * 0.5)
                lm_g = least_squares(
                    residual_fn, gauss_x0, method='trf', loss=_loss_g,
                    f_scale=_fsc_g, bounds=(lb.tolist(), ub.tolist()),
                    max_nfev=800, ftol=1e-8, xtol=1e-8, gtol=1e-8)
                rmse_g = float(np.sqrt(np.mean(residual_fn(lm_g.x)**2)))
                print(f"    Gauss→LM RMSE: {rmse_g:.2f}\"  nfev={lm_g.nfev}")
                if rmse_g < MAX_ACCEPTABLE_DE_RMSE_ARCSEC:
                    best_x, best_rmse = lm_g.x, rmse_g
                    try:
                        J = lm_g.jac
                        s2 = lm_g.cost / max(1, len(lm_g.fun)-6)
                        cov_params = s2 * np.linalg.pinv(J.T @ J)
                    except Exception:
                        cov_params = None
            except Exception as e:
                print(f"    Gauss→LM error: {e}")
        else:
            print("    ⚠  Gauss IOD: all rho2 trials rejected — trying admissible region IOD …")
            alt_x0 = _admissible_region_iod(
                jd_clean, ra_clean, dec_clean,
                lat_deg, lon_deg, alt_m, orbit_class, epoch_jd,
                n_rho=40, n_rhodot=20
            )
            if alt_x0 is None:
                print("    AdmissibleRegionIOD: no solution — trying altitude scan …")
                alt_x0 = _altitude_scan_iod(
                    jd_clean, ra_clean, dec_clean,
                    lat_deg, lon_deg, alt_m, orbit_class, epoch_jd)
            if alt_x0 is not None:
                gauss_x0 = np.clip(np.array(alt_x0, dtype=float), lb, ub)
                try:
                    _rmse_a0 = float(np.sqrt(np.mean(residual_fn(gauss_x0)**2)))
                    _loss_a  = 'linear' if _rmse_a0 > 1000. else 'soft_l1'
                    _fsc_a   = max(300., _rmse_a0 * 0.5)
                    lm_a = least_squares(
                        residual_fn, gauss_x0, method='trf', loss=_loss_a,
                        f_scale=_fsc_a, bounds=(lb.tolist(), ub.tolist()),
                        max_nfev=800, ftol=1e-8, xtol=1e-8, gtol=1e-8)
                    rmse_a = float(np.sqrt(np.mean(residual_fn(lm_a.x)**2)))
                    print(f"    AltScan→LM RMSE: {rmse_a:.2f}\"  nfev={lm_a.nfev}")
                    if rmse_a < MAX_ACCEPTABLE_DE_RMSE_ARCSEC:
                        best_x, best_rmse = lm_a.x, rmse_a
                        try:
                            J  = lm_a.jac
                            s2 = lm_a.cost / max(1, len(lm_a.fun)-6)
                            cov_params = s2 * np.linalg.pinv(J.T @ J)
                        except Exception:
                            cov_params = None
                except Exception as e:
                    print(f"    AltScan→LM error: {e}")

    # ═══════════════════════════════════════════════════════════════════════
    #  STAGE B — CATALOG-SEEDED LM  (cbassa/sattools strategy)
    # ═══════════════════════════════════════════════════════════════════════
    if (math.isnan(best_rmse) or best_rmse > MAX_ACCEPTABLE_DE_RMSE_ARCSEC) \
            and catalog_l1 is not None and catalog_l2 is not None:
        print("    Stage B: catalog-seeded LM …")
        cp, cr = catalog_seeded_refine(
            catalog_l1, catalog_l2,
            jd_fit, ra_fit, dec_fit,
            epoch_jd, lat_deg, lon_deg, alt_m, bounds_rad)
        if cp is not None and not math.isnan(cr):
            if math.isnan(best_rmse) or cr < best_rmse:
                best_x = np.clip(np.array([
                    math.radians(cp[0]), math.radians(cp[1])%(2*math.pi),
                    cp[2],
                    math.radians(cp[3])%(2*math.pi),
                    math.radians(cp[4])%(2*math.pi),
                    cp[5]*REV_PER_DAY_TO_RAD_PER_MIN,
                ], dtype=float), lb, ub)
                best_rmse  = cr
                cov_params = None

    # ═══════════════════════════════════════════════════════════════════════
    #  STAGE C — SEEDED DE FALLBACK
    # ═══════════════════════════════════════════════════════════════════════
    _DE_WORTH_TRYING_RMSE = 20000.0  # arcsec; above this the arc is
                                       # unconstrained and DE won't help

    if (math.isnan(best_rmse)
            or (MAX_ACCEPTABLE_DE_RMSE_ARCSEC < best_rmse < _DE_WORTH_TRYING_RMSE)):
        print("    Stage C: Differential Evolution (seeded) …")

        if gauss_x0 is not None:
            rng   = np.random.default_rng(seed=42)
            pop_n = 15
            ng    = int(0.8 * pop_n)
            pert  = rng.uniform(-0.15, 0.15, (ng, 6)) * (ub - lb)
            init_pop = np.vstack([
                np.clip(gauss_x0 + pert, lb, ub),
                rng.uniform(lb, ub, (pop_n - ng, 6))
            ])
        else:
            init_pop = 'latinhypercube'

        best_de = None
        _N_DE_STARTS = 1
        for _seed in range(_N_DE_STARTS):
            de_result = differential_evolution(
                cost_scalar, bounds_rad,
                strategy='best1bin', popsize=15, maxiter=60,
                tol=1e-7, seed=_seed, updating='deferred',
                mutation=(0.3, 1.7), recombination=0.9,
                polish=False, constraints=(_perigee_nlc,),
                init=init_pop if _seed == 0 else 'latinhypercube',
            )
            if best_de is None or de_result.fun < best_de.fun:
                best_de = de_result

        de_rmse = float(np.sqrt(np.mean(residual_fn(best_de.x)**2)))
        print(f"    DE (best of {_N_DE_STARTS}): RMSE≈{de_rmse:.2f}\"  "
              f"cost={best_de.fun:.4f}  nfev={best_de.nfev}")

        _bm = 0.02
        for _i, (_lo, _hi) in enumerate(bounds_rad):
            _sp = _hi - _lo
            if _sp > 0 and (best_de.x[_i]-_lo < _bm*_sp or
                             _hi-best_de.x[_i] < _bm*_sp):
                print(f"    ⚠  DE param[{_i}]={best_de.x[_i]:.6f} within "
                      f"{_bm*100:.0f}% of bound [{_lo:.6f},{_hi:.6f}]")

        print(f"    SGP4 call outcomes this track: {dict(_SGP4_ERROR_COUNTS)}")
        _SGP4_ERROR_COUNTS.clear()

        if math.isnan(best_rmse) or de_rmse < best_rmse:
            best_x, best_rmse = best_de.x, de_rmse
            cov_params = None

        if best_rmse > MAX_ACCEPTABLE_DE_RMSE_ARCSEC:
            print(f"    ⚠  DE RMSE {best_rmse:.0f}\" > threshold — skipping LM.")
            ir,rr,ec,ar,mr,mm = best_x
            return ([math.degrees(ir)%180., math.degrees(rr)%360.,
                     max(0.,min(0.9999,ec)),
                     math.degrees(ar)%360., math.degrees(mr)%360.,
                     mm/REV_PER_DAY_TO_RAD_PER_MIN],
                    best_rmse, None)
    elif not math.isnan(best_rmse):
        print(f"    Stage C skipped: pre-DE RMSE {best_rmse:.0f}\" "
              f"indicates an unconstrained arc — DE would not help.")

    # ── Final LM polish ───────────────────────────────────────────────────
    print("    Final LM polish …")
    try:
        _rmse_polish = float(np.sqrt(np.mean(residual_fn(best_x)**2)))
        _loss_p = 'linear' if _rmse_polish > 1000. else 'soft_l1'
        _fsc_p  = max(300., _rmse_polish * 0.5)
        _rmse_p0 = float(np.sqrt(np.mean(residual_fn(best_x)**2)))
        _loss_p  = 'linear' if _rmse_p0 > 1000. else 'soft_l1'
        _fsc_p   = max(300., _rmse_p0 * 0.5)
        lm_f = least_squares(
            residual_fn, np.clip(best_x, lb, ub),
            method='trf', loss=_loss_p, f_scale=_fsc_p,
            bounds=(lb.tolist(), ub.tolist()),
            max_nfev=800, ftol=1e-8, xtol=1e-8, gtol=1e-8)
        rmse_f = float(np.sqrt(np.mean(residual_fn(lm_f.x)**2)))
        if rmse_f <= best_rmse:
            best_x, best_rmse = lm_f.x, rmse_f
            try:
                J  = lm_f.jac
                s2 = lm_f.cost / max(1, len(lm_f.fun)-6)
                cov_params = s2 * np.linalg.pinv(J.T @ J)
            except Exception:
                cov_params = None
        print(f"    Final LM RMSE: {rmse_f:.2f}\"")
    except Exception as e:
        print(f"    Final LM failed ({e})")

    # ── Held-out test RMSE ────────────────────────────────────────────────
    test_res_fn = _build_residual_fn(
        jd_test, ra_test, dec_test, epoch_jd, lat_deg, lon_deg, alt_m)
    opt_cost = float(np.sqrt(np.mean(test_res_fn(best_x)**2)))
    print(f"    Held-Out Test RMSE: {opt_cost:.2f}\"")

    ir,rr,ec,ar,mr,mm = best_x
    return ([math.degrees(ir)%180., math.degrees(rr)%360.,
             max(0.,min(0.9999,ec)),
             math.degrees(ar)%360., math.degrees(mr)%360.,
             mm/REV_PER_DAY_TO_RAD_PER_MIN],
            opt_cost, cov_params)


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 6 – MULTI-CATALOG IDENTIFICATION (MAHALANOBIS)
# ══════════════════════════════════════════════════════════════════════════════

def _angular_covariance_from_residuals(residuals_arcsec):
    res = np.asarray(residuals_arcsec, dtype=float)
    if len(res) < 4:
        f2 = MAHAL_FLOOR_ARCSEC**2
        return np.diag([f2, f2])
    ra_res   = res[0::2]
    dec_res  = res[1::2]
    s_ra     = max(MAHAL_FLOOR_ARCSEC, float(np.std(ra_res)))
    s_dec    = max(MAHAL_FLOOR_ARCSEC, float(np.std(dec_res)))
    corr     = float(np.clip(
        np.corrcoef(ra_res, dec_res)[0, 1] if len(ra_res) > 1 else 0.,
        -0.9, 0.9
    ))
    return np.array([
        [s_ra**2,             corr * s_ra * s_dec],
        [corr * s_ra * s_dec, s_dec**2            ]
    ])


def _download_celestrak_catalog(name: str, url: str) -> str:
    path = os.path.join(CATALOG_DIR, f"celestrak_{name}.tle")
    if not os.path.exists(path):
        print(f"  Downloading CelesTrak catalog: {name} …")
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            with open(path, "wb") as f:
                f.write(r.content)
        except Exception as e:
            print(f"    ⚠  Failed: {e}")
            return ""
    return path


def _login_spacetrack() -> requests.Session:
    global SPACETRACK_USER, SPACETRACK_PASS
    if not SPACETRACK_USER:
        SPACETRACK_USER = UserSecretsClient().get_secret("SPACETRACK_USER")
    if not SPACETRACK_PASS:
        SPACETRACK_PASS = UserSecretsClient().get_secret("SPACETRACK_PASS")
    if not SPACETRACK_USER or not SPACETRACK_PASS:
        print("  ⚠  Space-Track credentials not set — skipping historical catalog.")
        return None
    session   = requests.Session()
    login_url = f"{SPACETRACK_URL}/ajaxauth/login"
    payload   = {"identity": SPACETRACK_USER, "password": SPACETRACK_PASS}
    resp      = session.post(login_url, data=payload, timeout=30)
    if resp.status_code != 200 or '"Login"' in resp.text:
        print("  ⚠  Space-Track login failed — check credentials.")
        return None
    print("  ✅ Space-Track.org authenticated.")
    return session


def _download_spacetrack_catalog() -> str:
    from datetime import datetime, timedelta

    path = os.path.join(
        CATALOG_DIR,
        f"spacetrack_gp_history_{START_DATE}_to_{END_DATE}.tle"
    )
    if os.path.exists(path) and os.path.getsize(path) > 1_000_000:
        print(f"  ✅ Space-Track historical catalog already cached.")
        return path

    session = _login_spacetrack()
    if session is None:
        return ""

    print(f"  Downloading Space-Track gp_history in 7-day chunks …")
    start_dt   = datetime.strptime(START_DATE, "%Y-%m-%d")
    end_dt     = datetime.strptime(END_DATE, "%Y-%m-%d")
    chunks, cur = [], start_dt
    while cur <= end_dt:
        chunk_end = min(cur + timedelta(days=6), end_dt)
        chunks.append((cur.strftime("%Y-%m-%d"), chunk_end.strftime("%Y-%m-%d")))
        cur = chunk_end + timedelta(days=1)

    try:
        with open(path, "wb") as f:
            for idx, (c_start, c_end) in enumerate(chunks):
                print(f"    Chunk {idx+1}/{len(chunks)}: {c_start}–{c_end} …",
                      end=" ", flush=True)
                url  = (
                    f"{SPACETRACK_URL}/basicspacedata/query/class/gp_history"
                    f"/EPOCH/{c_start}--{c_end}/orderby/NORAD_CAT_ID/format/3le"
                )
                resp = session.get(url, timeout=600, stream=True)
                resp.raise_for_status()
                for chunk in resp.iter_content(chunk_size=65536):
                    if chunk:
                        f.write(chunk)
                print("✅")
                time.sleep(3.0)

        session.get(f"{SPACETRACK_URL}/ajaxauth/logout")
        size_mb = os.path.getsize(path) / 1_000_000
        if size_mb < 1.0:
            print("  ⚠  Space-Track download seems too small.")
            return ""
        print(f"  ✅ Space-Track catalog saved ({size_mb:.1f} MB)")
    except Exception as e:
        print(f"  ⚠  Space-Track download failed: {e}")
        return ""

    return path


def _load_discos_catalog(token: str) -> list:
    if not token:
        print("  ⚠  DISCOS_TOKEN not set — skipping ESA DISCOS catalog.")
        return []

    print("  Downloading ESA DISCOS catalog …")
    all_entries, page, page_size = [], 1, 100
    headers = {
        "Authorization":         f"Bearer {token}",
        "Accept":                "application/vnd.api+json",
        "DiscosWeb-Api-Version": "2",
    }

    try:
        while True:
            resp = requests.get(
                "https://discosweb.esoc.esa.int/api/objects",
                headers=headers,
                params={"page[number]": page, "page[size]": page_size},
                timeout=60,
            )
            resp.raise_for_status()
            items = resp.json().get("data", [])
            if not items:
                break
            for item in items:
                attrs = item.get("attributes", {})
                all_entries.append({
                    "name":     attrs.get("name", "UNKNOWN"),
                    "cosparId": attrs.get("cosparId", ""),
                    "type":     attrs.get("orbitType", ""),
                })
            if len(items) < page_size:
                break
            page += 1
            time.sleep(1.0)
        print(f"    ESA DISCOS: {len(all_entries)} objects loaded")
    except Exception as e:
        print(f"  ⚠  ESA DISCOS failed: {e}")

    return all_entries


def _load_gcat_catalog() -> list:
    print("  Downloading GCAT/SATCAT from CelesTrak …", end=" ", flush=True)
    try:
        resp = requests.get(GCAT_URL, timeout=60)
        resp.raise_for_status()
        lines   = resp.text.splitlines()
        reader  = csv.DictReader(lines)
        entries = [row for row in reader]
        print(f"✅  ({len(entries)} objects)")
        return entries
    except Exception as e:
        print(f"❌  {e}")
        return []


def load_all_catalogs() -> tuple:
    print("\n  ── Loading catalogs ──")
    all_tles = []

    for name, url in CELESTRAK_GROUPS:
        path = _download_celestrak_catalog(name, url)
        if not path:
            continue
        with open(path, "r") as f:
            raw = [l.strip() for l in f if l.strip()]
        count = 0
        for i in range(0, len(raw) - 2, 3):
            sat_name = raw[i]
            l1, l2   = raw[i+1], raw[i+2]
            if l1.startswith("1") and l2.startswith("2"):
                all_tles.append((sat_name, l1, l2, "CelesTrak"))
                count += 1
        print(f"    CelesTrak {name:20s}: {count} objects")

    st_path = _download_spacetrack_catalog()
    if st_path:
        with open(st_path, "r") as f:
            raw = [l.strip() for l in f if l.strip()]
        count = 0
        for i in range(0, len(raw) - 2, 3):
            sat_name = raw[i]
            l1, l2   = raw[i+1], raw[i+2]
            if l1.startswith("1") and l2.startswith("2"):
                all_tles.append((sat_name, l1, l2, "Space-Track-2022"))
                count += 1
        print(f"    Space-Track gp_history: {count} objects")

    seen, dedup = set(), []
    for entry in all_tles:
        key = entry[1] + entry[2]
        if key not in seen:
            seen.add(key)
            dedup.append(entry)
    print(f"  Total unique TLEs loaded: {len(dedup)}")

    gcat_rows    = _load_gcat_catalog()
    gcat_lookup  = {}
    for row in gcat_rows:
        # Try multiple possible column names for NORAD ID
        norad = (row.get("NORAD_CAT_ID")
              or row.get("CATNR")
              or row.get("norad_cat_id")
              or row.get("OBJECT_ID", "")).strip()
        if norad:
            gcat_lookup[norad] = row
    discos_rows  = _load_discos_catalog(DISCOS_TOKEN)
    discos_lookup = {row.get("cosparId", ""): row for row in discos_rows}

    return dedup, gcat_lookup, discos_lookup

def filter_catalog_by_orbit_class(catalog, orbit_class):
    """
    Pre-filter the TLE catalog to only objects plausibly in the detected
    orbit class, based on mean motion in TLE Line 2.
    Mean motion (rev/day):
      GEO : 0.9 – 1.1
      MEO : 1.1 – 6.4  (GPS/GLONASS/Galileo/MEO debris)
      LEO : 6.4 – 17.0
    """
    lo, hi = {
        "GEO": (0.90,  1.10),
        "MEO": (1.10,  6.40),
        "LEO": (6.40, 17.00),
    }.get(orbit_class, (0.50, 17.00))   # UNKNOWN → search all

    filtered = []
    for entry in catalog:
        sat_name, l1, l2, source = entry
        try:
            mm = float(l2[52:63])   # mean motion field in TLE line 2
            if lo <= mm <= hi:
                filtered.append(entry)
        except Exception:
            continue
    return filtered


def _orbital_consistency_check(l2_cand, fitted_incl_deg, fitted_ecc, fitted_mm_rev_day,
                                incl_tol_deg=8.0, ecc_tol=0.08, mm_tol_rel=0.08):
    """
    Checks whether a fitted orbit's shape (inclination, eccentricity, mean
    motion) is physically consistent with a candidate catalog TLE's own
    elements -- not just whether the sky position happens to be nearby.

    This catches cases like: angular separation to "BEIDOU-3 M21" is small
    in a crowded MEO band, but the fitted orbit has e=0.65 and incl=95°
    while the real BeiDou satellite has e<0.003 and incl~55° -- clearly
    not the same physical object, regardless of how close the sky position
    match looked.

    Returns (is_consistent, delta_incl_deg, delta_ecc, delta_mm_rel).
    """
    try:
        cand_incl = float(l2_cand[8:16])
        cand_ecc  = float("0." + l2_cand[26:33].strip())
        cand_mm   = float(l2_cand[52:63])
    except (ValueError, IndexError):
        return False, float('nan'), float('nan'), float('nan')

    d_incl = abs(fitted_incl_deg - cand_incl)
    d_incl = min(d_incl, 180.0 - d_incl)  # inclination wraps at 180
    d_ecc  = abs(fitted_ecc - cand_ecc)
    d_mm_rel = (abs(fitted_mm_rev_day - cand_mm) / cand_mm
                if cand_mm > 0 else float('inf'))

    is_consistent = (d_incl < incl_tol_deg and d_ecc < ecc_tol
                      and d_mm_rel < mm_tol_rel)
    return is_consistent, d_incl, d_ecc, d_mm_rel


def identify_satellite(obs_sf_time, ra_true, dec_true, ts, observer,
                        catalog, gcat_lookup=None, discos_lookup=None,
                        residuals_arcsec=None, fitted_elements=None):
    """
    fitted_elements: optional (incl_deg, ecc, mm_rev_day) from the current
    orbit fit. When provided, a catalog match must pass BOTH the angular
    (Mahalanobis) gate AND an orbital-shape consistency check against the
    candidate's own TLE elements -- prevents accepting a sky-position-only
    match whose actual orbit shape (eccentricity, inclination) rules it out.
    """
    if math.isnan(ra_true) or math.isnan(dec_true):
        return "UNKNOWN", float('inf'), "none", "", ""

    if residuals_arcsec is not None and len(residuals_arcsec) >= 4:
        if any(math.isnan(x) for x in residuals_arcsec):
            f2 = MAHAL_FLOOR_ARCSEC**2
            C_ang = np.diag([f2, f2])
        else:
            C_ang = _angular_covariance_from_residuals(residuals_arcsec)
    else:
        f2    = MAHAL_FLOOR_ARCSEC**2
        C_ang = np.diag([f2, f2])

    try:
        C_inv = np.linalg.inv(C_ang)
    except np.linalg.LinAlgError:
        C_inv = np.eye(2) / MAHAL_FLOOR_ARCSEC**2

    cos_dec     = math.cos(math.radians(dec_true))
    best_name   = "UNKNOWN"
    best_mahal  = float('inf')
    best_ang    = float('inf')
    best_source = "none"
    best_l1     = ""
    best_l2     = ""

    for sat_name, l1, l2, source_label in catalog:
        try:
            sat = EarthSatellite(l1, l2, sat_name, ts)
            p_ra, p_dec, _ = (sat - observer).at(obs_sf_time).radec()
            if math.isnan(p_ra.hours) or math.isnan(p_dec.degrees):
                continue
            dRA  = (p_ra.hours * 15. - ra_true) * cos_dec
            dDec =  p_dec.degrees - dec_true
            delta = np.array([dRA, dDec])
            d_mah = float(math.sqrt(max(0., float(delta @ C_inv @ delta))))
            if d_mah < best_mahal or (d_mah == best_mahal and "Space-Track" in source_label):
                best_mahal  = d_mah
                best_ang    = float(math.sqrt(dRA**2 + dDec**2))
                best_name   = sat_name
                best_source = source_label
                best_l1     = l1
                best_l2     = l2
        except Exception:
            continue

    if best_mahal > MAHAL_MATCH_SIGMA:
        best_name   = f"UNKNOWN (Near miss: {best_name} at {best_mahal:.1f}σ)"
        best_source = "none"
        best_l1     = ""
        best_l2     = ""
        best_ang    = float('inf')
    elif fitted_elements is not None and best_l2:
        _fi, _fe, _fm = fitted_elements
        _ok, _di, _de, _dm = _orbital_consistency_check(best_l2, _fi, _fe, _fm)
        if not _ok:
            best_name   = (f"UNKNOWN (Sky match {best_name} but orbit shape "
                            f"inconsistent: Δincl={_di:.1f}° Δecc={_de:.3f} "
                            f"Δmm={_dm*100:.1f}%)")
            best_source = "none"
            best_l1     = ""
            best_ang    = float('inf')

    gcat_row  = {}
    gcat_info = ""
    if gcat_lookup and best_l1:
        norad_id  = best_l1[2:7].strip()
        gcat_row  = gcat_lookup.get(norad_id, {})
        if gcat_row:
            # CelesTrak SATCAT CSV uses SATNAME or OBJECT_NAME
            sat_name = (gcat_row.get("SATNAME")
                     or gcat_row.get("OBJECT_NAME")
                     or gcat_row.get("satname")
                     or "UNKNOWN").strip()
            status   = (gcat_row.get("OPS_STATUS_CODE")
                     or gcat_row.get("OPERATIONAL_STATUS")
                     or gcat_row.get("ops_status_code")
                     or "?").strip()
            launch   = (gcat_row.get("LAUNCH_DATE")
                     or gcat_row.get("launch_date", "")).strip()
            country  = (gcat_row.get("COUNTRY")
                     or gcat_row.get("OWNER")
                     or gcat_row.get("country", "")).strip()
            gcat_info = (f"{sat_name} | Status: {status}"
                        + (f" | Country: {country}" if country else "")
                        + (f" | Launch: {launch}" if launch else ""))

    discos_info = ""
    if discos_lookup and gcat_row:
        cospar     = gcat_row.get("INTLDES", "").strip()
        discos_row = discos_lookup.get(cospar, {})
        if discos_row:
            discos_info = f"OrbitType: {discos_row.get('type','?')}"

    return best_name, best_ang, best_source, gcat_info, discos_info


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 6B – MULTI-PASS TRACK ASSOCIATION & MASTER TLE
# ══════════════════════════════════════════════════════════════════════════════

def stage6b_master_tle_solve(multi_pass_registry: dict,
                              ts,
                              observer,
                              report_lines: list,
                              gcat_lookup: dict  = None,
                              discos_lookup: dict = None) -> None:

    print("\n" + "="*60)
    print("STAGE 6B: MULTI-PASS TRACK ASSOCIATION & MASTER TLE SOLVE")
    print("="*60)

    multi_pass_sats = {
        name: passes
        for name, passes in multi_pass_registry.items()
        if len(passes) >= 2
    }

    if not multi_pass_sats:
        print("  ℹ  No satellite observed on 2+ separate passes.")
        report_lines.append(
            "\n" + "="*60 + "\n"
            "STAGE 6B: MULTI-PASS MASTER TLE SOLUTIONS\n"
            + "="*60 + "\n"
            + "No satellite observed on multiple passes — section skipped.\n"
        )
        return

    report_lines.append(
        "\n" + "="*60 + "\n"
        "STAGE 6B: MULTI-PASS MASTER TLE SOLUTIONS\n"
        + "="*60 + "\n"
        + f"{len(multi_pass_sats)} satellite(s) on 2+ passes.\n\n"
    )

    for sat_name, passes in multi_pass_sats.items():
        print(f"\n  ── Master TLE for: {sat_name} ──")
        ids    = [f"USER{p['track_id']+1:03d}" for p in passes]
        merged = sorted(
            [(t, v, rc)
             for p in passes
             for t, v, rc in zip(p["obs_times"], p["obs_vecs"], p["raw_coords"])],
            key=lambda triple: triple[0].tt
        )
        fused_times  = [m[0] for m in merged]
        fused_vecs   = [m[1] for m in merged]
        fused_coords = [m[2] for m in merged]
        total_s      = (fused_times[-1].tt - fused_times[0].tt) * 86400.0
        n_obs        = len(fused_times)

        from astropy.time import Time as _ATime
        t0a = _ATime(fused_times[0].tt, format="jd", scale="tt").utc
        doy = (t0a.jd - _ATime(f"{t0a.datetime.year}-01-01T00:00:00",
                                format="isot", scale="utc").jd) + 1.0
        epoch_str = f"{t0a.datetime.year % 100:02d}{doy:012.8f}"

        fused_orbit_class, _fused_ang_vel = classify_orbit_class([
            {"ra": rc[0], "dec": rc[1], "time_jd": t.tt}
            for t, rc in zip(fused_times, fused_coords)
        ])
        try:
            opt_params, opt_cost, _cov = optimise_tle(
                fused_times, fused_vecs, epoch_str, ts, observer,
                arc_duration_s=total_s, orbit_class=fused_orbit_class,
                ransac_thresh_arcsec=30.0   # multi-pass: 1° threshold
            )
        except Exception as exc:
            print(f"    ❌  optimise_tle failed: {exc}")
            report_lines.append(
                f"MASTER TLE — {sat_name}\n  ERROR : {exc}\n" + "-"*60 + "\n"
            )
            continue

        if math.isnan(opt_cost):
            print("    ⚠  NaN cost — insufficient inliers.  Skipping.")
            report_lines.append(
                f"MASTER TLE — {sat_name}\n  NOTE: Insufficient inliers.\n"
                + "-"*60 + "\n"
            )
            continue

        master_l1, master_l2 = generate_tle_lines(*opt_params, epoch_str)
        sat_opt   = EarthSatellite(master_l1, master_l2, "MASTER", ts)
        residuals = []
        for t, (ra_obs, dec_obs) in zip(fused_times, fused_coords):
            try:
                p_ra, p_dec, _ = (sat_opt - observer).at(t).radec()
                residuals.append(ang_sep_arcsec(
                    ra_obs, dec_obs, p_ra.hours * 15., p_dec.degrees
                ))
            except Exception:
                pass

        if residuals:
            rmse_m = math.sqrt(sum(r**2 for r in residuals) / len(residuals))
            max_r  = max(residuals)
            med_r  = sorted(residuals)[len(residuals) // 2]
        else:
            rmse_m = max_r = med_r = float("nan")

        incl, raan, ecc, argp, ma, mm = opt_params
        print(f"    ✅ Master TLE: incl={incl:.4f}°  ecc={ecc:.6f}  "
              f"mm={mm:.6f} rev/day  RMSE={rmse_m:.2f}\"")

        if rmse_m > MAX_ACCEPTABLE_DE_RMSE_ARCSEC:
            print(f"    ⚠  Master TLE RMSE {rmse_m:.0f}\" exceeds acceptable threshold — "
                  f"treating as unreliable, not exporting as final solution.")
            report_lines.append(
                f"MASTER TLE — {sat_name}\n"
                f"  NOTE: Fit RMSE {rmse_m:.0f}\" exceeds acceptable threshold "
                f"({MAX_ACCEPTABLE_DE_RMSE_ARCSEC:.0f}\") — solution rejected as unreliable.\n"
                + "-"*60 + "\n"
            )
            continue

        span_deg = ang_sep_arcsec(fused_coords[0][0], fused_coords[0][1],
                                  fused_coords[-1][0], fused_coords[-1][1]) / 3600.0

        avg_vel  = (span_deg * 3600.0) / total_s if total_s > 0 else 0.0

        # --- VITO'S DARK GAP BENCHMARK ---
        t_gap_12h = t0a + 0.5  # Propagate +12 hours
        t_gap_24h = t0a + 1.0  # Propagate +24 hours

    
        if _cov is not None:
            trace_cov = float(np.trace(_cov))
            # Surrogate check against EASA 1x10^-5 safety standard limits
            easa_status = "PASS (< 1e-5 Threshold)" if trace_cov < 1e-3 else "FAIL (Uncertainty Diverges)"
        else:
            trace_cov = float("nan")
            easa_status = "UNKNOWN (Covariance Unavailable)"

        report_lines.append(
            f"MASTER TLE SOLUTION\n"
            f"SATELLITE    : {sat_name}\n"
            f"PASSES FUSED : {', '.join(ids)} ({len(passes)} passes, {n_obs} obs)\n"
            f"TOTAL ARC    : {total_s/3600:.4f} h  ({total_s:.1f} s)\n"
            f"KINEMATICS   : Span={span_deg:.2f}° | Velocity={avg_vel:.1f} arcsec/s\n"
            f"EPOCH        : {epoch_str}\n\n"
            f"ORBITAL ELEMENTS\n"
            f"  Inclination   : {incl:10.4f} deg\n"
            f"  RAAN          : {raan:10.4f} deg\n"
            f"  Eccentricity  : {ecc:.7f}\n"
            f"  Arg of Perigee: {argp:10.4f} deg\n"
            f"  Mean Anomaly  : {ma:10.4f} deg\n"
            f"  Mean Motion   : {mm:.8f} rev/day\n\n"
            f"OPERATIONAL PROPAGATION (DARK GAP BENCHMARK)\n"
            f"  +12 Hour Prop. JD : {t_gap_12h.jd:.5f}\n"
            f"  +24 Hour Prop. JD : {t_gap_24h.jd:.5f}\n"
            f"  Covariance Trace  : {trace_cov:.2e}\n"
            f"  EASA 1x10^-5 Check: {easa_status}\n\n"
            f"RESIDUALS (HELD-OUT TEST SET)\n"
            f"  RMSE  : {rmse_m:.2f} arcsec\n"
            f"  Max   : {max_r:.2f} arcsec\n"
            f"  Median: {med_r:.2f} arcsec\n"
            f"  N used: {len(residuals)}/{n_obs}\n\n"
            f"MASTER TLE:\n{master_l1}\n{master_l2}\n"
            + "-"*60 + "\n"
        )

    print(f"\n✅ Stage 6B complete — {len(multi_pass_sats)} master TLE(s)")


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN PIPELINE ORCHESTRATOR
# ══════════════════════════════════════════════════════════════════════════════



def classify_orbit_class(obs_list):
    """
    Classify a track as LEO / MEO / GEO from its observed angular velocity.
    Uses the first and last observations in the track.
    """
    if len(obs_list) < 2:
        return "UNKNOWN", 0.0

    obs_sorted = sorted(obs_list, key=lambda d: d.get("time_jd") or 0)
    first, last = obs_sorted[0], obs_sorted[-1]

    dt_s = (last["time_jd"] - first["time_jd"]) * 86400.0
    if dt_s < 1.0:
        return "UNKNOWN", 0.0

    ra1, dec1 = first["ra"], first["dec"]
    ra2, dec2 = last["ra"],  last["dec"]

    # Great-circle separation
    import math
    d  = math.radians
    sep_rad = 2.0 * math.asin(math.sqrt(
        math.sin((d(dec2) - d(dec1)) / 2)**2 +
        math.cos(d(dec1)) * math.cos(d(dec2)) *
        math.sin((d(ra2) - d(ra1)) / 2)**2
    ))
    ang_vel = math.degrees(sep_rad) * 3600.0 / dt_s   # arcsec/s

    # Classification thresholds from orbital mechanics — verified against
    # actual populated-orbit altitudes, not just the technical LEO/MEO/GEO
    # altitude band definitions:
    #   GEO         : <30 arcsec/s   (35,786 km)
    #   MEO (real)  : 30-50 arcsec/s (GPS 20,200km/Galileo 23,222km/
    #                 GLONASS 19,100km/BeiDou 21,500km all top out at
    #                 ~33-43 arcsec/s at zenith -- verified by direct
    #                 orbital-mechanics calculation, cross-checked against
    #                 each constellation's published altitude)
    #   LEO         : >50 arcsec/s   (200-2,000 km; horizon-to-zenith
    #                 range for a 550km pass alone spans ~360-2880 arcsec/s
    #                 per Kassas et al. 2021, MNRAS 509(2):1848 -- entirely
    #                 above the MEO ceiling)
    #
    # The old 30-400 arcsec/s "MEO" bucket was misclassifying real LEO
    # objects as MEO and locking optimise_tle's mean-motion search to the
    # MEO band (1.10-6.40 rev/day) for objects that actually needed the
    # LEO band (6.40-17.0 rev/day) -- physically impossible for the
    # optimizer to converge correctly. Confirmed directly from this run's
    # own log: every track in the misclassified 30-133 arcsec/s band that
    # reached OD produced garbage fits (median held-out RMSE 163", up to
    # 320,899") and mass "orbit shape inconsistent" catalog rejections
    # (905 of ~1554 tracks) -- the mean-motion mismatch this caused.
    #
    # NOTE: this deliberately misclassifies the rare low-altitude MEO
    # exception (SES O3b, ~5,400 km, ~20 satellites total) as LEO -- an
    # intentional tradeoff, since O3b is a tiny fleet next to how much
    # real LEO data the old threshold was destroying.
    if ang_vel < 30.0:
        orbit_class = "GEO"
    elif ang_vel < 50.0:
        orbit_class = "MEO"
    elif ang_vel < 2000.0:
        orbit_class = "LEO"
    else:
        orbit_class = "LEO"   # hypersonic streak — still attempt LEO IOD

    return orbit_class, ang_vel

def cross_field_link_tracks(all_detections, existing_tracks, max_dt_s=7200.0):
    """
    Links detections from DIFFERENT ZTF fields into extended multi-field tracks.
    Turns 3-observation same-field tracks into 8-15 observation multi-field tracks.
    """
    print(f"\n  Cross-field linking: {len(existing_tracks)} seed tracks × "
          f"{len(all_detections)} detections …")

    tracked_keys = set()
    for t in existing_tracks:
        for obs in t["obs"]:
            tracked_keys.add((obs["filename"], round(obs.get("time_jd", 0), 6)))

    free_dets = [
        d for d in all_detections
        if d.get("ra") is not None and d.get("time_jd") is not None
        and (d["filename"], round(d["time_jd"], 6)) not in tracked_keys
    ]
    free_dets.sort(key=lambda d: d["time_jd"])

    extended = []

    for track in existing_tracks:
        obs_sorted = sorted(track["obs"], key=lambda d: d.get("time_jd") or 0)
        if len(obs_sorted) < 2:
            extended.append(track)
            continue

        times = np.array([d["time_jd"] for d in obs_sorted], dtype=float)
        ras   = np.array([d["ra"]  for d in obs_sorted], dtype=float)
        decs  = np.array([d["dec"] for d in obs_sorted], dtype=float)
        t0      = times[0]
        dt      = times - t0
        cos_dec = math.cos(math.radians(np.mean(decs)))

        if dt[-1] < 1e-8:
            extended.append(track)
            continue

        ra_rate  = float(np.polyfit(dt, ras,  1)[0])
        dec_rate = float(np.polyfit(dt, decs, 1)[0])
        ang_vel_arcsec_s = math.sqrt(
            (ra_rate * cos_dec)**2 + dec_rate**2
        ) * 3600.0 / 86400.0

        if len(obs_sorted) >= 3:
            ra_resid  = ras  - np.polyval(np.polyfit(dt, ras,  1), dt)
            dec_resid = decs - np.polyval(np.polyfit(dt, decs, 1), dt)
            scatter   = math.sqrt(float(np.mean(ra_resid**2 + dec_resid**2)))
            scatter   = max(scatter, 0.005)
        else:
            scatter = 0.02

        search_radius_deg = max(0.5 * scatter,
                                ang_vel_arcsec_s * 10.0 / 3600.0)
        search_radius_deg = min(search_radius_deg, 0.5)  # hard cap 0.5°

        new_obs = list(obs_sorted)

        for det in free_dets:
            dt_s = (det["time_jd"] - t0) * 86400.0
            if abs(dt_s) > max_dt_s:
                continue
            dt_day   = det["time_jd"] - t0
            ra_pred  = ras[0]  + ra_rate  * dt_day
            dec_pred = decs[0] + dec_rate * dt_day
            dra  = ((det["ra"]  - ra_pred + 180) % 360 - 180) * cos_dec
            ddec = det["dec"] - dec_pred
            sep  = math.sqrt(dra**2 + ddec**2)
            if sep < search_radius_deg:
                new_obs.append(det)
                tracked_keys.add((det["filename"], round(det["time_jd"], 6)))

        if len(new_obs) > len(obs_sorted):
            print(f"    Extended track: {len(obs_sorted)} → {len(new_obs)} obs "
                  f"(ang_vel={ang_vel_arcsec_s:.1f}\"/s)")
            track = dict(track)
            track["obs"] = sorted(new_obs, key=lambda d: d.get("time_jd") or 0)
            track["cross_field"] = True

        extended.append(track)

    n_extended = sum(1 for t in extended if t.get("cross_field"))
    print(f"  Cross-field linking: {n_extended}/{len(extended)} tracks extended")
    return extended

def _split_obs_by_night(obs_list, gap_hours=6.0):
    """
    Splits a fused group's observations into 'nights' by detecting time
    gaps larger than gap_hours. Returns a list of obs sub-lists, ordered
    chronologically, one per night/session.
    """
    obs_sorted = sorted(obs_list, key=lambda d: d.get("time_jd") or 0)
    if not obs_sorted:
        return []
    nights = [[obs_sorted[0]]]
    for prev, cur in zip(obs_sorted, obs_sorted[1:]):
        gap_h = (cur["time_jd"] - prev["time_jd"]) * 24.0
        if gap_h > gap_hours:
            nights.append([])
        nights[-1].append(cur)
    return nights


def correlate_and_fuse_tracklets_across_nights(
        good_tracks, ts, observer,
        min_gap_s=3600.0, max_gap_s=5 * 86400.0,
        screen_thresh_deg=1.5, verify_rmse_thresh_arcsec=200.0,
        max_group_size=6):
    """
    Cross-night tracklet-to-tracklet correlation with INCREMENTAL group
    growth and mutual-consistency verification.

    Earlier version verified each candidate B only against the anchor A,
    which let a chain of individually-plausible-but-mutually-inconsistent
    tracklets accumulate into one group (confirmed by real-run evidence:
    an 11-tracklet group that RANSAC later had to reject 64/70 merged
    points from, and that failed held-out cross-night validation by
    ~90,000 km despite passing the observability check on its own
    contaminated training data).

    Fix: every candidate is now verified against the CURRENT accumulated
    group (all previously accepted members), not just the original anchor.
    This means later candidates are tested against the joint evidence of
    everything accepted so far, which is what actually prevents mutually
    inconsistent tracklets from being merged.
    """
    lat_deg = float(observer.latitude.degrees)
    lon_deg = float(observer.longitude.degrees)
    alt_m   = float(observer.elevation.m)

    def _build_optimise_inputs(obs_list):
        obs_times, obs_vecs, raw_coords = [], [], []
        for d in sorted(obs_list, key=lambda x: x["time_jd"]):
            t_astropy = Time(d["time_utc"], format="isot", scale="utc")
            obs_times.append(ts.from_astropy(t_astropy))
            obs_vecs.append(radec_to_vec(d["ra"], d["dec"]))
            raw_coords.append((d["ra"], d["dec"]))
        t0 = Time(obs_list[0]["time_utc"], format="isot", scale="utc")
        doy = (t0.jd - Time(f"{t0.datetime.year}-01-01T00:00:00",
                             format="isot", scale="utc").jd) + 1.0
        epoch_str = f"{t0.datetime.year % 100:02d}{doy:012.8f}"
        return obs_times, obs_vecs, raw_coords, epoch_str

    def _screen_against_satrec(sat, jd_arr, ra_arr, dec_arr):
        seps = []
        for jd_o, ra_o, dec_o in zip(jd_arr, ra_arr, dec_arr):
            ji, jf = math.floor(jd_o), jd_o - math.floor(jd_o)
            ec, rt, _ = sat.sgp4(ji, jf)
            if ec != 0:
                seps.append(1e6)
                continue
            Ro  = _observatory_eci(jd_o, lat_deg, lon_deg, alt_m)
            top = np.array(rt) - Ro
            tm  = float(np.linalg.norm(top))
            if tm < 1:
                seps.append(1e6)
                continue
            th  = top / tm
            rp  = math.degrees(math.atan2(th[1], th[0])) % 360.
            dp  = math.degrees(math.asin(max(-1., min(1., th[2]))))
            cd  = math.cos(math.radians(dec_o))
            dr  = ((rp - ra_o + 180) % 360 - 180) * cd
            dd  = (dp - dec_o)
            seps.append(math.sqrt(dr**2 + dd**2))
        return seps

    def _satrec_from_params(params, epoch_jd):
        try:
            return Satrec.twoline2rv(*generate_tle_lines(*params,
                _time_to_epoch_str(epoch_jd)))
        except Exception:
            return None

    def _time_to_epoch_str(epoch_jd):
        t = Time(epoch_jd, format='jd', scale='utc')
        doy = (t.jd - Time(f"{t.datetime.year}-01-01T00:00:00",
                            format="isot", scale="utc").jd) + 1.0
        return f"{t.datetime.year % 100:02d}{doy:012.8f}"

    # ── Build cleaned per-tracklet summaries (unchanged from before) ──
    summaries = []
    for tid, t in enumerate(good_tracks):
        obs = sorted(t["obs"], key=lambda d: d.get("time_jd") or 0)
        if len(obs) < 2:
            continue
        oc, _ = classify_orbit_class(obs)
        if oc == "UNKNOWN":
            continue

        jd_raw  = np.array([d["time_jd"] for d in obs])
        ra_raw  = np.array([d["ra"] for d in obs])
        dec_raw = np.array([d["dec"] for d in obs])

        if len(obs) >= 4:
            _mask, ra_c, dec_c, jd_c = ransac_reject_outliers(
                jd_raw, ra_raw, dec_raw,
                inlier_thresh_arcsec=120.0, n_iterations=300, min_inliers=2)
            n_rej = int(np.sum(~_mask))
            if n_rej > 0:
                print(f"    Track {tid}: RANSAC-cleaned {n_rej}/{len(obs)} "
                      f"outlier obs before correlation screening")
        else:
            jd_c, ra_c, dec_c = jd_raw, ra_raw, dec_raw

        if len(jd_c) < 2:
            continue

        epoch_jd_A = float(jd_c[0])
        cand = _admissible_region_iod(
            jd_c, ra_c, dec_c, lat_deg, lon_deg, alt_m,
            oc, epoch_jd_A, n_rho=25, n_rhodot=12)

        summaries.append({
            "track_id": tid, "orbit_class": oc,
            "jd": jd_c, "ra": ra_c, "dec": dec_c,
            "mid_jd": float(np.median(jd_c)),
            "obs_clean": [{"time_jd": j, "ra": r, "dec": d,
                           "time_utc": Time(j, format='jd', scale='utc').isot}
                          for j, r, d in zip(jd_c, ra_c, dec_c)],
            "cand": cand, "epoch_jd": epoch_jd_A,
        })

    print(f"\n  Cross-night correlation: {len(summaries)} candidate tracklets "
          f"({sum(1 for s in summaries if s['cand'] is not None)} with a "
          f"usable admissible-region candidate)")

    # ── Deduplicate near-identical tracklets (unchanged from before) ──
    seen_signatures = set()
    deduped = []
    for s in summaries:
        sig = tuple(round(j, 4) for j in s["jd"])
        if sig in seen_signatures:
            continue
        seen_signatures.add(sig)
        deduped.append(s)
    print(f"  Deduplication: {len(summaries)} -> {len(deduped)} tracklets "
          f"({len(summaries) - len(deduped)} duplicate/overlapping removed)")
    summaries = deduped

    fused_groups = []
    used = set()
    _diag = {"no_cand": 0, "satrec_construct_fail": 0, "orbit_class_mismatch": 0,
              "gap_out_of_range": 0, "reached_screen": 0}

    for i in range(len(summaries)):
        if i in used:
            continue
        A = summaries[i]
        if A["cand"] is None:
            _diag["no_cand"] += 1
            continue

        try:
            tr = Satrec()
            tr.sgp4init(WGS84, 'i', 99999, A["epoch_jd"] - 2433281.5,
                1e-5, 0., 0.,
                float(A["cand"][2]),
                float(A["cand"][3]) % (2 * math.pi),
                max(0., min(math.pi, float(A["cand"][0]))),
                float(A["cand"][4]) % (2 * math.pi),
                max(1e-4, float(A["cand"][5])),
                float(A["cand"][1]) % (2 * math.pi))
        except Exception as _e:
            _diag["satrec_construct_fail"] += 1
            if _diag["satrec_construct_fail"] <= 3:
                print(f"    [debug] Satrec construction failed for track "
                      f"{A['track_id']}: {_e}  cand={A['cand']}")
            continue

        group_members = [A]
        group_used = {i}
        group_obs = list(A["obs_clean"])
        current_screen_sat = tr   # screening satellite, refreshed on each accept

        # Sort remaining candidates by time separation from A -- grow the
        # group outward in time, closest first.
        candidates_j = sorted(
            [j for j in range(len(summaries)) if j not in group_used],
            key=lambda j: abs(summaries[j]["mid_jd"] - A["mid_jd"])
        )

        for j in candidates_j:
            if j in used or j in group_used:
                continue
            if len(group_members) >= max_group_size:
                break
            B = summaries[j]
            if B["orbit_class"] != A["orbit_class"]:
                _diag["orbit_class_mismatch"] += 1
                continue
            dt_s = abs(B["mid_jd"] - A["mid_jd"]) * 86400.0
            if not (min_gap_s <= dt_s <= max_gap_s):
                _diag["gap_out_of_range"] += 1
                continue
            _diag["reached_screen"] += 1
            # Stage 1: loose screen against the CURRENT group's best-fit
            # satellite (not just A's original single-tracklet guess).
            seps = _screen_against_satrec(current_screen_sat,
                                           B["jd"], B["ra"], B["dec"])
            if not seps:
                continue
            min_sep_deg = min(seps)
            if min_sep_deg > screen_thresh_deg:
                if min_sep_deg < screen_thresh_deg * 3:
                    print(f"    SCREEN-NEAR-MISS: group[{A['track_id']}] <-> "
                          f"track {B['track_id']}  dt={dt_s/3600:.1f}h  "
                          f"screen_sep={min_sep_deg*3600:.0f}\" "
                          f"(threshold={screen_thresh_deg}°)")
                continue

            # Stage 2: verify B against the FULL accumulated group, not
            # just A. This is what actually prevents mutually inconsistent
            # tracklets from chaining onto a shared anchor.
            trial_obs = group_obs + B["obs_clean"]
            obs_times, obs_vecs, raw_coords, epoch_str = \
                _build_optimise_inputs(trial_obs)
            arc_s = (obs_times[-1].tt - obs_times[0].tt) * 86400.0

            v_params, v_rmse, v_cov = optimise_tle(
                obs_times, obs_vecs, epoch_str, ts, observer,
                arc_duration_s=arc_s, orbit_class=A["orbit_class"])

            if math.isnan(v_rmse) or v_rmse > verify_rmse_thresh_arcsec:
                if not math.isnan(v_rmse) and v_rmse < verify_rmse_thresh_arcsec * 3:
                    print(f"    RMSE-NEAR-MISS: group[{A['track_id']}] <-> "
                          f"track {B['track_id']}  dt={dt_s/3600:.1f}h  "
                          f"joint_rmse={v_rmse:.1f}\" "
                          f"(threshold={verify_rmse_thresh_arcsec:.0f}\")")
                continue
            v_mm_rad_min = v_params[5] * REV_PER_DAY_TO_RAD_PER_MIN
            sigma_mm, rel_sigma_mm, well_observed = _fit_observability_metric(
                v_cov, v_mm_rad_min)
            if not well_observed:
                print(f"    NEAR-MISS: group[{A['track_id']}] <-> "
                      f"track {B['track_id']}  dt={dt_s/3600:.1f}h  "
                      f"joint_rmse={v_rmse:.1f}\"  sigma_mm={sigma_mm:.5f}  "
                      f"rel={rel_sigma_mm*100:.2f}%  "
                      f"REJECTED (range unconstrained despite good RMSE)")
                continue

            # Accepted: fold B into the group, and refresh the screening
            # satellite so subsequent candidates are tested against the
            # UPDATED joint fit, not the stale single-tracklet guess.
            print(f"    LINK: group[{A['track_id']}] <-> track {B['track_id']}  "
                  f"dt={dt_s/3600:.1f}h  group_size={len(group_members)+1}  "
                  f"joint_rmse={v_rmse:.1f}\"  sigma_mm={sigma_mm:.5f}  "
                  f"class={A['orbit_class']}")
            group_members.append(B)
            group_used.add(j)
            group_obs = trial_obs

            new_sat = _satrec_from_params(v_params, float(obs_times[0].tt))
            if new_sat is not None:
                current_screen_sat = new_sat

        if len(group_members) > 1:
            used.update(group_used)
            # Use group_obs -- the already RANSAC-cleaned, verified points
            # that actually earned this group's LINK acceptances -- rather
            # than re-pulling each member's raw, uncleaned observation set.
            # Raw tracks are frequently 90%+ noise (e.g. rejected 136/143,
            # 178/184 seen in this run's individual-track cleaning), and
            # merging raw multi-track data let Stage 4B's own RANSAC
            # silently discard an entire genuinely-linked night as
            # "outliers" in favor of whichever night contributed the
            # noisiest, most numerous raw points -- producing a fit that
            # was secretly single-night despite looking multi-night, which
            # is why cross-night validation failed catastrophically even
            # when the observability check reported WELL-OBSERVED.
            fused_obs = sorted(group_obs, key=lambda d: d.get("time_jd") or 0)
            fused_groups.append({
                "orbit_class": A["orbit_class"],
                "member_track_ids": [m["track_id"] for m in group_members],
                "obs": fused_obs,
            })

    print(f"  Cross-night correlation: {len(fused_groups)} fused multi-night "
          f"group(s) from {len(summaries)} tracklets")
    print(f"  Diagnostic breakdown: {_diag}")
    return fused_groups



def solve_fused_groups(fused_groups, ts, observer, catalog,
                        gcat_lookup, discos_lookup, report_lines):
    """
    Fits an orbit to each cross-night fused group using the FULL combined
    multi-night baseline for the fit itself (this is where the actual
    accuracy gain from tracklet fusion comes from -- a real multi-day
    baseline resolves range/mean-motion the way no single night can).

    Held-out validation reuses optimise_tle's own internal RANSAC-based
    train/test split, which already partitions ALL fused points (not by
    night) and reports a genuine out-of-sample RMSE computed from a fit
    that still had access to the real baseline. This replaces the earlier
    whole-night-holdout design, which trained on a single isolated short
    arc and then tested a 3-day extrapolation against it -- reproducing
    the exact short-arc mean-motion degeneracy this pipeline exists to
    avoid, just relocated into the validation step instead of Stage 4A.

    When a group spans 3+ nights, an ADDITIONAL true held-out-night check
    is performed as a bonus rigor pass, since in that case enough baseline
    remains in the training set even after removing one full night.
    """
    print("\n" + "="*60)
    print("STAGE 4B: CROSS-NIGHT FUSED ORBIT SOLUTIONS")
    print("="*60)

    lat_deg = float(observer.latitude.degrees)
    lon_deg = float(observer.longitude.degrees)
    alt_m   = float(observer.elevation.m)

    results = []

    for gi, group in enumerate(fused_groups):
        orbit_class = group["orbit_class"]
        member_ids  = group["member_track_ids"]
        all_obs     = sorted(group["obs"], key=lambda d: d.get("time_jd") or 0)
        nights      = _split_obs_by_night(all_obs, gap_hours=6.0)

        print(f"\n  ── Fused group {gi+1}/{len(fused_groups)}  "
              f"(tracks {member_ids}, class={orbit_class}, "
              f"{len(nights)} nights, {len(all_obs)} obs total) ──")

        if len(nights) < 2:
            print("    ⚠  Fused group collapsed to 1 night -- skipping.")
            continue
        if len(all_obs) < 4:
            print("    ⚠  Fewer than 4 total observations -- skipping "
                  "(insufficient for a meaningful multi-night fit).")
            continue

        obs_times, obs_vecs, raw_coords = [], [], []
        for row in all_obs:
            t_astropy = Time(row["time_utc"], format="isot", scale="utc")
            obs_times.append(ts.from_astropy(t_astropy))
            obs_vecs.append(radec_to_vec(row["ra"], row["dec"]))
            raw_coords.append((row["ra"], row["dec"]))

        t0 = Time(all_obs[0]["time_utc"], format="isot", scale="utc")
        doy = (t0.jd - Time(f"{t0.datetime.year}-01-01T00:00:00",
                             format="isot", scale="utc").jd) + 1.0
        epoch_str    = f"{t0.datetime.year % 100:02d}{doy:012.8f}"
        full_span_s  = (obs_times[-1].tt - obs_times[0].tt) * 86400.0

        cat_l1, cat_l2, pre_sep = _pre_search_catalog(
            obs_times, raw_coords, ts, observer, catalog, orbit_class)

        print(f"    Full multi-night baseline: {len(all_obs)} obs over "
              f"{full_span_s/86400.:.2f} days ({len(nights)} nights)")

        opt_params, opt_cost, cov_params = optimise_tle(
            obs_times, obs_vecs, epoch_str, ts, observer,
            arc_duration_s=full_span_s, orbit_class=orbit_class,
            catalog_l1=cat_l1, catalog_l2=cat_l2,
        )

        if math.isnan(opt_cost):
            print("    ❌ Fit failed -- skipping this group.")
            continue

        l1, l2 = generate_tle_lines(*opt_params, epoch_str)

        mm_rad_min = opt_params[5] * REV_PER_DAY_TO_RAD_PER_MIN
        sigma_mm, rel_sigma_mm, well_observed = _fit_observability_metric(
            cov_params, mm_rad_min)

        # Physical-unit error estimate at the fit epoch (circular approx)
        n_rad_s = mm_rad_min / 60.0
        a_km = (MU_EARTH_KM3_S2 / (n_rad_s ** 2)) ** (1.0 / 3.0) if n_rad_s > 0 else float('nan')

        print(f"    Held-Out Test RMSE (within full baseline): {opt_cost:.2f}\"")
        print(f"    Observability: σ_mm={sigma_mm:.5f} rev/day "
              f"({rel_sigma_mm*100:.2f}% relative)  "
              f"{'WELL-OBSERVED' if well_observed else 'RANGE UNCONSTRAINED'}")

        # ── Bonus check: true independent-night holdout, only when a
        # spare night is actually affordable (3+ nights) ──
        extra_night_rmse = float('nan')
        if len(nights) >= 3:
            test_night = nights[-1]
            train_obs  = [d for n in nights[:-1] for d in n]
            if len(train_obs) >= 4:
                tr_times, tr_vecs, tr_coords = [], [], []
                for row in train_obs:
                    t_astropy = Time(row["time_utc"], format="isot", scale="utc")
                    tr_times.append(ts.from_astropy(t_astropy))
                    tr_vecs.append(radec_to_vec(row["ra"], row["dec"]))
                    tr_coords.append((row["ra"], row["dec"]))
                t0b = Time(train_obs[0]["time_utc"], format="isot", scale="utc")
                doyb = (t0b.jd - Time(f"{t0b.datetime.year}-01-01T00:00:00",
                                      format="isot", scale="utc").jd) + 1.0
                epoch_str_b = f"{t0b.datetime.year % 100:02d}{doyb:012.8f}"
                arc_b = (tr_times[-1].tt - tr_times[0].tt) * 86400.0

                _, _, cov_b = optimise_tle(
                    tr_times, tr_vecs, epoch_str_b, ts, observer,
                    arc_duration_s=arc_b, orbit_class=orbit_class)
                try:
                    sat_test = Satrec.twoline2rv(l1, l2)
                    resid = []
                    for d in test_night:
                        rp, dp = _sgp4_topocentric_radec(
                            sat_test, d["time_jd"], lat_deg, lon_deg, alt_m)
                        if rp is not None:
                            resid.append(ang_sep_arcsec(d["ra"], d["dec"], rp, dp))
                    if resid:
                        extra_night_rmse = math.sqrt(sum(r**2 for r in resid) / len(resid))
                        print(f"    Bonus check -- true held-out night "
                              f"({len(test_night)} obs): RMSE={extra_night_rmse:.2f}\"")
                except Exception:
                    pass

        # ── Confidence gate: based on the genuine multi-night fit quality,
        # not a crippled single-night-vs-extrapolation test ──
        if not well_observed:
            confidence = "LOW (Range Unconstrained)"
        elif opt_cost < 30.0:
            confidence = "HIGH (Multi-Night Fused)"
        elif opt_cost < 200.0:
            confidence = "MEDIUM (Multi-Night Fused)"
        else:
            confidence = "LOW (Fused RMSE Too High)"

        if not math.isnan(extra_night_rmse) and extra_night_rmse > 500.0:
            confidence = "LOW (Failed Independent-Night Check)"

        result = {
            "group_index": gi, "member_track_ids": member_ids,
            "orbit_class": orbit_class, "n_nights": len(nights),
            "n_obs": len(all_obs), "full_span_days": full_span_s / 86400.0,
            "l1": l1, "l2": l2, "epoch_str": epoch_str,
            "fused_rmse": opt_cost, "sigma_mm": sigma_mm,
            "rel_sigma_mm": rel_sigma_mm, "well_observed": well_observed,
            "semi_major_axis_km": a_km, "extra_night_rmse": extra_night_rmse,
            "confidence": confidence,
        }
        results.append(result)

        mid_idx = len(obs_times) // 2
        cat_celestrak  = filter_catalog_by_orbit_class(
            [c for c in catalog if "CelesTrak" in c[3]], orbit_class)
        cat_spacetrack = filter_catalog_by_orbit_class(
            [c for c in catalog if "Space-Track" in c[3]], orbit_class)
        _fitted_elems = (opt_params[0], opt_params[2], opt_params[5])
        ct_match, ct_dist, _, ct_gcat, _ = identify_satellite(
            obs_times[mid_idx], raw_coords[mid_idx][0], raw_coords[mid_idx][1],
            ts, observer, cat_celestrak, gcat_lookup, discos_lookup,
            fitted_elements=_fitted_elems)
        st_match, st_dist, _, st_gcat, _ = identify_satellite(
            obs_times[mid_idx], raw_coords[mid_idx][0], raw_coords[mid_idx][1],
            ts, observer, cat_spacetrack, gcat_lookup, discos_lookup,
            fitted_elements=_fitted_elems)

        incl, raan, ecc, argp, ma, mm = opt_params
        report_lines.append(
            f"FUSED CROSS-NIGHT ORBIT SOLUTION\n"
            f"GROUP        : {gi+1} (tracklets {member_ids})\n"
            f"ORBIT CLASS  : {orbit_class}\n"
            f"NIGHTS FUSED : {len(nights)}  ({len(all_obs)} obs, "
            f"full baseline {full_span_s/86400.:.2f} days)\n"
            f"EPOCH        : {epoch_str}\n\n"
            f"ORBITAL ELEMENTS\n"
            f"  Semi-major axis : {a_km:10.2f} km\n"
            f"  Inclination     : {incl:10.4f} deg\n"
            f"  RAAN            : {raan:10.4f} deg\n"
            f"  Eccentricity    : {ecc:.7f}\n"
            f"  Arg of Perigee  : {argp:10.4f} deg\n"
            f"  Mean Anomaly    : {ma:10.4f} deg\n"
            f"  Mean Motion     : {mm:.8f} rev/day\n\n"
            f"VALIDATION\n"
            f"  Held-out RMSE (full-baseline fit) : {opt_cost:.2f}\"\n"
            f"  σ(mean motion)                    : {sigma_mm:.5f} rev/day "
            f"({rel_sigma_mm*100:.2f}% relative)\n"
            f"  Well-observed                     : {well_observed}\n"
            f"  Independent-night check           : "
            f"{'N/A (only 2 nights)' if math.isnan(extra_night_rmse) else f'{extra_night_rmse:.2f}\"'}\n\n"
            f"IDENTIFICATION\n"
            f"  CELESTRAK    : {ct_match} ({ct_dist:.4f}°)\n"
            f"  SPACE-TRACK  : {st_match} ({st_dist:.4f}°)\n"
            f"  GCAT INFO    : {ct_gcat or st_gcat or 'N/A'}\n\n"
            f"CONFIDENCE   : {confidence}\n"
            f"TLE:\n{l1}\n{l2}\n"
            + "-"*60 + "\n"
        )

        print(f"    ✅ Confidence: {confidence}")

    print(f"\n✅ Stage 4B complete — {len(results)} fused orbit(s) solved, "
          f"{sum(1 for r in results if 'HIGH' in r['confidence'])} HIGH confidence")
    return results


def load_seed_tles(seed_path):
    """
    Load TLEs saved by a previous night's run to use as warm-start
    seeds for the current night. This extends arc duration across
    nights, dramatically improving orbit accuracy (10-100× RMSE reduction).

    Expected file format (written by _multi_night_extend):
        # RMSE=248.7" arc=424s
        1 99999U ...
        2 99999 ...
    """
    if not seed_path or not os.path.exists(seed_path):
        return []
    seeds = []
    try:
        with open(seed_path) as f:
            lines = [l.strip() for l in f if l.strip()]
        i = 0
        while i < len(lines):
            if lines[i].startswith("#"):
                i += 1
                continue
            if i + 1 < len(lines) and lines[i].startswith("1") and lines[i+1].startswith("2"):
                seeds.append((lines[i], lines[i+1]))
                i += 2
            else:
                i += 1
        print(f"  ✅ Loaded {len(seeds)} seed TLEs from previous night: {seed_path}")
    except Exception as e:
        print(f"  ⚠  Failed to load seed TLEs: {e}")
    return seeds


def _multi_night_extend(good_tracks, report_lines, ts, observer,
                        start_date, end_date):
    """
    Saves reliable TLEs to disk for use as seeds in the next night's run.
    Also attempts to load a previous night's seeds if SEED_TLE_PATH is set.
    """
    reliable = [
        t for t in good_tracks
        if not math.isnan(t.get("opt_cost", float("nan")))
        and t.get("opt_cost", float("inf")) < 2000.
        and t.get("l1") is not None
        and "99999" not in t.get("l1", "")
    ]

    if not reliable:
        print("\n  Multi-night: no reliable TLEs to save.")
        return

    seed_path = os.path.join(WORKING_DIR, "reliable_tles_for_next_night.txt")
    written   = 0
    with open(seed_path, "w") as f:
        for t in reliable:
            rmse = t.get("opt_cost", float("nan"))
            arc  = t.get("arc_s",   0.)
            mag  = t.get("magnitude", None)
            mag_s = f"  mag={mag:.2f}" if mag else ""
            f.write(f"# RMSE={rmse:.1f}\" arc={arc:.0f}s{mag_s}\n")
            f.write(t["l1"] + "\n")
            f.write(t["l2"] + "\n")
            written += 1

    print(f"\n  Multi-night: saved {written} reliable TLEs → {seed_path}")
    print(f"  To use on next run, set:")
    print(f"    SEED_TLE_PATH = '{seed_path}'")
    print(f"  in the CONFIGURATION section before running.")

    report_lines.append(
        f"\n{'='*60}\n"
        f"MULTI-NIGHT SEED FILE\n"
        f"{'='*60}\n"
        f"Saved {written} reliable TLEs to:\n  {seed_path}\n"
        f"Set SEED_TLE_PATH = '{seed_path}' in next run to extend arcs.\n"
    )



def main():
    t_pipeline_start = time.time()

    stage0_install()
    _lazy_imports()

    all_detections = stage1_to_3_streaming()
    if not all_detections:
        print("❌ No detections from streaming pipeline. Aborting.")
        return

    print(f"\nClassifying {len(all_detections)} detections …")
    moving, static = classify(all_detections, MOTION_ARCSEC)
    # Require high elongation AND cap per-epoch to prevent build_tracks explosion
    satellites = [d for d in moving if d["elongation"] >= MIN_ELONGATION_P1]

    # Hard cap: keep only the top 50 highest-flux candidates per epoch
    # (real satellite streaks are bright; galaxies and artifacts are fainter)
    from collections import defaultdict as _dd
    _by_epoch = _dd(list)
    for d in satellites:
        _by_epoch[round(d["time_jd"], 4)].append(d)
    satellites = []
    for _epoch_dets in _by_epoch.values():
        _epoch_dets.sort(key=lambda x: x["flux"], reverse=True)
        satellites.extend(_epoch_dets[:50])
    print(f"  After per-epoch cap (top 50/epoch): {len(satellites)} candidates")
    print(f"  Moving (satellite candidates): {len(satellites)}")
    print(f"  Static (stars):                {len(static)}")

    print("\nBuilding tracks …")
    all_tracks  = build_tracks(satellites, MAX_SPEED_ARCSEC_S,
                               MAX_GAP_FRAMES, MAX_PA_DIFF_DEG)

    # Cross-field same-night linking — extends 3-obs tracks to 8-15 obs
    print("\nCross-field linking …")
    print("  Pass 1: LEO cross-field linking (30 s window — fast movers) …")
    all_tracks = cross_field_link_tracks(
        all_detections, all_tracks, max_dt_s=30.0
    )
    print("  Pass 2: LEO cross-field linking (120 s window — slow LEO) …")
    all_tracks = cross_field_link_tracks(
        all_detections, all_tracks, max_dt_s=120.0
    )
    print("  Pass 3: MEO cross-field linking (7200 s window) …")
    all_tracks = cross_field_link_tracks(
        all_detections, all_tracks, max_dt_s=7200.0
    )
    print("  Pass 4: GEO cross-field linking (25200 s / 7h window) …")
    all_tracks = cross_field_link_tracks(
        all_detections, all_tracks, max_dt_s=25200.0
    )

    def _n_unique_times(track):
        return len({
            round(d["time_jd"], 4)
            for d in track["obs"]
            if d.get("time_jd") is not None
        })

    good_tracks = sorted(
        [t for t in all_tracks
         if len(set(d["frame_index"] for d in t["obs"])) >= MIN_TRACK_LEN
         and _n_unique_times(t) >= MIN_TRACK_LEN],
        key=lambda t: -_n_unique_times(t)
    )
    print(f"  Good tracks (after temporal uniqueness filter): {len(good_tracks)}")

    if not good_tracks:
        print("\n❌ No tracks found. Check FITS data or adjust thresholds.")
        return

    fields = ["Type", "Frame", "Time_UTC", "Time_JD", "RA_deg", "Dec_deg",
              "Elongation", "Eccentricity", "Flux", "X_pix", "Y_pix", "Filename"]
    with open(OUTPUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for obj_type, obj_list in [("SATELLITE", satellites), ("STAR", static)]:
            for d in obj_list:
                writer.writerow({
                    "Type":         obj_type,
                    "Frame":        d["frame_index"] + 1,
                    "Time_UTC":     d["time_utc"],
                    "Time_JD":      f"{d['time_jd']:.8f}" if d["time_jd"] else "",
                    "RA_deg":       f"{d['ra']:.6f}"  if d["ra"]  is not None else "",
                    "Dec_deg":      f"{d['dec']:.6f}" if d["dec"] is not None else "",
                    "Elongation":   f"{d['elongation']:.3f}",
                    "Eccentricity": f"{d['eccentricity']:.4f}",
                    "Flux":         f"{d['flux']:.2f}",
                    "X_pix":        f"{d['x_pix']:.2f}",
                    "Y_pix":        f"{d['y_pix']:.2f}",
                    "Filename":     d["filename"],
                })
    print(f"✅ CSV written: {OUTPUT_CSV}")

    n_mpc = write_mpc_file(good_tracks, OUTPUT_MPC)
    print(f"✅ MPC file written ({n_mpc} observations): {OUTPUT_MPC}")

    print("\n" + "="*60)
    print("STAGE 4+5: TLE OPTIMISATION & MULTI-CATALOG IDENTIFICATION")
    print("="*60)

    ts       = load.timescale()
    observer = wgs84.latlon(
        SITE_LATITUDE, SITE_LONGITUDE, elevation_m=SITE_ALTITUDE
    )

    print("\nPreparing catalogs …")
    catalog, gcat_lookup, discos_lookup = load_all_catalogs()

    # Load previous-night seed TLEs and prepend to catalog for warm-starting
    seed_tles = load_seed_tles(SEED_TLE_PATH)
    if seed_tles:
        seed_entries = [("PREV_NIGHT_SEED", l1, l2, "SeedTLE")
                        for l1, l2 in seed_tles]
        catalog = seed_entries + catalog
        print(f"  ✅ {len(seed_entries)} previous-night seeds prepended to catalog")

    report_lines = [
        "SATELLITE TRACKING & IDENTIFICATION REPORT\n" + "="*60 + "\n",
        f"Observatory   : Palomar Mountain / ZTF (MPC {OBSERVATORY_CODE})\n",
        f"Location      : lon={SITE_LONGITUDE}°E  lat={SITE_LATITUDE}°N  "
        f"alt={SITE_ALTITUDE}m\n",
        f"Period        : {START_DATE} → {END_DATE}\n",
        f"Instrument    : ZTF — FOV≈{_ZTF_FOV_DEG}°  "
        f"scale≈{_ZTF_SCALE_ARCSEC_PX}\"/px\n",
        f"Catalogs      : CelesTrak ({len(CELESTRAK_GROUPS)} groups) + "
        f"Space-Track gp_history (2021-11-01→{END_DATE}) + GCAT + ESA DISCOS\n",
        f"Tracks found  : {len(good_tracks)}\n",
        "="*60 + "\n\n",
    ]

    # ── STAGE 3L: LEO SINGLE-FRAME PRIMARY IDENTIFICATION ──────────────────
    # Runs FIRST, ahead of the MEO/GEO-oriented multi-night correlation and
    # per-track Gauss-IOD loop below. Most true LEO objects cross a ZTF
    # field within a single exposure (>=400"/s), so the within-frame streak
    # vector — not multi-frame tracking — is the primary LEO detection
    # channel. Previously this ran disconnected, after everything else,
    # with no catalog cross-match or confidence score at all.
    leo_csv_path = os.path.join(WORKING_DIR, "leo_single_frame.csv")
    cat_leo_celestrak  = filter_catalog_by_orbit_class(
        [c for c in catalog if "CelesTrak"   in c[3]], "LEO")
    cat_leo_spacetrack = filter_catalog_by_orbit_class(
        [c for c in catalog if "Space-Track" in c[3]], "LEO")
    if os.path.exists(leo_csv_path):
        print("\n" + "="*60)
        print("STAGE 3L: LEO SINGLE-FRAME PRIMARY IDENTIFICATION")
        print("="*60)
        leo_report = []
        leo_high = leo_med = leo_low = 0
        with open(leo_csv_path, newline="") as f:
            for i, row in enumerate(csv.DictReader(f)):
                try:
                    result = leo_streak_to_tle(
                        float(row["ra"]),
                        float(row["dec"]),
                        float(row["angular_vel"]),
                        float(row["pa_deg"]),
                        float(row["Time_JD"]),
                    )
                    if not result:
                        continue
                    l1, l2, h_km, note = result

                    # Catalog cross-match — this was completely missing
                    # before: single-frame LEO hits were never checked
                    # against CelesTrak/Space-Track at all.
                    t_obs = ts.tt_jd(float(row["Time_JD"]))
                    ct_match, ct_dist, _, ct_gcat, ct_discos = identify_satellite(
                        t_obs, float(row["ra"]), float(row["dec"]),
                        ts, observer, cat_leo_celestrak, gcat_lookup, discos_lookup,
                        residuals_arcsec=None, fitted_elements=None,
                    )
                    st_match, st_dist, _, st_gcat, st_discos = identify_satellite(
                        t_obs, float(row["ra"]), float(row["dec"]),
                        ts, observer, cat_leo_spacetrack, gcat_lookup, discos_lookup,
                        residuals_arcsec=None, fitted_elements=None,
                    )
                    best_dist_arcsec = min(ct_dist, st_dist) * 3600.0
                    # Single-frame circular-orbit IOD has no cross-validated
                    # RMSE, so — same policy as the 2-obs TSA path — it can
                    # never be HIGH. Score confidence off catalog separation.
                    if best_dist_arcsec < 60.0:
                        confidence = "MEDIUM (Single-Frame)"; leo_med += 1
                    elif best_dist_arcsec < 300.0:
                        confidence = "LOW (Single-Frame)"; leo_low += 1
                    else:
                        confidence = "UCT"; leo_low += 1

                    print(f"  LEO{i+1:03d}: altitude≈{h_km:.0f} km  "
                          f"conf={confidence}  "
                          f"best_match={(st_match if st_dist<=ct_dist else ct_match)} "
                          f"(sep={best_dist_arcsec:.1f}\")")
                    leo_report.append(
                        f"LEO STREAK   : LEO{i+1:03d}\n"
                        f"FILE         : {row['filename']}\n"
                        f"TIME (JD)    : {row['Time_JD']}\n"
                        f"POSITION     : RA={row['ra']}° Dec={row['dec']}°\n"
                        f"ANG VEL      : {row['angular_vel']:.1f} arcsec/s\n"
                        f"EST ALTITUDE : {h_km:.0f} km\n"
                        f"CELESTRAK    : {ct_match} ({ct_dist:.4f}°)\n"
                        f"SPACE-TRACK  : {st_match} ({st_dist:.4f}°)\n"
                        f"GCAT INFO    : {ct_gcat or st_gcat or 'N/A'}\n"
                        f"DISCOS INFO  : {ct_discos or st_discos or 'N/A'}\n"
                        f"CONFIDENCE   : {confidence}\n"
                        f"NOTE         : {note}\n"
                        f"TLE:\n{l1}\n{l2}\n"
                        + "-"*60 + "\n"
                    )
                except Exception as e:
                    print(f"  ⚠  LEO{i+1:03d} failed: {e}")

        if leo_report:
            report_lines.append(
                "\n" + "="*60 + "\n"
                "STAGE 3L: LEO SINGLE-FRAME IDENTIFICATIONS "
                f"(MEDIUM={leo_med}  LOW/UCT={leo_low})\n"
                + "="*60 + "\n"
                + "\n".join(leo_report)
            )
            print(f"  ✅ {len(leo_report)} LEO single-frame TLE(s): "
                  f"{leo_med} MEDIUM, {leo_low} LOW/UCT")

        # Cross-field LEO linking (unaffected by the reorder — epoch_str
        # is unused inside leo_cross_field_iod)
        leo_links = leo_cross_field_iod(
            leo_csv_path, ts, observer, epoch_str=None)
        if leo_links:
            leo_link_report = []
            for pair in leo_links:
                leo_link_report.append(
                    f"LEO LINK\n"
                    f"  File A    : {pair['det1']['filename']}\n"
                    f"  File B    : {pair['det2']['filename']}\n"
                    f"  Time sep  : {pair['dt_s']:.1f} s\n"
                    f"  Angular   : {pair['sep_deg']:.3f}°\n"
                    f"  Alt A     : {pair['h_km_A']:.0f} km\n"
                    f"  Alt B     : {pair['h_km_B']:.0f} km\n"
                    f"  TLE (from A):\n{pair['l1_A']}\n{pair['l2_A']}\n"
                    f"  TLE (from B):\n{pair['l1_B']}\n{pair['l2_B']}\n"
                    + "-"*60 + "\n"
                )
            report_lines.append(
                "\n" + "="*60 + "\n"
                "STAGE 3L-X: LEO CROSS-FIELD LINKED PAIRS\n"
                + "="*60 + "\n"
                + "\n".join(leo_link_report)
            )
            print(f"  ✅ {len(leo_links)} LEO cross-field link(s) written to report")

    print("\n" + "="*60)
    print("STAGE 4A: CROSS-NIGHT TRACKLET CORRELATION (MEO/GEO FALLBACK)")
    print("="*60)
    fused_groups = correlate_and_fuse_tracklets_across_nights(
        good_tracks, ts, observer,
        min_gap_s=3600.0, max_gap_s=5 * 86400.0,
        screen_thresh_deg=1.5, verify_rmse_thresh_arcsec=200.0,
    )

    fused_results = solve_fused_groups(
        fused_groups, ts, observer, catalog,
        gcat_lookup, discos_lookup, report_lines
    )

    fused_track_ids = {tid for g in fused_groups for tid in g["member_track_ids"]}

    multi_pass_registry: dict = defaultdict(list)

    for track_id, track_dict in enumerate(good_tracks):
        obs_list = track_dict["obs"]
        if len(obs_list) < MIN_TRACK_LEN:
            continue

        print(f"\nTrack USER{track_id+1:03d}  ({len(obs_list)} observations)")

        if track_id in fused_track_ids:
            print("  ℹ  Superseded by cross-night fused solution (Stage 4B) — skipping individual report entry.")
            continue

        _well_observed = False   # reset every track -- prevents leaking a
                                   # stale value from a previous track's
                                   # iteration via locals()

        orbit_class, ang_vel_arcsec_s = classify_orbit_class(obs_list)
        _leo_note = " [LEO — full IOD pipeline active]" if orbit_class == "LEO" else ""
        print(f"  Orbit class    : {orbit_class}  ({ang_vel_arcsec_s:.1f} arcsec/s){_leo_note}")

        obs_times, obs_vecs, raw_coords = [], [], []
        for row in obs_list:
            try:
                t_astropy = Time(row["time_utc"], format="isot", scale="utc")
            except Exception:
                continue
            obs_times.append(ts.from_astropy(t_astropy))
            obs_vecs.append(radec_to_vec(row["ra"], row["dec"]))
            raw_coords.append((row["ra"], row["dec"]))

        if len(obs_times) < MIN_TRACK_LEN:
            continue

        t_0 = Time(obs_list[0]["time_utc"], format="isot", scale="utc")
        doy_frac = (t_0.jd - Time(
            f"{t_0.datetime.year}-01-01T00:00:00",
            format="isot", scale="utc"
        ).jd) + 1.0
        epoch_str      = f"{t_0.datetime.year % 100:02d}{doy_frac:012.8f}"
        arc_duration_s = (obs_times[-1].tt - obs_times[0].tt) * 86400.0

        # Pre-search catalog for Stage B seed (new)
        cat_l1, cat_l2, pre_sep = _pre_search_catalog(
            obs_times, raw_coords, ts, observer, catalog, orbit_class)
        if cat_l1:
            print(f"  Catalog pre-search: nearest TLE at {pre_sep:.3f}°")
        else:
            print(f"  Catalog pre-search: no match found")

        # GEO UCT gate: 3-obs 7-min arcs cannot determine range to GEO.
        # Only attempt OD if a catalog seed is within 5° (Stage B can then
        # converge from a physically consistent starting point).
        _cov_str  = "Covariance unavailable"   # default; overwritten if OD runs
        _best_x_rad = None
        _tsa_provisional = False   # True only for the 2-obs LEO TSA path
        

        # ROOT-CAUSE FIX #3: previously EVERY GEO track with the nearest
        # catalogued object > 5° away was dumped to UCT with no OD attempt —
        # that gated purely on catalog proximity, not on whether the arc could
        # actually be solved, and it sent all 175 GEO tracks straight to UCT.
        # Now we only hard-skip the genuinely hopeless case (a sparse arc);
        # richer GEO arcs get a best-effort OD. Confidence stays honestly gated
        # downstream by the observability check, so this cannot manufacture a
        # false HIGH — at worst it turns an unsolved UCT into an explicit LOW.
        if orbit_class == "GEO" and pre_sep > 5.0 and len(obs_times) < 6:
            print(f"  ℹ  GEO UCT: pre_sep={pre_sep:.2f}° > 5.0° and only "
                  f"{len(obs_times)} obs — skipping OD (arc too short for range)")
            l1 = f"1 99999U 22999A   {epoch_str}  .00000000  00000-0  00000-0 0  999"
            l1 = f"{l1:<68}{compute_checksum(l1)}"
            l2 = (f"2 99999   0.0000   0.0000 0000000   0.0000   "
                  f"0.0000  1.00000000000013")
            l2 = f"{l2:<68}{compute_checksum(l2)}"
            rmse       = float("nan")
            opt_cost   = float("nan")
            confidence = "UCT"
        elif len(obs_times) == 2 and orbit_class == "LEO":
            # Gauss IOD needs >=3 points, but a 2-point LEO tracklet still
            # carries a full attributable (RA, Dec, RA-dot, Dec-dot) — the
            # same information a single-frame streak provides. Reuse the
            # circular-orbit altitude-scan TSA solver instead of dropping
            # straight to UCT (see two_point_leo_iod docstring for refs).
            _tsa = two_point_leo_iod(obs_list)
            if _tsa is not None:
                l1, l2, _h_km, _tsa_note = _tsa
                print(f"  ℹ  2-obs LEO tracklet — TSA altitude-scan IOD: "
                      f"h≈{_h_km:.0f} km")
                rmse       = float("nan")   # no cross-validation possible w/ 2 pts
                opt_cost   = float("nan")
                confidence = "LOW"          # provisional until catalog match below
                _tsa_provisional = True
            else:
                print(f"  ℹ  Only 2 observations — TSA altitude-scan IOD "
                      f"did not converge — UCT")
                l1 = f"1 99999U 22999A   {epoch_str}  .00000000  00000-0  00000-0 0  999"
                l1 = f"{l1:<68}{compute_checksum(l1)}"
                l2 = (f"2 99999   0.0000   0.0000 0000000   0.0000   "
                      f"0.0000  1.00000000000013")
                l2 = f"{l2:<68}{compute_checksum(l2)}"
                rmse       = float("nan")
                opt_cost   = float("nan")
                confidence = "UCT"
        elif len(obs_times) < 3:
            print(f"  ℹ  Only {len(obs_times)} observations — skipping OD (minimum 3 required)")
            l1 = f"1 99999U 22999A   {epoch_str}  .00000000  00000-0  00000-0 0  999"
            l1 = f"{l1:<68}{compute_checksum(l1)}"
            l2 = (f"2 99999   0.0000   0.0000 0000000   0.0000   "
                  f"0.0000  1.00000000000013")
            l2 = f"{l2:<68}{compute_checksum(l2)}"
            rmse       = float("nan")
            opt_cost   = float("nan")
            confidence = "UCT"
        elif arc_duration_s < 10.0:
            print(f"  ⚠  Short arc ({arc_duration_s:.1f}s) — skipping optimisation")
            l1 = f"1 99999U 22999A   {epoch_str}  .00000000  00000-0  00000-0 0  999"
            l1 = f"{l1:<68}{compute_checksum(l1)}"
            l2 = (f"2 99999   0.0000   0.0000 0000000   0.0000   "
                  f"0.0000  1.00000000000013")
            l2 = f"{l2:<68}{compute_checksum(l2)}"
            rmse       = float("nan")
            opt_cost   = float("nan")
            confidence = "LOW"
        else:
            print("  Optimising TLE (3-stage: Gauss IOD → Catalog seed → DE) …")
            opt_params, opt_cost, _cov = optimise_tle(
                obs_times, obs_vecs, epoch_str, ts, observer,
                arc_duration_s=arc_duration_s, orbit_class=orbit_class,
                catalog_l1=cat_l1, catalog_l2=cat_l2,
            )
            if math.isnan(opt_cost) or opt_cost >= MAX_ACCEPTABLE_DE_RMSE_ARCSEC:
                print(f"  ⚠  Orbit fit failed (cost={opt_cost:.0f}\") — no reliable TLE")
                l1 = f"1 99999U 22999A   {epoch_str}  .00000000  00000-0  00000-0 0  999"
                l1 = f"{l1:<68}{compute_checksum(l1)}"
                l2 = (f"2 99999   0.0000   0.0000 0000000   0.0000   "
                      f"0.0000  1.00000000000013")
                l2 = f"{l2:<68}{compute_checksum(l2)}"
                rmse       = float("nan")
                confidence = "LOW"
            else:
                l1, l2 = generate_tle_lines(*opt_params, epoch_str)

                # Convert opt_params (degrees) back to radians for covariance
                _best_x_rad = np.array([
                    math.radians(opt_params[0]),                    # incl
                    math.radians(opt_params[1]) % (2*math.pi),     # raan
                    opt_params[2],                                   # ecc
                    math.radians(opt_params[3]) % (2*math.pi),     # argp
                    math.radians(opt_params[4]) % (2*math.pi),     # ma
                    opt_params[5] * REV_PER_DAY_TO_RAD_PER_MIN,   # mm
                ])

                # Propagate covariance to +24h for position uncertainty estimate
                yy = int(epoch_str[:2])
                year = (2000 + yy) if yy < 57 else (1900 + yy)
                doy_f = float(epoch_str[2:])
                epoch_jd = float(Time(f"{year}-01-01T00:00:00",
                                       format='isot', scale='utc').jd) + (doy_f - 1.0)
                _future_jd = epoch_jd + 1.0
                _sig_ra, _sig_dec, _sig_tot, _cov_sky = _propagate_covariance(
                    _best_x_rad,
                    _cov, epoch_jd, _future_jd,
                    SITE_LATITUDE, SITE_LONGITUDE, SITE_ALTITUDE,
                    np.array([]), np.array([]), np.array([]),
                )
                _cov_str = (
                    f"σ_RA={_sig_ra:.1f}\"  σ_Dec={_sig_dec:.1f}\"  "
                    f"σ_total={_sig_tot:.1f}\" (1σ at +24h)"
                    if not math.isnan(_sig_ra) else "Covariance unavailable"
                )

                _sig_mm, _rel_sig_mm, _well_observed = _fit_observability_metric(
                    _cov, _best_x_rad[5])
                _cov_str += (
                    f"  |  σ_mm={_sig_mm:.5f} rev/day ({_rel_sig_mm*100:.2f}% rel)  "
                    f"{'[WELL-OBSERVED]' if _well_observed else '[RANGE UNCONSTRAINED]'}"
                )
            rmse = opt_cost

        
        residuals_arcsec_vec = None
        if arc_duration_s >= 10.0 and not np.isnan(opt_cost):
            sat_cov  = EarthSatellite(l1, l2, "COV", ts)
            _cov_res = []
            for _t, (_ra_o, _dec_o) in zip(obs_times, raw_coords):
                try:
                    _pr, _pd, _ = (sat_cov - observer).at(_t).radec()
                    _cos_d = math.cos(math.radians(_dec_o))
                    _cov_res.extend([
                        (_pr.hours * 15. - _ra_o) * _cos_d * 3600.,
                        (_pd.degrees - _dec_o) * 3600.
                    ])
                except Exception:
                    _cov_res.extend([0., 0.])
            residuals_arcsec_vec = _cov_res

        mid_idx        = len(obs_times) // 2
        cat_celestrak  = filter_catalog_by_orbit_class(
            [c for c in catalog if "CelesTrak"   in c[3]], orbit_class)
        cat_spacetrack = filter_catalog_by_orbit_class(
            [c for c in catalog if "Space-Track" in c[3]], orbit_class)
        print(f"  Catalog search : {len(cat_celestrak)} CelesTrak + "
              f"{len(cat_spacetrack)} Space-Track ({orbit_class} band)")

        _fitted_elems = ((opt_params[0], opt_params[2], opt_params[5])
                          if not math.isnan(opt_cost) else None)
        ct_match, ct_dist, _, ct_gcat, ct_discos = identify_satellite(
            obs_times[mid_idx], raw_coords[mid_idx][0], raw_coords[mid_idx][1],
            ts, observer, cat_celestrak, gcat_lookup, discos_lookup,
            residuals_arcsec=residuals_arcsec_vec, fitted_elements=_fitted_elems
        )
        st_match, st_dist, _, st_gcat, st_discos = identify_satellite(
            obs_times[mid_idx], raw_coords[mid_idx][0], raw_coords[mid_idx][1],
            ts, observer, cat_spacetrack, gcat_lookup, discos_lookup,
            residuals_arcsec=residuals_arcsec_vec, fitted_elements=_fitted_elems
        )

        best_dist = min(ct_dist, st_dist)
        best_dist_arcsec = best_dist * 3600.0
        # _well_observed is guaranteed to be set (True only if this track's
        # fit reached the observability check above; False otherwise).
        if arc_duration_s >= 10.0 and not math.isnan(opt_cost):
            if not _well_observed:
                # Angular fit may look good, but range/mm is unconstrained.
                # Never call this HIGH or MEDIUM regardless of RMSE.
                confidence = "LOW (Range Unconstrained — Short Arc)"
            elif opt_cost < 30.0:
                confidence = "HIGH"   if best_dist_arcsec < 60.0  else "MEDIUM"
            elif opt_cost < 200.0:
                confidence = "MEDIUM" if best_dist_arcsec < 300.0 else "LOW"
            elif opt_cost < MAX_ACCEPTABLE_DE_RMSE_ARCSEC:
                confidence = "LOW"
            else:
                confidence = "LOW (Orbit Fit Failed)"
        elif _tsa_provisional:
            # 2-point TSA IOD has no cross-validation, so it can never reach
            # HIGH — but a tight catalog match is still meaningful evidence,
            # so score it on separation rather than blanket-labeling it a
            # failed fit.
            confidence = ("MEDIUM (TSA)" if best_dist_arcsec < 60.0
                          else "LOW (TSA)" if best_dist_arcsec < 300.0
                          else "LOW (TSA — No Catalog Match)")
        elif confidence != "UCT":
            confidence = "LOW (Orbit Fit Failed)"

        # Store results back for post-loop noise filtering
        track_dict["confidence"] = confidence
        track_dict["opt_cost"]   = opt_cost
        track_dict["rmse"]       = rmse
        track_dict["l1"]         = l1
        track_dict["l2"]         = l2
        track_dict["arc_s"]      = arc_duration_s

        rmse_str = "N/A" if math.isnan(rmse) else f"{rmse:.2f}\""

        print(f"     Arc Length  : {arc_duration_s:.1f}s")
        print(f"     Orbit RMSE  : {rmse_str}")
        print(f"     Confidence  : {confidence}")
        print(f"  ✅ CELESTRAK   : {ct_match} (Sep: {ct_dist:.3f}°)")
        print(f"  ✅ SPACE-TRACK : {st_match} (Sep: {st_dist:.3f}°)")

        # Compute mean magnitude for the track
        mags = [d.get("magnitude") for d in obs_list if d.get("magnitude") is not None]
        mean_mag     = round(sum(mags)/len(mags), 2) if mags else None
        mag_rng      = (round(min(mags),2), round(max(mags),2)) if mags else (None,None)
        mag_str      = (f"{mean_mag} (range {mag_rng[0]}–{mag_rng[1]})"
                        if mean_mag is not None else "N/A")

        report_lines.append(
            f"TRACK        : USER{track_id+1:03d}\n"
            f"ORBIT CLASS  : {orbit_class} ({ang_vel_arcsec_s:.1f} arcsec/s)\n"
            f"OBSERVATIONS : {len(obs_times)}\n"
            f"ARC DURATION : {arc_duration_s:.1f} s\n"
            f"MAGNITUDE    : {mag_str} (AB, ZTF calibrated)\n"
            f"CELESTRAK    : {ct_match} ({ct_dist:.4f}°)\n"
            f"SPACE-TRACK  : {st_match} ({st_dist:.4f}°)\n"
            f"GCAT INFO    : {ct_gcat or st_gcat or 'N/A'}\n"
            f"DISCOS INFO  : {ct_discos or st_discos or 'N/A'}\n"
            f"CONFIDENCE   : {confidence}\n"
            f"ORBIT RMSE   : {rmse_str}\n"
            f"POSITION UNC : {_cov_str if '_cov_str' in dir() else 'N/A'}\n"
            f"OPT. TLE:\n{l1}\n{l2}\n"
            f"{'-'*60}\n"
        )

        # Space-Track gp_history has epoch-matched TLEs and is more reliable
        # for identification. Prefer it when its separation is < 5° and
        # meaningfully closer than CelesTrak.
        if "UNKNOWN" not in st_match and st_dist < 5.0:
            best_name = st_match
        elif "UNKNOWN" not in ct_match and ct_dist < 5.0:
            best_name = ct_match
        elif "UNKNOWN" not in st_match and st_dist <= ct_dist:
            best_name = st_match
        elif "UNKNOWN" not in ct_match:
            best_name = ct_match
        else:
            best_name = "UNKNOWN"

        if "UNKNOWN" not in best_name:
            multi_pass_registry[best_name].append({
                "track_id":   track_id,
                "obs_times":  obs_times,
                "obs_vecs":   obs_vecs,
                "raw_coords": raw_coords,
            })

    # ── Noise track filter ────────────────────────────────────────────────
    print("\n" + "="*60)
    print("POST-LOOP: NOISE TRACK REMOVAL")
    print("="*60)
    reliable = [t for t in good_tracks
                if t.get("confidence") in ("HIGH", "MEDIUM", "UCT")]
    dropped_n = len(good_tracks) - len(reliable)
    print(f"  Input tracks  : {len(good_tracks)}")
    print(f"  Dropped (LOW) : {dropped_n}")
    print(f"  Kept          : {len(reliable)}")



    # Actually replace good_tracks with only reliable ones so that
    # the report, MPC file, and Stage 6B only contain quality tracks.
    if reliable:
        good_tracks = reliable
        print(f"  ✅ good_tracks updated to {len(good_tracks)} reliable tracks only.")
    else:
        print(f"  ⚠  No reliable tracks — keeping all {len(good_tracks)} for inspection.")

    # Rewrite MPC file with only reliable tracks
    n_mpc_reliable = write_mpc_file(good_tracks,
                                     OUTPUT_MPC.replace(".txt", "_reliable.txt"))
    print(f"  ✅ Reliable MPC file: {n_mpc_reliable} obs → "
          f"{OUTPUT_MPC.replace('.txt', '_reliable.txt')}")

    # Rebuild multi_pass_registry using only reliable tracks
    multi_pass_registry_reliable = defaultdict(list)
    for sat_name, passes in multi_pass_registry.items():
        for p in passes:
            t_id  = p["track_id"]
            t_obj = good_tracks[t_id] if t_id < len(good_tracks) else None
            if t_obj and t_obj.get("confidence") in ("HIGH", "MEDIUM"):
                multi_pass_registry_reliable[sat_name].append(p)
    if multi_pass_registry_reliable:
        print(f"  Reliable multi-pass satellites: "
              f"{len(multi_pass_registry_reliable)}")
        multi_pass_registry = multi_pass_registry_reliable

    # Pipeline metrics summary
    rmse_vals = [t["opt_cost"] for t in good_tracks
                 if not math.isnan(t.get("opt_cost", float("nan")))
                 and t.get("opt_cost", float("inf")) < MAX_ACCEPTABLE_DE_RMSE_ARCSEC]
    high_c  = sum(1 for t in good_tracks if t.get("confidence") == "HIGH")
    med_c   = sum(1 for t in good_tracks if t.get("confidence") == "MEDIUM")
    uct_c   = sum(1 for t in good_tracks if t.get("confidence") == "UCT")
    xf_c    = sum(1 for t in good_tracks if t.get("cross_field"))
    obs_avg = sum(len(t["obs"]) for t in good_tracks) / max(1, len(good_tracks))
    print(f"\n  PIPELINE METRICS")
    print(f"  Total tracks          : {len(good_tracks)}")
    print(f"  Confidence HIGH       : {high_c}")
    print(f"  Confidence MEDIUM     : {med_c}")
    print(f"  UCT                   : {uct_c}")
    print(f"  Cross-field extended  : {xf_c}")
    print(f"  Mean obs/track        : {obs_avg:.1f}")
    if rmse_vals:
        print(f"  Best RMSE            : {min(rmse_vals):.1f}\"")
        print(f"  Median RMSE          : {sorted(rmse_vals)[len(rmse_vals)//2]:.1f}\"")
        print(f"  RMSE < 500\"          : {sum(1 for r in rmse_vals if r < 500)}")
        print(f"  RMSE < 100\"          : {sum(1 for r in rmse_vals if r < 100)}")

    if ENABLE_LEGACY_STAGE6B_FUSION:
        print("  ⚠  ENABLE_LEGACY_STAGE6B_FUSION=True — catalog-name-based "
              "fusion output will appear in the report. This mechanism is "
              "known to group unrelated objects at large separations; treat "
              "its output as debugging information only, not a result.")
        stage6b_master_tle_solve(
            multi_pass_registry, ts, observer, report_lines,
            gcat_lookup=gcat_lookup, discos_lookup=discos_lookup
        )
    else:
        print("  ℹ  Stage 6B (legacy catalog-name fusion) disabled — "
              "relying on Stage 4A/4B geometry-based fusion only.")

    with open(OUTPUT_REPORT, "w", encoding="utf-8") as f:
        f.writelines(report_lines)
    print(f"\n✅ Report written: {OUTPUT_REPORT}")

    print("\n" + "="*60)
    print("STAGE 7: PACKAGING OUTPUTS")
    print("="*60)

    zip_src = os.path.join(WORKING_DIR, "pipeline_outputs")
    os.makedirs(zip_src, exist_ok=True)
    for src in [OUTPUT_CSV, OUTPUT_MPC, OUTPUT_REPORT]:
        if os.path.exists(src):
            shutil.copy(src, zip_src)
    shutil.make_archive(ZIP_PATH, "zip", zip_src)
    shutil.rmtree(zip_src, ignore_errors=True)

    elapsed = time.time() - t_pipeline_start
    print(f"\n{'='*60}")
    print(f"✅ PIPELINE COMPLETE  —  {elapsed/60:.1f} min total")
    print(f"   Output zip : {ZIP_PATH}.zip")
    print(f"   CSV        : {OUTPUT_CSV}")
    print(f"   MPC        : {OUTPUT_MPC}")
    print(f"   Report     : {OUTPUT_REPORT}")
    print("="*60)


if __name__ == "__main__":
    main()


── Credential check ──────────────────────────────────────────
  ✅ IRSA_USER            ab***********************om
  ✅ IRSA_PASS            K**********21
  ✅ SPACETRACK_USER      ab***********************om
  ✅ SPACETRACK_PASS      S*********************12
  ✅ DISCOS_TOKEN         Ij***************************************************************************Gk
──────────────────────────────────────────────────────────────


STAGE 0: INSTALLING DEPENDENCIES
  scipy pre-installed in Cell 1 — skipping reinstall ✅
  Activating CuPy GPU backend (T4) … ⚠  CuPy not available — falling back to CPU (No module named 'cupy')
  CUDA 12 detected → installing cupy-cuda12x
  Installing cupy-cuda12x … ✅
  Installing astropy … ✅
  Installing photutils.git … ✅
  Installing sgp4 … ✅
  Installing skyfield … ✅
  Installing pandas … ✅
  Installing beautifulsoup4 … ✅
  Installing lxml … ✅
  Installing GTK2 + Xvfb for ASTAP … ✅
  Installing ASTAP CLI engine … Selecting previously unselected package astap.
(R